# Strawberry Ripeness Classification Under LED-Induced Domain Shift
### WMG9B7-15 — Artificial Intelligence & Deep Learning | Individual Assessment 2025/26

---

## README — How to Run This Notebook

**One instruction: `Runtime → Run All`**

This notebook is fully self-contained. It downloads all required data programmatically, installs all dependencies, and executes all five experiments end-to-end without any manual steps.

### Requirements
| Requirement | Detail |
|---|---|
| Platform | Google Colab (recommended) or any Python 3.10+ environment |
| GPU | T4 or better strongly recommended (CPU fallback available but slow) |
| RAM | 12 GB minimum |
| Disk | 5 GB free space |
| Internet | Required for first run (dataset download ~1.49 GB) |
| Estimated runtime | ~90–120 minutes on Colab T4 |

### Project Overview
**Problem**: Dogtooth Technologies Gen5 strawberry-harvesting robots are trained on daytime field images but operate 24 hours per day under onboard LED illumination. The spectral shift from natural sunlight to LED causes domain shift — ripe (red) berries appear fundamentally different under LED light, degrading classification accuracy.

**Approach**: Patch classification (binary: ripe vs unripe) on real commercial field images. Five controlled experiments quantify the domain shift and test augmentation-based domain adaptation.

**Dataset**: Pastell, M. & Liakka, V. (2022). *Strawberry dataset for object detection*. Zenodo. https://doi.org/10.5281/zenodo.6126677. CC BY 4.0.

### Experiment Summary
| ID | Description |
|---|---|
| E1 | DL baseline — ResNet trained on daytime patches, evaluated on daytime test set |
| E2 | SVM baseline — hand-crafted colour + texture features, evaluated on daytime test set |
| E3 | Domain shift — E1 model evaluated on LED-simulated night test set (no retraining) |
| E4 | Degradation quantification — E1 vs E3 metric delta |
| E5 | Domain adaptation — retrained on day + night-augmented patches, evaluated on night test set |

### Output Locations
All outputs are saved under `/content/strawberry_project/results/`:
- `figures/` — all plots at 300 DPI
- `metrics/` — JSON files with full metric results per experiment
- `models/` — saved model checkpoints

### Developer Note (optional Drive caching)
Set `USE_DRIVE_CACHE = True` in the Configuration cell below to cache the dataset to Google Drive — this skips the download on subsequent sessions. **Leave as `False` for standard single-session runs.**

---

## Phase 0 — Environment & Infrastructure Setup

Sets up the runtime environment, validates GPU availability, establishes the project directory structure, and configures global reproducibility settings. All subsequent phases depend on the constants and utilities defined here.

### Step 0.1 — Configuration, Directory Architecture & Drive Cache (optional)

In [10]:
# ============================================================
# STEP 0.1 — CONFIGURATION & DIRECTORY ARCHITECTURE
# ============================================================
import os
import sys
from pathlib import Path

USE_DRIVE_CACHE = False

IS_COLAB = 'google.colab' in sys.modules
if not IS_COLAB:
    try:
        import google.colab; IS_COLAB = True
    except ImportError:
        IS_COLAB = False

print(f"Runtime environment : {'Google Colab' if IS_COLAB else 'Local / Other'}")
print(f"Drive cache enabled : {USE_DRIVE_CACHE}")

DRIVE_PROJECT_DIR = None
if USE_DRIVE_CACHE and IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/AIDL_Strawberry_Project')
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir   : {DRIVE_PROJECT_DIR}")
elif USE_DRIVE_CACHE and not IS_COLAB:
    print("[WARNING] USE_DRIVE_CACHE=True but not running in Colab — disabled.")
    USE_DRIVE_CACHE = False

BASE_DIR = Path('/content/strawberry_project') if IS_COLAB else Path('./strawberry_project')

DIRS = {
    'raw'              : BASE_DIR / 'data' / 'raw',
    'patches'          : BASE_DIR / 'data' / 'patches',
    'train_ripe'       : BASE_DIR / 'data' / 'patches' / 'train'      / 'ripe',
    'train_unripe'     : BASE_DIR / 'data' / 'patches' / 'train'      / 'unripe',
    'val_ripe'         : BASE_DIR / 'data' / 'patches' / 'val'        / 'ripe',
    'val_unripe'       : BASE_DIR / 'data' / 'patches' / 'val'        / 'unripe',
    'test_day_ripe'    : BASE_DIR / 'data' / 'patches' / 'test_day'   / 'ripe',
    'test_day_unripe'  : BASE_DIR / 'data' / 'patches' / 'test_day'   / 'unripe',
    'test_night_ripe'  : BASE_DIR / 'data' / 'patches' / 'test_night' / 'ripe',
    'test_night_unripe': BASE_DIR / 'data' / 'patches' / 'test_night' / 'unripe',
    'splits'           : BASE_DIR / 'data' / 'splits',
    'model_E1'         : BASE_DIR / 'models' / 'E1_day',
    'model_E5'         : BASE_DIR / 'models' / 'E5_adapted',
    'model_svm'        : BASE_DIR / 'models' / 'svm',
    'figures'          : BASE_DIR / 'results' / 'figures',
    'metrics'          : BASE_DIR / 'results' / 'metrics',
    'logs'             : BASE_DIR / 'logs' / 'training',
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print(f"\nProject base dir    : {BASE_DIR}")
print(f"Directories created : {len(DIRS)}")

ZENODO_URL      = 'https://zenodo.org/record/6126677/files/strawberries.zip'
ZENODO_MD5      = 'db8d5dcb4b8adebf1621788373fd3031'
ARCHIVE_PATH    = DIRS['raw'] / 'strawberries.zip'
DATASET_DIR     = DIRS['raw'] / 'strawberries'
CLASS_MAP       = {0: 'ripe', 1: 'unripe'}
CLASS_NAMES     = ['ripe', 'unripe']
NUM_CLASSES     = 2
N_TEST_IMAGES   = 150
TRAIN_VAL_RATIO = 0.80
PATCH_SIZE      = 224
MIN_PATCH_PX    = 20
PATCH_PADDING   = 0.10
SPLIT_CSV       = DIRS['splits'] / 'split_assignments.csv'

print(f"\nDataset URL         : {ZENODO_URL}")
print(f"Expected MD5        : {ZENODO_MD5}")
print(f"Class mapping       : {CLASS_MAP}")
print(f"Patch size          : {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Test images (held-out): {N_TEST_IMAGES}")

Runtime environment : Google Colab
Drive cache enabled : False

Project base dir    : /content/strawberry_project
Directories created : 17

Dataset URL         : https://zenodo.org/record/6126677/files/strawberries.zip
Expected MD5        : db8d5dcb4b8adebf1621788373fd3031
Class mapping       : {0: 'ripe', 1: 'unripe'}
Patch size          : 224x224
Test images (held-out): 150


#### Step 0.1b — Drive Cache Verification (runs only if USE_DRIVE_CACHE = True)

In [11]:
# ============================================================
# STEP 0.1b — DRIVE CACHE VERIFICATION
# ============================================================
import shutil

CACHE_HIT = False

if USE_DRIVE_CACHE and DRIVE_PROJECT_DIR is not None:
    drive_patches = DRIVE_PROJECT_DIR / 'data' / 'patches'
    drive_split   = DRIVE_PROJECT_DIR / 'data' / 'splits' / 'split_assignments.csv'
    drive_raw     = DRIVE_PROJECT_DIR / 'data' / 'raw' / 'strawberries'

    if drive_patches.exists() and drive_split.exists() and drive_raw.exists():
        print("[CACHE HIT] Copying from Drive to local working directory...")
        if not DATASET_DIR.exists():
            shutil.copytree(str(drive_raw), str(DATASET_DIR))
        if not (DIRS['patches'] / 'train').exists():
            shutil.copytree(str(drive_patches), str(DIRS['patches']), dirs_exist_ok=True)
        shutil.copy2(str(drive_split), str(SPLIT_CSV))
        CACHE_HIT = True
        print("[OK] Drive cache restored. Phases 1-2 will be skipped.")
    else:
        missing = [k for k, v in {'raw': drive_raw, 'patches': drive_patches,
                                   'split': drive_split}.items() if not v.exists()]
        print(f"[CACHE MISS] Missing: {missing}. Full download will proceed.")
else:
    print("[INFO] Drive cache disabled. Fresh download will occur in Phase 1.")

print(f"CACHE_HIT = {CACHE_HIT}")

[INFO] Drive cache disabled. Fresh download will occur in Phase 1.
CACHE_HIT = False


### Step 0.2 — Dependency Installation & Version Pinning

Installs the three packages not pre-installed in Colab: `albumentations`, `grad-cam`, and `scikit-image`. All other packages use Colab's pre-installed versions to avoid CUDA conflicts. Package versions are logged to `metadata.json`.

In [12]:
# ============================================================
# STEP 0.2a — INSTALL MISSING DEPENDENCIES
# ============================================================
print("Installing missing dependencies...")
print("(torch, torchvision, numpy, sklearn, cv2, scipy left untouched)\n")

%pip install -q "albumentations>=1.3.0,<2.0.0"
%pip install -q "grad-cam>=1.4.6"
%pip install -q "scikit-image>=0.21.0"

print("\n[OK] Installation complete.")

Installing missing dependencies...
(torch, torchvision, numpy, sklearn, cv2, scipy left untouched)


[OK] Installation complete.


In [13]:
# ============================================================
# STEP 0.2b — GLOBAL IMPORTS
# ============================================================
import json, math, random, time, warnings, hashlib, zipfile
import urllib.request
from copy import deepcopy
from datetime import datetime
import importlib, importlib.metadata, platform

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import bootstrap as scipy_bootstrap

import cv2
from PIL import Image
from skimage.feature import local_binary_pattern
from skimage.feature import hog as skimage_hog

import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms as T
from torchvision import models

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
import joblib

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

print("[OK] All imports successful.")

[OK] All imports successful.


In [14]:
# ============================================================
# STEP 0.2c — GLOBAL SETTINGS, UTILITIES & VERSION LOGGING
# ============================================================
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 300,
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.titlesize': 13, 'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})
sns.set_theme(style='whitegrid', palette='muted')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected        : {GPU_NAME}")
    print(f"VRAM available      : {GPU_VRAM:.1f} GB")
else:
    GPU_NAME, GPU_VRAM = 'CPU', 0.0
    print("[WARNING] No GPU detected — enable via Runtime > Change runtime type > T4 GPU")
print(f"Compute device      : {DEVICE}")

def print_section(title: str, width: int = 60) -> None:
    bar = '=' * width
    print(f"\n{bar}\n  {title}\n{bar}")

def _get_version(pkg: str) -> str:
    try: return importlib.metadata.version(pkg)
    except Exception: return getattr(importlib.import_module(pkg.replace('-','_')), '__version__', 'unknown')

version_map = {p: _get_version(p) for p in [
    'torch','torchvision','albumentations','grad-cam','scikit-learn',
    'scikit-image','numpy','pandas','scipy','matplotlib','seaborn','Pillow','tqdm','joblib']}
version_map.update({'torch': torch.__version__, 'cv2': cv2.__version__,
                    'numpy': np.__version__, 'pandas': pd.__version__})

METADATA = {
    'project': 'Strawberry LED Domain Shift — WMG9B7-15',
    'timestamp': datetime.now().isoformat(),
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'gpu_name': GPU_NAME, 'gpu_vram_gb': round(GPU_VRAM, 2),
    'cuda_version': torch.version.cuda if torch.cuda.is_available() else 'N/A',
    'device': str(DEVICE), 'packages': version_map,
    'config': {'patch_size': PATCH_SIZE, 'min_patch_px': MIN_PATCH_PX,
               'patch_padding': PATCH_PADDING, 'n_test_images': N_TEST_IMAGES,
               'train_val_ratio': TRAIN_VAL_RATIO, 'class_map': CLASS_MAP,
               'use_drive_cache': USE_DRIVE_CACHE}
}
metadata_path = BASE_DIR / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print_section("Step 0.2 — Package Versions")
for pkg, ver in version_map.items():
    print(f"  {pkg:<30} {ver}")
print(f"\n  Metadata saved → {metadata_path}")
print(f"\n[OK] Step 0.2 complete.")

GPU detected        : Tesla T4
VRAM available      : 15.6 GB
Compute device      : cuda

  Step 0.2 — Package Versions
  torch                          2.10.0+cu128
  torchvision                    0.25.0+cu128
  albumentations                 1.4.24
  grad-cam                       1.5.5
  scikit-learn                   1.6.1
  scikit-image                   0.25.2
  numpy                          2.0.2
  pandas                         2.2.2
  scipy                          1.16.3
  matplotlib                     3.10.0
  seaborn                        0.13.2
  Pillow                         11.3.0
  tqdm                           4.67.3
  joblib                         1.5.3
  cv2                            4.13.0

  Metadata saved → /content/strawberry_project/metadata.json

[OK] Step 0.2 complete.


### Step 0.3 — Global Reproducibility Setup

Defines and applies seed control across all seven sources of randomness in the project: Python `random`, NumPy, PyTorch CPU, PyTorch CUDA, cuDNN algorithm selection, cuDNN kernel determinism, and DataLoader worker processes. Reproducibility is **best-effort on same hardware** — minor numerical variation across different CUDA driver versions or GPU architectures is expected and documented, consistent with standard ML research practice.

In [15]:
# ============================================================
# STEP 0.3 — GLOBAL REPRODUCIBILITY SETUP
# ============================================================

# ── Seed constants ───────────────────────────────────────────
# GLOBAL_SEED: used for all non-training operations
#   (split generation, SVM, EDA sampling, patch ordering)
# EXPERIMENT_SEEDS: one per DL training run — three independent
#   runs per experiment allow mean ± std reporting.
#   Seeds are well-separated in seed space to ensure genuinely
#   different weight initialisations.
# ------------------------------------------------------------
GLOBAL_SEED      = 42
EXPERIMENT_SEEDS = [42, 123, 456]

# ── sklearn note ─────────────────────────────────────────────
# sklearn does not have a global seed — every sklearn call that
# accepts random_state= is explicitly passed GLOBAL_SEED.
# This is enforced by convention throughout the notebook.

def set_seed(seed: int) -> None:
    """
    Set all random seeds for best-effort reproducibility.

    Covers 7 sources of randomness:
      1. os.environ PYTHONHASHSEED  — Python hash randomisation (affects child processes)
      2. Python random              — stdlib random, used by some albumentations internals
      3. NumPy                      — array ops, albumentations augmentation sampling
      4. PyTorch CPU                — weight initialisation, CPU op sampling
      5. PyTorch CUDA               — GPU kernel launches, atomic operations
      6. cuDNN deterministic        — forces same conv algorithm across runs
      7. cuDNN benchmark off        — disables profiling-based algorithm selection

    Note: torch.use_deterministic_algorithms(warn_only=True) is used rather than
    strict mode. This prints a warning if a non-deterministic CUDA kernel is
    encountered without crashing the notebook — appropriate for a submission where
    third-party layers may use ops without a deterministic implementation.

    Reproducibility guarantee: identical results on same hardware and CUDA version.
    Minor numerical variation across different GPU architectures is expected
    (standard in published DL research — see Pham et al., 2020).
    """
    # 1. Python hash seed (effective for worker subprocesses spawned after this call)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 1b. cuBLAS workspace config -- required by PyTorch when
    # torch.use_deterministic_algorithms(True) is enabled on CUDA >= 10.2.
    # Without this, deterministic CUDA matmul ops fall back to non-deterministic
    # behaviour under warn_only=True. Cost: ~8 MB GPU memory.
    # Ref: https://pytorch.org/docs/stable/notes/randomness.html
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    # 2. Python stdlib random
    random.seed(seed)

    # 3. NumPy
    np.random.seed(seed)

    # 4. PyTorch CPU
    torch.manual_seed(seed)

    # 5. PyTorch CUDA (all devices)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 6 & 7. cuDNN determinism — disable benchmark auto-selection
    #   benchmark=False: no cost for fixed input shapes (all patches are 224x224)
    #   deterministic=True: same CUDA kernel chosen every run
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

    # Soft determinism enforcement — warns without crashing if a non-deterministic
    # op is unavoidably triggered by a third-party layer.
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """
    DataLoader worker initialisation function.

    Each DataLoader worker is a separate subprocess with its own independent
    RNG state. Without this function, shuffling and augmentation in workers
    varies between runs even when the main process is seeded.

    worker_seed is derived from torch.initial_seed() which itself is determined
    by the Generator passed to the DataLoader — ensuring full traceability.

    Usage: pass as worker_init_fn= to every DataLoader in this notebook.
    """
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader_generator(seed: int) -> torch.Generator:
    """
    Create a seeded torch.Generator for DataLoader batch sampling.

    Ensures deterministic batch ordering and shuffling within the DataLoader.
    A separate generator per DataLoader call prevents cross-contamination
    between training and evaluation loaders.

    Usage: pass as generator= to every DataLoader in this notebook.
    """
    g = torch.Generator()
    g.manual_seed(seed)
    return g


# ── Apply global seed immediately ────────────────────────────
set_seed(GLOBAL_SEED)

# ── Verification: confirm identical outputs for same seed ────
# Calls set_seed(99) twice and checks that Python, NumPy, and
# PyTorch all produce identical values — proves the function works.
_s = 99
set_seed(_s)
_v1 = (random.random(), float(np.random.rand()), float(torch.rand(1)))
set_seed(_s)
_v2 = (random.random(), float(np.random.rand()), float(torch.rand(1)))
assert _v1 == _v2, f"Seed verification FAILED: {_v1} != {_v2}"
del _s, _v1, _v2

# ── Restore global seed after verification ───────────────────
set_seed(GLOBAL_SEED)

# ── Update metadata with seed information ────────────────────
METADATA['reproducibility'] = {
    'global_seed'       : GLOBAL_SEED,
    'experiment_seeds'  : EXPERIMENT_SEEDS,
    'cudnn_deterministic': torch.backends.cudnn.deterministic,
    'cudnn_benchmark'   : torch.backends.cudnn.benchmark,
    'guarantee'         : 'Best-effort: identical on same hardware/CUDA version. '
                          'Minor variation expected across GPU architectures.',
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

# ── Summary ──────────────────────────────────────────────────
print_section("Step 0.3 — Reproducibility Configuration")
print(f"  Global seed              : {GLOBAL_SEED}")
print(f"  Experiment seeds         : {EXPERIMENT_SEEDS}")
print(f"  cudnn.deterministic      : {torch.backends.cudnn.deterministic}")
print(f"  cudnn.benchmark          : {torch.backends.cudnn.benchmark}")
print(f"  PYTHONHASHSEED           : {os.environ.get('PYTHONHASHSEED', 'not set')}")
print(f"  Deterministic algorithms : warn_only=True (soft enforcement)")
print(f"  Seed verification        : PASSED")
print(f"  Metadata updated         : {metadata_path}")
print(f"\n  Functions available:")
print(f"    set_seed(seed)              — call before each experiment")
print(f"    seed_worker(worker_id)      — pass as worker_init_fn= to DataLoader")
print(f"    make_loader_generator(seed) — pass as generator= to DataLoader")
print(f"\n[OK] Step 0.3 complete.")


  Step 0.3 — Reproducibility Configuration
  Global seed              : 42
  Experiment seeds         : [42, 123, 456]
  cudnn.deterministic      : True
  cudnn.benchmark          : False
  PYTHONHASHSEED           : 42
  Deterministic algorithms : warn_only=True (soft enforcement)
  Seed verification        : PASSED
  Metadata updated         : /content/strawberry_project/metadata.json

  Functions available:
    set_seed(seed)              — call before each experiment
    seed_worker(worker_id)      — pass as worker_init_fn= to DataLoader
    make_loader_generator(seed) — pass as generator= to DataLoader

[OK] Step 0.3 complete.


### Step 0.4 — GPU Verification & Hardware Logging

Validates GPU availability, reports detailed hardware specifications, and performs a CUDA context warm-up before any training. Hardware details are persisted to `metadata.json` for reproducibility reporting. A `RECOMMENDED_BATCH_SIZE` global is derived from available VRAM and consumed by Phase 6 training loops.

In [16]:
# ============================================================
# STEP 0.4 — GPU VERIFICATION & HARDWARE LOGGING
# ============================================================

# ── CUDA context warm-up ─────────────────────────────────────
# A small tensor is pushed to the device before any timing or
# memory queries. CUDA context initialisation (driver load +
# kernel JIT) takes 2-5 s on first use and would otherwise
# skew timing benchmarks in Phases 6-8.
# ------------------------------------------------------------
_warmup_start = time.time()
_ = torch.zeros(1).to(DEVICE)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
_warmup_s = time.time() - _warmup_start

print_section("Step 0.4 — GPU Verification & Hardware Logging")

# ── CPU / system info ────────────────────────────────────────
cpu_count_logical = os.cpu_count()
cpu_count_physical = None
sys_ram_gb = None
try:
    import psutil
    cpu_count_physical = psutil.cpu_count(logical=False)
    sys_ram_gb = psutil.virtual_memory().total / 1e9
except ImportError:
    pass  # psutil not critical — skip gracefully

print(f"  CPU logical cores        : {cpu_count_logical}")
if cpu_count_physical is not None:
    print(f"  CPU physical cores       : {cpu_count_physical}")
if sys_ram_gb is not None:
    print(f"  System RAM               : {sys_ram_gb:.1f} GB")

hardware_info = {
    'device': str(DEVICE),
    'cpu_cores_logical': cpu_count_logical,
    'cpu_cores_physical': cpu_count_physical,
    'sys_ram_gb': round(sys_ram_gb, 1) if sys_ram_gb is not None else None,
    'cuda_available': torch.cuda.is_available(),
}

# ── GPU info ─────────────────────────────────────────────────
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)

    # mem_get_info(device) -> (free_bytes, total_bytes); added in PyTorch 1.10
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    free_vram_gb  = free_bytes  / 1e9
    total_vram_gb = total_bytes / 1e9
    used_vram_gb  = total_vram_gb - free_vram_gb

    compute_cap = f"{props.major}.{props.minor}"

    # Compute capability → architecture name mapping (Colab-relevant GPUs)
    _CC_NAMES = {
        '6.0': 'Pascal (P100)',     '7.0': 'Volta (V100)',
        '7.5': 'Turing (T4)',       '8.0': 'Ampere (A100)',
        '8.6': 'Ampere (A10G)',     '8.9': 'Ada Lovelace (L4)',
        '9.0': 'Hopper (H100)',
    }
    arch_name = _CC_NAMES.get(compute_cap, f'Unknown (CC {compute_cap})')

    # Minimum viable VRAM for ConvNeXt-Tiny + AMP at batch_size=32 on 224x224 patches.
    # Below this, even AMP + gradient checkpointing struggles for the mixed E5 setup.
    _MIN_VRAM_GB = 6.0
    vram_ok = total_vram_gb >= _MIN_VRAM_GB

    # Batch size recommendation based on measured total VRAM.
    # Tuned for ConvNeXt-Tiny + AMP fp16 at 224x224 with LLRD fine-tuning.
    # Initial backbone-freeze phase uses far less memory; values below are safe
    # for the unfrozen LLRD stage (the memory-peak phase).
    if   total_vram_gb >= 14: rec_batch = 64
    elif total_vram_gb >= 10: rec_batch = 48
    elif total_vram_gb >=  6: rec_batch = 32
    else:                     rec_batch = 16

    print(f"\n  GPU name                 : {props.name}")
    print(f"  Architecture             : {arch_name}")
    print(f"  Compute capability       : {compute_cap}")
    print(f"  Multiprocessors          : {props.multi_processor_count}")
    print(f"  Warp size                : {props.warp_size}")
    print(f"  Total VRAM               : {total_vram_gb:.2f} GB")
    print(f"  Used VRAM (post-warmup)  : {used_vram_gb:.2f} GB")
    print(f"  Free VRAM                : {free_vram_gb:.2f} GB")
    print(f"  VRAM sufficient (>={_MIN_VRAM_GB:.0f}GB)  : {'YES' if vram_ok else 'NO — switch to T4'}")
    # Tensor Core capability (informs Phase 6 AMP strategy)
    _cc_major = props.major
    _supports_fp16_tc = _cc_major >= 7                       # Volta/Turing+ (V100, T4, A100, ...)
    _supports_bf16    = _cc_major >= 8                       # Ampere+ (A100, A10, L4, ...)
    _supports_tf32    = _cc_major >= 8                       # Ampere+ (matmul fast path)

    print(f"  CUDA version             : {torch.version.cuda}")
    print(f"  cuDNN version            : {torch.backends.cudnn.version()}")
    print(f"  fp16 Tensor Cores        : {'YES' if _supports_fp16_tc else 'NO'}")
    print(f"  BF16 / TF32 support      : {'YES' if _supports_bf16 else 'NO'}")
    print(f"  Context warm-up time     : {_warmup_s:.2f} s")
    print(f"  Recommended batch size   : {rec_batch}  (for {PATCH_SIZE}x{PATCH_SIZE} patches)")

    if not vram_ok:
        print(f"\n  [WARNING] VRAM ({total_vram_gb:.1f} GB) below minimum ({_MIN_VRAM_GB} GB).")
        print(f"            Go to Runtime > Change runtime type > T4 GPU.")

    hardware_info.update({
        'gpu_name'              : props.name,
        'gpu_architecture'      : arch_name,
        'compute_capability'    : compute_cap,
        'multi_processor_count' : props.multi_processor_count,
        'warp_size'             : props.warp_size,
        'total_vram_gb'         : round(total_vram_gb, 2),
        'free_vram_gb'          : round(free_vram_gb,  2),
        'cuda_version'          : torch.version.cuda,
        'cudnn_version'         : torch.backends.cudnn.version(),
        'warmup_time_s'         : round(_warmup_s, 3),
        'vram_sufficient'       : vram_ok,
        'recommended_batch_size': rec_batch,
        'supports_fp16_tensor_cores': _supports_fp16_tc,
        'supports_bf16_tf32'       : _supports_bf16,
    })

    # Expose as a global so Phase 6 training loops can consume it automatically
    RECOMMENDED_BATCH_SIZE = rec_batch

else:
    print(f"\n  [WARNING] Running on CPU — GPU not available.")
    print(f"            Training will be very slow (hours, not minutes).")
    print(f"            Enable T4 GPU via Runtime > Change runtime type.")
    print(f"  Context warm-up time     : {_warmup_s:.4f} s")
    RECOMMENDED_BATCH_SIZE = 16   # conservative CPU-safe fallback

print(f"\n  RECOMMENDED_BATCH_SIZE (global) : {RECOMMENDED_BATCH_SIZE}")

# ── Persist to metadata.json ─────────────────────────────────
METADATA['hardware'] = hardware_info
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f"  Metadata updated         : {metadata_path}")
print(f"\n[OK] Step 0.4 complete.")


  Step 0.4 — GPU Verification & Hardware Logging
  CPU logical cores        : 2
  CPU physical cores       : 1
  System RAM               : 13.6 GB

  GPU name                 : Tesla T4
  Architecture             : Turing (T4)
  Compute capability       : 7.5
  Multiprocessors          : 40
  Warp size                : 32
  Total VRAM               : 15.64 GB
  Used VRAM (post-warmup)  : 0.13 GB
  Free VRAM                : 15.51 GB
  VRAM sufficient (>=4GB)  : YES
  CUDA version             : 12.8
  cuDNN version            : 91002
  Context warm-up time     : 0.00 s
  Recommended batch size   : 64  (for 224x224 patches)

  RECOMMENDED_BATCH_SIZE (global) : 64
  Metadata updated         : /content/strawberry_project/metadata.json

[OK] Step 0.4 complete.


---

## Phase 1 — Data Acquisition & Integrity Validation

Downloads the Pastell & Liakka strawberry dataset from Zenodo, verifies MD5 integrity, extracts the archive, and audits the file structure. All operations are idempotent — re-running the notebook skips steps already completed.

### Step 1.1 — Dataset Download, MD5 Verification & Extraction

Streams `strawberries.zip` (~1.49 GB) from Zenodo with a live `tqdm` progress bar, computing MD5 on-the-fly during download (single pass, zero extra memory). Extraction is skipped if the archive already exists with a valid MD5.

In [17]:
# ============================================================
# STEP 1.1 — DATASET DOWNLOAD, MD5 VERIFICATION & EXTRACTION
# ============================================================
import requests

print_section('Step 1.1 — Dataset Download & Extraction')


# ── Helper: compute MD5 of a file ───────────────────────────
def md5_of_file(path, chunk=8 * 1024 * 1024):
    # Read file in chunks to avoid loading 1.49 GB into RAM
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            buf = f.read(chunk)
            if not buf:
                break
            h.update(buf)
    return h.hexdigest()


# ── Guard 1: skip download if archive already valid ──────────
_download_needed = True
if ARCHIVE_PATH.exists():
    print(f'  Archive found at: {ARCHIVE_PATH}')
    print(f'  Verifying MD5...')
    _existing_md5 = md5_of_file(ARCHIVE_PATH)
    if _existing_md5 == ZENODO_MD5:
        print(f'  MD5 match — download skipped. ({_existing_md5})')
        _download_needed = False
    else:
        print(f'  MD5 mismatch ({_existing_md5[:12]}...) — re-downloading.')
        ARCHIVE_PATH.unlink()


# ── Download: streaming with on-the-fly MD5 + 3-attempt retry ──
# Zenodo occasionally returns 503/504 under load -- retry transparently.
# Backoff: 5s, 15s, 45s (exponential).
if _download_needed:
    print(f'  Downloading: {ZENODO_URL}')
    print(f'  Destination: {ARCHIVE_PATH}')

    _MAX_ATTEMPTS = 3
    _last_err = None
    for _attempt in range(1, _MAX_ATTEMPTS + 1):
        try:
            _hash = hashlib.md5()
            _resp = requests.get(ZENODO_URL, stream=True, timeout=300)
            _resp.raise_for_status()
            _total = int(_resp.headers.get('content-length', 0))

            _t0 = time.time()
            with open(ARCHIVE_PATH, 'wb') as _fout:
                with tqdm(total=_total, unit='B', unit_scale=True, unit_divisor=1024,
                          desc=f'  strawberries.zip (attempt {_attempt}/{_MAX_ATTEMPTS})',
                          ncols=80) as _bar:
                    for _chunk in _resp.iter_content(chunk_size=8 * 1024 * 1024):
                        _fout.write(_chunk)
                        _hash.update(_chunk)
                        _bar.update(len(_chunk))

            _download_s = time.time() - _t0
            _downloaded_md5 = _hash.hexdigest()
            _size_gb = ARCHIVE_PATH.stat().st_size / 1e9

            print(f'\n  Downloaded    : {_size_gb:.2f} GB in {_download_s:.1f} s')
            print(f'  Computed MD5  : {_downloaded_md5}')
            print(f'  Expected MD5  : {ZENODO_MD5}')

            if _downloaded_md5 != ZENODO_MD5:
                raise RuntimeError(
                    f'MD5 mismatch -- got {_downloaded_md5}, expected {ZENODO_MD5}'
                )
            print(f'  [OK] MD5 verified.')
            _last_err = None
            break  # success
        except (requests.exceptions.RequestException, RuntimeError) as _e:
            _last_err = _e
            if ARCHIVE_PATH.exists():
                ARCHIVE_PATH.unlink()
            if _attempt < _MAX_ATTEMPTS:
                _wait_s = 5 * (3 ** (_attempt - 1))   # 5, 15, 45
                print(f'\n  [WARN] Attempt {_attempt} failed: {_e}')
                print(f'         Retrying in {_wait_s}s...')
                time.sleep(_wait_s)
            else:
                print(f'\n  [ERROR] All {_MAX_ATTEMPTS} download attempts failed.')

    if _last_err is not None:
        raise RuntimeError(
            f'Download failed after {_MAX_ATTEMPTS} attempts. Last error: {_last_err}\n'
            f'  Check network connection and retry the cell.'
        )


# ── Guard 2: skip extraction if already done ─────────────────
_extract_needed = not DATASET_DIR.exists() or not any(DATASET_DIR.iterdir())

if _extract_needed:
    print(f'\n  Extracting to: {DIRS["raw"]}')
    _t0 = time.time()
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as _zf:
        _members = _zf.namelist()
        with tqdm(total=len(_members), desc='  Extracting', ncols=80, unit='file') as _bar:
            for _member in _members:
                _zf.extract(_member, DIRS['raw'])
                _bar.update(1)
    _extract_s = time.time() - _t0
    print(f'  Extracted {len(_members):,} files in {_extract_s:.1f} s')
else:
    print(f'\n  Extraction skipped — {DATASET_DIR} already populated.')


# ── Guard 3: Drive sync (only if USE_DRIVE_CACHE=True & no prior hit) ──
if USE_DRIVE_CACHE and DRIVE_PROJECT_DIR is not None and not CACHE_HIT:
    import shutil
    _drive_raw = DRIVE_PROJECT_DIR / 'data' / 'raw'
    _drive_raw.mkdir(parents=True, exist_ok=True)
    _drive_dataset = _drive_raw / 'strawberries'
    if not _drive_dataset.exists():
        print(f'\n  Syncing raw dataset to Drive: {_drive_dataset}')
        shutil.copytree(str(DATASET_DIR), str(_drive_dataset))
        print(f'  [OK] Drive sync complete.')
    else:
        print(f'\n  Drive dataset already exists — sync skipped.')


# ── Summary ──────────────────────────────────────────────────
_total_files = sum(1 for _ in DATASET_DIR.rglob('*') if _.is_file())
print(f'\n  Dataset directory: {DATASET_DIR}')
print(f'  Total files      : {_total_files:,}')

METADATA['dataset_download'] = {
    'zenodo_url'    : ZENODO_URL,
    'md5_verified'  : ZENODO_MD5,
    'archive_size_gb': round(ARCHIVE_PATH.stat().st_size / 1e9, 3),
    'total_files'   : _total_files,
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f'  Metadata updated : {metadata_path}')
print(f'\n[OK] Step 1.1 complete.')



  Step 1.1 — Dataset Download & Extraction
  Archive found at: /content/strawberry_project/data/raw/strawberries.zip
  Verifying MD5...
  MD5 match — download skipped. (db8d5dcb4b8adebf1621788373fd3031)

  Extraction skipped — /content/strawberry_project/data/raw/strawberries already populated.

  Dataset directory: /content/strawberry_project/data/raw/strawberries
  Total files      : 1,628
  Metadata updated : /content/strawberry_project/metadata.json

[OK] Step 1.1 complete.


### Step 1.2 — File Structure Audit & Annotation Validation

Discovers the extracted directory layout, counts image/label file pairs, validates YOLO annotation coordinates, and tallies per-class annotation counts. Raises a `ValueError` if critical integrity invariants are violated. Results are logged to `metadata.json`.

In [18]:
# ============================================================
# STEP 1.2 — FILE STRUCTURE AUDIT & ANNOTATION VALIDATION
# ============================================================
from collections import Counter, defaultdict

print_section('Step 1.2 — File Structure Audit & Annotation Validation')


# ── 1. Locate / normalise image and label directories ────────
# Roboflow ships datasets pre-split into training/ and validation/.
# We flatten into a single images/ + labels/ pair (project does its
# own stratified split in Step 2.3).
import shutil

_img_dir = DATASET_DIR / 'images'
_lbl_dir = DATASET_DIR / 'labels'

# F1.2.b idempotency: only flatten if images/ is missing OR empty
#                     AND non-flat copies still exist somewhere.
def _has_files(d, exts):
    return d.exists() and any(d.glob(e) for e in exts)

_img_exts = ('*.jpg', '*.jpeg', '*.png')
_needs_img_flatten = (
    not _has_files(_img_dir, _img_exts)
    and any(p for ext in _img_exts for p in DATASET_DIR.rglob(ext)
            if p.parent != _img_dir)
)
_needs_lbl_flatten = (
    not _has_files(_lbl_dir, ('*.txt',))
    and any(p for p in DATASET_DIR.rglob('*.txt')
            if p.parent != _lbl_dir and 'README' not in p.name)
)

if _needs_img_flatten:
    print('  [INFO] Flattening image splits into images/ ...')
    _img_dir.mkdir(parents=True, exist_ok=True)
    _seen = {}   # stem -> destination path (for collision detection)
    for ext in _img_exts:
        for p in DATASET_DIR.rglob(ext):
            if p.parent == _img_dir:
                continue
            if p.stem in _seen:
                # Collision: same filename in two splits (e.g. training/110.jpg & validation/110.jpg)
                # Prefix with parent split folder name to keep both images
                _split_name = p.parent.name  # e.g. 'training' or 'validation'
                _new_name = f'{_split_name}_{p.name}'
                print(f'  [INFO] Collision on {p.name!r} — renaming to {_new_name}')
                shutil.move(str(p), str(_img_dir / _new_name))
            else:
                _seen[p.stem] = _img_dir / p.name
                shutil.move(str(p), str(_img_dir / p.name))
    del _seen

if _needs_lbl_flatten:
    print('  [INFO] Flattening label splits into labels/ ...')
    _lbl_dir.mkdir(parents=True, exist_ok=True)
    _seen = {}
    for p in DATASET_DIR.rglob('*.txt'):
        if p.parent == _lbl_dir or 'README' in p.name:
            continue
        if p.stem in _seen:
            _split_name = p.parent.name
            _new_name = f'{_split_name}_{p.name}'
            print(f'  [INFO] Label collision on {p.name!r} — renaming to {_new_name}')
            shutil.move(str(p), str(_lbl_dir / _new_name))
        else:
            _seen[p.stem] = _lbl_dir / p.name
            shutil.move(str(p), str(_lbl_dir / p.name))
    del _seen

# F1.2.d: remove now-empty training/ and validation/ directories
for _stale in ('training', 'validation', 'train', 'val', 'test'):
    _stale_dir = DATASET_DIR / _stale
    if _stale_dir.exists() and _stale_dir.is_dir():
        try:
            # Remove empty subdirs first (e.g. training/labels)
            for _sub in sorted(_stale_dir.rglob('*'), reverse=True):
                if _sub.is_dir() and not any(_sub.iterdir()):
                    _sub.rmdir()
            if not any(_stale_dir.iterdir()):
                _stale_dir.rmdir()
        except OSError:
            pass   # leave non-empty dirs alone

# F1.2.a: expose canonical globals (referenced by sections below + Phase 2)
IMG_DIR = _img_dir
LBL_DIR = _lbl_dir

# F1.2.e: print directory layout AFTER normalisation so the audit trail
# reflects the final state the rest of the pipeline will see.
_top_dirs = sorted(set(
    p.parent for p in DATASET_DIR.rglob('*') if p.is_file()
))
print('  Final directory layout:')
for _d in _top_dirs:
    _nf = sum(1 for _ in _d.glob('*') if _.is_file())
    print(f'    {_d.relative_to(BASE_DIR)}  ({_nf} files)')


# ── 2. Discover image / label file inventories ───────────────
_img_files = sorted(IMG_DIR.glob('*.jpg')) + sorted(IMG_DIR.glob('*.jpeg')) + sorted(IMG_DIR.glob('*.png'))
_img_stems = {p.stem: p for p in _img_files}

_lbl_files = sorted(LBL_DIR.glob('*.txt'))
_lbl_stems = {p.stem: p for p in _lbl_files}

_paired   = set(_img_stems) & set(_lbl_stems)  # has both image and label
_img_only = set(_img_stems) - set(_lbl_stems)  # image but no label file
_lbl_only = set(_lbl_stems) - set(_img_stems)  # label but no image

print(f'\n  Total image files   : {len(_img_stems):>6,}')
print(f'  Total label files   : {len(_lbl_stems):>6,}')
print(f'  Paired (img+lbl)    : {len(_paired):>6,}')
print(f'  Image only (no lbl) : {len(_img_only):>6,}')
print(f'  Label only (no img) : {len(_lbl_only):>6,}')

if _lbl_only:
    print(f'  [WARNING] {len(_lbl_only)} orphan label files (no matching image).')


# ── 4. Parse all annotations ──────────────────────────────────
# Count per-class annotations; flag invalid coordinate ranges
_class_counter   = Counter()  # class_id -> total annotation count
_empty_labels    = []         # stems with 0-line label files
_invalid_annots  = []         # (stem, line_idx, reason)
_valid_image_classes = defaultdict(set)  # stem -> set of class ids present

for _stem in tqdm(_paired, desc='  Parsing labels', ncols=80):
    _lp = _lbl_stems[_stem]
    _lines = _lp.read_text().strip().splitlines()

    if not _lines:
        _empty_labels.append(_stem)
        continue

    for _li, _line in enumerate(_lines):
        _parts = _line.strip().split()
        if len(_parts) != 5:
            _invalid_annots.append((_stem, _li, f'expected 5 fields, got {len(_parts)}'))
            continue
        try:
            _cls, _cx, _cy, _bw, _bh = int(_parts[0]), float(_parts[1]), float(_parts[2]), float(_parts[3]), float(_parts[4])
        except ValueError:
            _invalid_annots.append((_stem, _li, 'non-numeric field'))
            continue

        # Coordinate validation: all values must be in (0, 1]
        if not (0 < _cx <= 1 and 0 < _cy <= 1 and 0 < _bw <= 1 and 0 < _bh <= 1):
            _invalid_annots.append((_stem, _li, f'coords out of range: cx={_cx:.3f} cy={_cy:.3f} w={_bw:.3f} h={_bh:.3f}'))
            continue

        _class_counter[_cls] += 1
        _valid_image_classes[_stem].add(_cls)


# ── 5. Report annotation statistics ──────────────────────────
_CLASS_LABELS = {0: 'ripe', 1: 'unripe', 2: 'peduncle (excluded)'}
print(f'\n  Annotation counts by class:')
_total_annots = sum(_class_counter.values())
for _cid in sorted(_class_counter):
    _pct = 100 * _class_counter[_cid] / _total_annots
    _lbl = _CLASS_LABELS.get(_cid, f'class_{_cid}')
    print(f'    Class {_cid} ({_lbl:<22}) : {_class_counter[_cid]:>5,}  ({_pct:.1f}%)')
print(f'    Total                         : {_total_annots:>5,}')

# In-scope annotations (classes 0 and 1 only)
_inscope_annots = _class_counter[0] + _class_counter[1]
_inscope_images = sum(1 for s in _valid_image_classes
                      if _valid_image_classes[s] & {0, 1})
print(f'\n  In-scope annotations (ripe+unripe) : {_inscope_annots:,}')
print(f'  Images with in-scope annotations   : {_inscope_images:,}')
print(f'  Empty label files                  : {len(_empty_labels):,}')
print(f'  Invalid annotation lines            : {len(_invalid_annots):,}')

if _invalid_annots:
    print(f'  [WARNING] First 5 invalid annotations:')
    for _x in _invalid_annots[:5]:
        print(f'    {_x}')


# ── 6. Integrity check — hard stop if critical invariants fail ─
if len(_img_stems) == 0:
    raise ValueError('AUDIT FAILED: No image files found. Check extraction.')
if _inscope_images < 100:
    raise ValueError(f'AUDIT FAILED: Only {_inscope_images} in-scope images — too few to split.')
if _class_counter.get(0, 0) == 0 or _class_counter.get(1, 0) == 0:
    raise ValueError('AUDIT FAILED: One or both in-scope classes (ripe/unripe) have 0 annotations.')


# ── 7. Expose globals for Phase 2 ────────────────────────────
# IMG_DIR, LBL_DIR already set above
VALID_IMAGE_CLASSES = dict(_valid_image_classes)  # stem -> set of class ids
ALL_IMAGE_STEMS = sorted(_img_stems.keys())


# ── 8. Update metadata ────────────────────────────────────────
METADATA['file_audit'] = {
    'img_dir'           : str(IMG_DIR),
    'lbl_dir'           : str(LBL_DIR),
    'total_images'      : len(_img_stems),
    'total_labels'      : len(_lbl_stems),
    'paired_count'      : len(_paired),
    'img_only_count'    : len(_img_only),
    'lbl_only_count'    : len(_lbl_only),
    'empty_label_count' : len(_empty_labels),
    'invalid_annot_count': len(_invalid_annots),
    'annotations_by_class': {str(k): v for k, v in _class_counter.items()},
    'total_annotations' : _total_annots,
    'inscope_annotations': _inscope_annots,
    'inscope_images'    : _inscope_images,
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f'\n  Metadata updated : {metadata_path}')
print(f'\n  Globals set: IMG_DIR, LBL_DIR, VALID_IMAGE_CLASSES, ALL_IMAGE_STEMS')
print(f'\n[OK] Step 1.2 complete — dataset integrity verified.')



  Step 1.2 — File Structure Audit & Annotation Validation
  Extracted directory layout:
    data/raw/strawberries  (2 files)
    data/raw/strawberries/training  (1308 files)
    data/raw/strawberries/validation  (318 files)


ValueError: Cannot resolve images directory. Candidates: [PosixPath('/content/strawberry_project/data/raw/strawberries/validation'), PosixPath('/content/strawberry_project/data/raw/strawberries/training')]

---

## Phase 2 — Data Pipeline & Patch Extraction

Converts raw YOLO object-detection annotations into a patch classification dataset. Each bounding box annotation (classes 0=ripe, 1=unripe only; peduncle excluded) is cropped from its source image with 10% padding, resized to 224×224 via Lanczos, and saved as JPEG. The resulting patch pool is then split with stratified image-level sampling to prevent data leakage.

### Step 2.1 — YOLO Annotation Parser & Patch Extraction

Parses every valid YOLO annotation (classes 0 and 1 only), applies 10% bounding-box padding, clamps to image boundaries, filters patches below `MIN_PATCH_PX`, and saves 224×224 JPEG patches. All patches are saved to a temporary flat pool; the train/val/test split is applied in Step 2.3.

In [ ]:
# ============================================================
# STEP 2.1 — YOLO ANNOTATION PARSER & PATCH EXTRACTION
# ============================================================
print_section('Step 2.1 — YOLO Parser & Patch Extraction')


# ── Guard: skip if CACHE_HIT or local re-run with valid manifest ──
# When USE_DRIVE_CACHE=True and Drive had full patch data, Phase 2 is a no-op.
# Also skip if patch_pool.csv already exists locally and matches disk state
# (F2.1.d idempotency: avoids ~30 s re-extraction when re-running cells).
_patch_pool      = DIRS['patches'] / 'pool'
_patch_csv_local = DIRS['splits'] / 'patch_pool.csv'

_local_idempotent_skip = False
if not CACHE_HIT and _patch_csv_local.exists() and _patch_pool.exists():
    try:
        _existing_df = pd.read_csv(_patch_csv_local)
        _disk_patches = sum(1 for _ in _patch_pool.rglob('*.jpg'))
        if len(_existing_df) > 0 and _disk_patches >= len(_existing_df):
            _local_idempotent_skip = True
            print(f'  [LOCAL IDEMPOTENT] {_disk_patches:,} patches on disk match manifest ({len(_existing_df):,} rows).')
            print('  Re-extraction skipped. Delete patch_pool.csv to force a rebuild.')
    except Exception:
        pass  # corrupt csv -> rebuild

if CACHE_HIT or _local_idempotent_skip:
    print('  Scanning existing patches...')
    _all_patches = list(_patch_pool.rglob('*.jpg')) if _patch_pool.exists() else []
    print(f'  Found {len(_all_patches):,} patches in pool.')
else:

    _patch_pool.mkdir(parents=True, exist_ok=True)

    def yolo_to_pixel(cx, cy, bw, bh, W, H, padding=PATCH_PADDING):
        # Convert normalised YOLO box to pixel coords with padding.
        # Returns (x1, y1, x2, y2) clamped to image boundaries.
        # Design note: subsequent crop().resize((PATCH_SIZE, PATCH_SIZE)) intentionally
        # forces a square aspect ratio. Strawberries are roughly elliptical, so the
        # ~10-20% aspect distortion is acceptable and is what ImageNet-pretrained
        # CNNs (which were trained on 224x224 squares) actually expect.
        px_cx, px_cy = cx * W, cy * H
        px_bw, px_bh = bw * W, bh * H
        pad_x = px_bw * padding
        pad_y = px_bh * padding
        x1 = max(0, int(px_cx - px_bw / 2 - pad_x))
        y1 = max(0, int(px_cy - px_bh / 2 - pad_y))
        x2 = min(W, int(px_cx + px_bw / 2 + pad_x))
        y2 = min(H, int(px_cy + px_bh / 2 + pad_y))
        return x1, y1, x2, y2

    _patch_counter   = Counter()  # class_id -> patches extracted
    _skipped_size    = 0          # patches too small
    _skipped_class   = 0          # class 2 (peduncle) skipped
    _errors          = 0          # read/crop errors

    # Store patch metadata for split assignment in Step 2.3
    # Each entry: {'patch_path': str, 'class_id': int, 'source_stem': str}
    _patch_records = []

    _img_stems_list = sorted(VALID_IMAGE_CLASSES.keys())

    for _stem in tqdm(_img_stems_list, desc='  Extracting patches', ncols=80):
        _lp = LBL_DIR / (_stem + '.txt')
        if not _lp.exists():
            continue
        _lines = _lp.read_text().strip().splitlines()
        if not _lines:
            continue

        # Find the image file (try .jpg, .jpeg, .png)
        _ip = None
        for _ext in ('.jpg', '.jpeg', '.png'):
            _candidate = IMG_DIR / (_stem + _ext)
            if _candidate.exists():
                _ip = _candidate
                break
        if _ip is None:
            continue

        try:
            _img = Image.open(_ip).convert('RGB')
            _W, _H = _img.size
        except Exception:
            _errors += 1
            continue

        for _idx, _line in enumerate(_lines):
            _parts = _line.strip().split()
            if len(_parts) != 5:
                continue
            try:
                _cls = int(_parts[0])
                _cx, _cy, _bw, _bh = float(_parts[1]), float(_parts[2]), float(_parts[3]), float(_parts[4])
            except ValueError:
                continue

            # Skip peduncle (class 2) — out of project scope
            if _cls not in (0, 1):
                _skipped_class += 1
                continue

            # Convert to pixel coords with padding
            _x1, _y1, _x2, _y2 = yolo_to_pixel(_cx, _cy, _bw, _bh, _W, _H)
            _pw, _ph = _x2 - _x1, _y2 - _y1

            # Filter minimum size
            if _pw < MIN_PATCH_PX or _ph < MIN_PATCH_PX:
                _skipped_size += 1
                continue

            # Crop and resize
            try:
                _patch = _img.crop((_x1, _y1, _x2, _y2)).resize(
                    (PATCH_SIZE, PATCH_SIZE), Image.LANCZOS
                )
            except Exception:
                _errors += 1
                continue

            # Save to pool directory — class subfolder
            _cls_dir = _patch_pool / CLASS_NAMES[_cls]
            _cls_dir.mkdir(exist_ok=True)
            _patch_name = f'{_stem}_{_idx:04d}.jpg'
            _out_path = _cls_dir / _patch_name
            _patch.save(str(_out_path), 'JPEG', quality=95)

            _patch_counter[_cls] += 1
            _patch_records.append({
                'patch_path'  : str(_out_path.relative_to(BASE_DIR)),
                'class_id'    : _cls,
                'class_name'  : CLASS_NAMES[_cls],
                'source_stem' : _stem,
                'annot_idx'   : _idx,
            })

    # Save patch records as CSV for Step 2.3 (stratified split)
    _patch_df = pd.DataFrame(_patch_records)
    _patch_csv = DIRS['splits'] / 'patch_pool.csv'
    _patch_df.to_csv(_patch_csv, index=False)

    # Report
    _total_extracted = sum(_patch_counter.values())
    print(f'\n  Patches extracted by class:')
    for _cid in sorted(_patch_counter):
        print(f'    Class {_cid} ({CLASS_NAMES[_cid]:<8}) : {_patch_counter[_cid]:>5,}')
    print(f'    Total               : {_total_extracted:>5,}')
    print(f'\n  Skipped (class 2 peduncle) : {_skipped_class:,}')
    print(f'  Skipped (< {MIN_PATCH_PX}px)          : {_skipped_size:,}')
    print(f'  Errors (read/crop)         : {_errors:,}')
    print(f'  Patch records CSV          : {_patch_csv}')

    METADATA['patch_extraction'] = {
        'patch_pool_dir'    : str(_patch_pool),
        'total_patches'     : _total_extracted,
        'patches_by_class'  : {CLASS_NAMES[k]: v for k, v in _patch_counter.items()},
        'skipped_peduncle'  : _skipped_class,
        'skipped_too_small' : _skipped_size,
        'errors'            : _errors,
        'patch_size'        : PATCH_SIZE,
        'padding'           : PATCH_PADDING,
        'min_patch_px'      : MIN_PATCH_PX,
        'jpeg_quality'      : 95,
    }
    with open(metadata_path, 'w') as f:
        json.dump(METADATA, f, indent=2)

print(f'\n  Metadata updated : {metadata_path}')
print(f'\n[OK] Step 2.1 complete.')


### Step 2.2 — Class Imbalance Analysis

Loads the patch pool manifest, computes the ripe/unripe class distribution and imbalance ratio, derives balanced class weights using `sklearn.utils.class_weight.compute_class_weight`, and saves a class distribution bar chart. The `CLASS_WEIGHTS` tensor is used by Phase 6 loss functions.

In [ ]:
# ============================================================
# STEP 2.2 — CLASS IMBALANCE ANALYSIS
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

print_section('Step 2.2 — Class Imbalance Analysis')


# ── Load patch pool manifest ──────────────────────────────────
_patch_csv = DIRS['splits'] / 'patch_pool.csv'
PATCH_DF = pd.read_csv(_patch_csv)
print(f'  Total patches loaded : {len(PATCH_DF):,}')
print(f'  Columns              : {list(PATCH_DF.columns)}')


# ── Class distribution ────────────────────────────────────────
_counts = PATCH_DF['class_id'].value_counts().sort_index()
_n_ripe   = _counts.get(0, 0)
_n_unripe = _counts.get(1, 0)
_n_total  = _n_ripe + _n_unripe
_imbalance_ratio = max(_n_ripe, _n_unripe) / max(min(_n_ripe, _n_unripe), 1)

print(f'\n  Class distribution:')
print(f'    Ripe   (class 0) : {_n_ripe:>5,}  ({100*_n_ripe/_n_total:.1f}%)')
print(f'    Unripe (class 1) : {_n_unripe:>5,}  ({100*_n_unripe/_n_total:.1f}%)')
print(f'    Imbalance ratio  : {_imbalance_ratio:.2f}:1')

if _imbalance_ratio > 1.5:
    print(f'  [NOTE] Imbalance ratio > 1.5 — class weighting will be applied.')
else:
    print(f'  [NOTE] Classes roughly balanced — weighting applied anyway for safety.')


# ── Compute balanced class weights ────────────────────────────
# Formula: n_samples / (n_classes * bincount(y))
# Same formula used for SVM class_weight='balanced' — ensures fair comparison.
_y_all = PATCH_DF['class_id'].values
_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=_y_all
)
CLASS_WEIGHTS = torch.tensor(_weights_np, dtype=torch.float32).to(DEVICE)

print(f'\n  Balanced class weights (sklearn):')
for _cid, _w in enumerate(_weights_np):
    print(f'    {CLASS_NAMES[_cid]:<8} (class {_cid}) : {_w:.4f}')
print(f'  CLASS_WEIGHTS tensor -> {DEVICE}')


# ── Plot class distribution bar chart ────────────────────────
_fig, _ax = plt.subplots(figsize=(5, 4))
_bars = _ax.bar(
    CLASS_NAMES, [_n_ripe, _n_unripe],
    color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=0.8, width=0.5
)
for _bar, _cnt in zip(_bars, [_n_ripe, _n_unripe]):
    _ax.text(
        _bar.get_x() + _bar.get_width() / 2,
        _bar.get_height() + 20,
        f'{_cnt:,}', ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
_ax.set_title('Patch Pool — Class Distribution', fontsize=13, fontweight='bold')
_ax.set_ylabel('Number of Patches')
_ax.set_xlabel('Class')
_ax.set_ylim(0, max(_n_ripe, _n_unripe) * 1.15)
sns.despine(ax=_ax)
_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'class_distribution.png', dpi=300)
plt.show()
print(f'  Figure saved → {DIRS["figures"] / "class_distribution.png"}')


# ── Update metadata ───────────────────────────────────────────
METADATA['class_distribution'] = {
    'n_ripe'          : int(_n_ripe),
    'n_unripe'        : int(_n_unripe),
    'imbalance_ratio' : round(float(_imbalance_ratio), 3),
    'class_weights'   : {CLASS_NAMES[i]: round(float(w), 4) for i, w in enumerate(_weights_np)},
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f'\n[OK] Step 2.2 complete.')


### Step 2.3 — Stratified Image-Level Split

Assigns each source image to train, val, or test at the image level — ensuring patches from the same field scene never appear in two different splits (prevents data leakage). The test set comprises `N_TEST_IMAGES=150` stratified images held out throughout the project. Patches are copied from the pool into ImageFolder-compatible directories.

In [ ]:
# ============================================================
# STEP 2.3 — STRATIFIED IMAGE-LEVEL SPLIT
# ============================================================
import shutil
from sklearn.model_selection import train_test_split

print_section('Step 2.3 — Stratified Image-Level Split')


# ── Guard: skip if CACHE_HIT (split already on disk) ─────────
if CACHE_HIT and SPLIT_CSV.exists():
    print('  [CACHE HIT] Loading split assignments from Drive cache.')
    SPLIT_DF = pd.read_csv(SPLIT_CSV)
    print(f'  Loaded {len(SPLIT_DF):,} patch assignments.')
else:

    set_seed(GLOBAL_SEED)  # Fix seed for reproducible split

    # ── Build image-level labels for stratification ───────────
    # Each image gets a 'dominant class' label for stratification.
    # If an image has both ripe and unripe patches, it gets label=2 (mixed).
    # Stratification preserves the mix ratio across splits.
    _image_classes_list = {}
    for _, _row in PATCH_DF.iterrows():
        _s = _row['source_stem']
        if _s not in _image_classes_list:
            _image_classes_list[_s] = set()
        _image_classes_list[_s].add(_row['class_id'])

    _stems = sorted(_image_classes_list.keys())
    _strat_labels = []
    for _s in _stems:
        _cls_set = _image_classes_list[_s]
        if _cls_set == {0}:
            _strat_labels.append(0)   # ripe-only
        elif _cls_set == {1}:
            _strat_labels.append(1)   # unripe-only
        else:
            _strat_labels.append(2)   # mixed

    _strat_arr = np.array(_strat_labels)
    _strat_counter = Counter(_strat_arr)
    print(f'  Image-level strat categories:')
    for _lbl, _name in [(0,'ripe-only'), (1,'unripe-only'), (2,'mixed')]:
        print(f'    {_name:<12} : {_strat_counter.get(_lbl, 0):>4} images')
    print(f'    Total        : {len(_stems):>4} images')

    # ── F2.3.c: defensive pre-flight before sklearn split ─────
    # Catches cryptic "least populated class has 1 member" errors with a clear message.
    if len(_stems) < N_TEST_IMAGES + 20:
        raise ValueError(
            f'AUDIT FAILED: only {len(_stems)} images available; need '
            f'>= {N_TEST_IMAGES + 20} (N_TEST_IMAGES + 20 train/val buffer).'
        )
    for _lbl, _name in [(0,'ripe-only'), (1,'unripe-only'), (2,'mixed')]:
        _cnt = _strat_counter.get(_lbl, 0)
        if 0 < _cnt < 2:
            raise ValueError(
                f'AUDIT FAILED: stratification category {_name!r} has only {_cnt} '
                f'image -- sklearn requires >= 2 per category. Consider merging '
                f'this category or relaxing stratification.'
            )

    # ── Step A: hold out N_TEST_IMAGES images as test set ────
    _test_frac = N_TEST_IMAGES / len(_stems)
    _trainval_stems, _test_stems, _trainval_strat, _ = train_test_split(
        _stems, _strat_arr,
        test_size=_test_frac,
        stratify=_strat_arr,
        random_state=GLOBAL_SEED
    )

    # ── Step B: split remaining 80/20 into train and val ─────
    _val_frac = 1 - TRAIN_VAL_RATIO   # 0.20
    _train_stems, _val_stems = train_test_split(
        _trainval_stems,
        test_size=_val_frac,
        stratify=_trainval_strat,
        random_state=GLOBAL_SEED
    )

    _split_map = {}
    for _s in _train_stems: _split_map[_s] = 'train'
    for _s in _val_stems:   _split_map[_s] = 'val'
    for _s in _test_stems:  _split_map[_s] = 'test_day'

    print(f'\n  Split sizes (at image level):')
    print(f'    Train    : {len(_train_stems):>4} images')
    print(f'    Val      : {len(_val_stems):>4} images')
    print(f'    Test_day : {len(_test_stems):>4} images  (held-out)')

    # ── F2.3.f: assert leakage-free split (disjoint image-stem sets) ──
    _set_train, _set_val, _set_test = set(_train_stems), set(_val_stems), set(_test_stems)
    assert _set_train.isdisjoint(_set_val), 'LEAKAGE: train/val share image stems'
    assert _set_train.isdisjoint(_set_test), 'LEAKAGE: train/test_day share image stems'
    assert _set_val.isdisjoint(_set_test),   'LEAKAGE: val/test_day share image stems'
    assert _set_train | _set_val | _set_test == set(_stems), \
        'PARTITION: split union != original stems'
    print('  Leakage check : PASS  (disjoint train/val/test_day stems)')

    # ── Step C: assign patches to splits & copy to ImageFolder dirs ─
    PATCH_DF['split'] = PATCH_DF['source_stem'].map(_split_map)

    # ── F2.3.d: detect orphan source_stems (mapped to NaN) ────
    _n_orphan = int(PATCH_DF['split'].isna().sum())
    if _n_orphan > 0:
        _orphan_stems = sorted(PATCH_DF.loc[PATCH_DF['split'].isna(), 'source_stem'].unique())
        print(f'\n  [WARN] {_n_orphan:,} patches from {len(_orphan_stems)} orphan stems will be dropped.')
        print(f'         First 5 orphan stems: {_orphan_stems[:5]}')
        # Drop them rather than silently let downstream NaN logic skip
        PATCH_DF = PATCH_DF.dropna(subset=['split']).reset_index(drop=True)

    # Directory mapping: split -> class_id -> DIRS key
    _dir_map = {
        ('train', 0)   : DIRS['train_ripe'],
        ('train', 1)   : DIRS['train_unripe'],
        ('val',   0)   : DIRS['val_ripe'],
        ('val',   1)   : DIRS['val_unripe'],
        ('test_day', 0): DIRS['test_day_ripe'],
        ('test_day', 1): DIRS['test_day_unripe'],
    }

    _pool_dir = DIRS['patches'] / 'pool'
    _copy_count = 0
    for _, _row in tqdm(PATCH_DF.iterrows(), total=len(PATCH_DF),
                         desc='  Copying patches', ncols=80):
        _src = BASE_DIR / _row['patch_path']
        _dst_dir = _dir_map.get((_row['split'], _row['class_id']))
        if _dst_dir is None:
            continue
        _dst = _dst_dir / _src.name
        if not _dst.exists():
            shutil.copy2(str(_src), str(_dst))
        _copy_count += 1

    # ── Save split assignments CSV ────────────────────────────
    PATCH_DF.to_csv(SPLIT_CSV, index=False)
    SPLIT_DF = PATCH_DF

    print(f'  Patches copied       : {_copy_count:,}')
    print(f'  Split CSV saved      : {SPLIT_CSV}')

    # ── Drive sync (if enabled) ───────────────────────────────
    if USE_DRIVE_CACHE and DRIVE_PROJECT_DIR is not None and not CACHE_HIT:
        _drive_patches = DRIVE_PROJECT_DIR / 'data' / 'patches'
        _drive_splits  = DRIVE_PROJECT_DIR / 'data' / 'splits'
        _drive_splits.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(SPLIT_CSV), str(_drive_splits / SPLIT_CSV.name))
        print(f'  Drive sync: split CSV copied.')


# ── Patch count per split & class ────────────────────────────
print(f'\n  Patch counts per split & class:')
_split_class = SPLIT_DF.groupby(['split', 'class_name']).size().unstack(fill_value=0)
print(_split_class.to_string())

# ── F2.3.e: per-split class proportions vs. pool baseline ────
# Demonstrates that stratification preserved the global ripe:unripe ratio.
_pool_ripe_pct = 100 * (SPLIT_DF['class_id'] == 0).mean()
print(f'\n  Per-split ripe% (baseline pool ripe% = {_pool_ripe_pct:.2f}):')
for _split_name in ['train', 'val', 'test_day']:
    _sub = SPLIT_DF[SPLIT_DF['split'] == _split_name]
    if len(_sub) == 0:
        continue
    _r = 100 * (_sub['class_id'] == 0).mean()
    _delta = _r - _pool_ripe_pct
    _flag = '' if abs(_delta) < 5 else '  [ALERT: > 5pp drift]'
    print(f'    {_split_name:<8} : ripe={_r:5.2f}%  (Δ vs pool = {_delta:+.2f}pp){_flag}')


# ── Update metadata ───────────────────────────────────────────
METADATA['split'] = {
    'strategy'       : 'image-level stratified (no patch-level leakage)',
    'global_seed'    : GLOBAL_SEED,
    'n_test_images'  : N_TEST_IMAGES,
    'train_val_ratio': TRAIN_VAL_RATIO,
    'split_csv'      : str(SPLIT_CSV),
    'counts'         : {f'{split}_{cls}': count for (split, cls), count in SPLIT_DF.groupby(['split', 'class_name']).size().to_dict().items()},
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f'\n  Metadata updated : {metadata_path}')
print(f'\n[OK] Step 2.3 complete — data pipeline ready for EDA and training.')


---

## Phase 3 — Exploratory Data Analysis (EDA)

Characterises the patch dataset visually and statistically before any model training. Key goals: (1) confirm visual separability of ripe vs unripe; (2) quantify colour statistics in LAB space — the colour space used by the LED augmentation in Phase 4; (3) establish that colour features alone provide partial discriminability, justifying the SVM-colour-feature baseline in Phase 5.

### Step 3.1 — Patch Visual Inspection

Displays a grid of sample patches for each class to confirm extraction quality and visual separability.

In [ ]:
# ============================================================
# STEP 3.1 — PATCH VISUAL INSPECTION
# ============================================================
print_section('Step 3.1 — Patch Visual Inspection')

set_seed(GLOBAL_SEED)
_N_SAMPLES = 8   # patches per class to display

_fig, _axes = plt.subplots(2, _N_SAMPLES, figsize=(2.5 * _N_SAMPLES, 6))
_fig.suptitle('Sample Patches — Ripe (top) vs Unripe (bottom)', fontsize=14, fontweight='bold', y=1.02)

for _row, _cls in enumerate([0, 1]):
    _cls_name = CLASS_NAMES[_cls]
    _cls_patches = SPLIT_DF[
        (SPLIT_DF['class_id'] == _cls) & (SPLIT_DF['split'] == 'train')
    ]['patch_path'].sample(_N_SAMPLES, random_state=GLOBAL_SEED).tolist()

    for _col, _rel_path in enumerate(_cls_patches):
        _img = np.array(Image.open(BASE_DIR / _rel_path).convert('RGB'))
        _axes[_row, _col].imshow(_img)
        _axes[_row, _col].axis('off')
        if _col == 0:
            _axes[_row, _col].set_ylabel(
                _cls_name.capitalize(), fontsize=12, fontweight='bold',
                rotation=90, labelpad=10
            )

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'eda_sample_patches.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'  Figure saved → eda_sample_patches.png')
print(f'\n[OK] Step 3.1 complete.')


### Step 3.2 — Colour Statistics Analysis (RGB & LAB)

Computes per-channel mean and standard deviation in RGB and LAB colour spaces for a stratified sample of training patches. LAB statistics directly motivate the LED simulation augmentation in Phase 4.

In [ ]:
# ============================================================
# STEP 3.2 — COLOUR STATISTICS (RGB & LAB)
# ============================================================
print_section('Step 3.2 — Colour Statistics Analysis')

set_seed(GLOBAL_SEED)
_SAMPLE_N = 300   # patches per class to sample for statistics

_stats_records = []

for _cls in [0, 1]:
    _cls_name = CLASS_NAMES[_cls]
    _subset = SPLIT_DF[
        (SPLIT_DF['class_id'] == _cls) & (SPLIT_DF['split'] == 'train')
    ]
    _sample = _subset.sample(min(_SAMPLE_N, len(_subset)), random_state=GLOBAL_SEED)

    for _, _row in tqdm(_sample.iterrows(), total=len(_sample),
                         desc=f'  {_cls_name} patches', ncols=80):
        _img_np = np.array(Image.open(BASE_DIR / _row['patch_path']).convert('RGB'), dtype=np.float32)

        # RGB means
        _r_mean, _g_mean, _b_mean = _img_np[:,:,0].mean(), _img_np[:,:,1].mean(), _img_np[:,:,2].mean()

        # LAB conversion via OpenCV
        _bgr = cv2.cvtColor(_img_np.astype(np.uint8), cv2.COLOR_RGB2BGR)
        _lab = cv2.cvtColor(_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        _L_mean  = _lab[:,:,0].mean()   # Luminance: 0-255 (maps to 0-100 in standard LAB)
        _a_mean  = _lab[:,:,1].mean()   # a*: green(-) to red(+), 0-255 (maps to -128 to +127)
        _b_mean  = _lab[:,:,2].mean()   # b*: blue(-) to yellow(+)

        _stats_records.append({
            'class_id': _cls, 'class_name': _cls_name,
            'R': _r_mean, 'G': _g_mean, 'B': _b_mean,
            'L': _L_mean, 'a': _a_mean, 'b_star': _b_mean,
        })

_stats_df = pd.DataFrame(_stats_records)

# ── Print summary table ───────────────────────────────────────
print(f'\n  Colour channel means by class (training sample, N={_SAMPLE_N}/class):')
print(f'  {"Channel":<10} {"Ripe (mean±std)":<25} {"Unripe (mean±std)":<25}')
print(f'  {"-"*60}')
for _ch in ['R', 'G', 'B', 'L', 'a', 'b_star']:
    _ch_display = _ch.replace('b_star', 'b*')
    _ripe_vals   = _stats_df[_stats_df['class_id']==0][_ch]
    _unripe_vals = _stats_df[_stats_df['class_id']==1][_ch]
    print(f'  {_ch_display:<10} {_ripe_vals.mean():>6.1f} ± {_ripe_vals.std():<15.1f} {_unripe_vals.mean():>6.1f} ± {_unripe_vals.std():.1f}')

# ── Box plots ─────────────────────────────────────────────────
_channels = ['R', 'G', 'B', 'L', 'a', 'b_star']
_labels   = ['R', 'G', 'B', 'L*', 'a*', 'b*']
_fig, _axes = plt.subplots(1, len(_channels), figsize=(2.5*len(_channels), 4), sharey=False)
_fig.suptitle('Colour Channel Distributions by Class (Training Patches)', fontsize=13, fontweight='bold')

for _i, (_ch, _lbl) in enumerate(zip(_channels, _labels)):
    _ax = _axes[_i]
    _data_ripe   = _stats_df[_stats_df['class_id']==0][_ch].values
    _data_unripe = _stats_df[_stats_df['class_id']==1][_ch].values
    _bp = _ax.boxplot(
        [_data_ripe, _data_unripe],
        labels=['Ripe', 'Unripe'],
        patch_artist=True,
        medianprops={'color': 'black', 'linewidth': 2},
        widths=0.5
    )
    _bp['boxes'][0].set_facecolor('#e74c3c80')
    _bp['boxes'][1].set_facecolor('#2ecc7180')
    _ax.set_title(_lbl, fontsize=12)
    _ax.set_xlabel('')
    _ax.tick_params(axis='x', labelsize=9)
    sns.despine(ax=_ax)

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'eda_colour_distributions.png', dpi=300)
plt.show()
print(f'  Figure saved → eda_colour_distributions.png')

# ── LAB scatter: a* vs b* ─────────────────────────────────────
_fig2, _ax2 = plt.subplots(figsize=(6, 5))
_colors = {0: '#e74c3c', 1: '#2ecc71'}
for _cls in [0, 1]:
    _sub = _stats_df[_stats_df['class_id'] == _cls]
    _ax2.scatter(_sub['a'], _sub['b_star'], c=_colors[_cls],
                 label=CLASS_NAMES[_cls].capitalize(), alpha=0.5, s=15, edgecolors='none')
_ax2.set_xlabel('a* (green ← → red)')
_ax2.set_ylabel('b* (blue ← → yellow)')
_ax2.set_title('LAB Colour Space: a* vs b* by Class', fontsize=13, fontweight='bold')
_ax2.legend()
sns.despine(ax=_ax2)
_fig2.tight_layout()
_fig2.savefig(DIRS['figures'] / 'eda_lab_scatter.png', dpi=300)
plt.show()
print(f'  Figure saved → eda_lab_scatter.png')

# ── Update metadata ───────────────────────────────────────────
METADATA['eda_colour'] = {
    cls_name: {
        ch: {'mean': round(float(_stats_df[_stats_df['class_id']==cid][ch].mean()), 2),
             'std':  round(float(_stats_df[_stats_df['class_id']==cid][ch].std()), 2)}
        for ch in ['R','G','B','L','a','b_star']
    }
    for cid, cls_name in enumerate(CLASS_NAMES)
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f'\n[OK] Step 3.2 complete.')


### Step 3.3 — Raw Patch Size Distribution

Plots the distribution of raw bounding-box pixel dimensions (before 224×224 resize). Validates the `MIN_PATCH_PX=20` threshold and characterises the upsampling magnitude applied to small patches.

In [ ]:
# ============================================================
# STEP 3.3 — RAW PATCH SIZE DISTRIBUTION
# ============================================================
# NOTE: Raw patch sizes are not stored in SPLIT_DF (only the resized 224x224 patches
# are on disk). We re-derive approximate sizes from YOLO coordinates + image dims.
# This is a representative sample, not exhaustive, and is used only for EDA.
print_section('Step 3.3 — Raw Patch Size Distribution')

set_seed(GLOBAL_SEED)
_SIZE_SAMPLE = 500  # images to sample for size analysis

_patch_widths, _patch_heights = [], []
_sample_stems = random.sample(sorted(VALID_IMAGE_CLASSES.keys()), min(_SIZE_SAMPLE, len(VALID_IMAGE_CLASSES)))

for _stem in _sample_stems:
    _lp = LBL_DIR / (_stem + '.txt')
    _ip = None
    for _ext in ('.jpg', '.jpeg', '.png'):
        if (IMG_DIR / (_stem + _ext)).exists():
            _ip = IMG_DIR / (_stem + _ext)
            break
    if _ip is None or not _lp.exists():
        continue

    try:
        _W, _H = Image.open(_ip).size
    except Exception:
        continue

    for _line in _lp.read_text().strip().splitlines():
        _parts = _line.strip().split()
        if len(_parts) != 5:
            continue
        try:
            _cls = int(_parts[0])
            _bw, _bh = float(_parts[3]), float(_parts[4])
        except ValueError:
            continue
        if _cls not in (0, 1):
            continue
        _pw = int(_bw * _W * (1 + 2 * PATCH_PADDING))
        _ph = int(_bh * _H * (1 + 2 * PATCH_PADDING))
        _patch_widths.append(min(_pw, _W))
        _patch_heights.append(min(_ph, _H))

_fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(10, 4))
_fig.suptitle(f'Raw Patch Size Distribution (sample N={len(_patch_widths)})', fontsize=13, fontweight='bold')

_ax1.hist(_patch_widths, bins=40, color='#3498db', edgecolor='white', linewidth=0.4)
_ax1.axvline(MIN_PATCH_PX, color='red', linestyle='--', linewidth=1.5, label=f'MIN={MIN_PATCH_PX}px')
_ax1.axvline(224, color='orange', linestyle='--', linewidth=1.5, label='Target 224px')
_ax1.set_xlabel('Patch Width (px)')
_ax1.set_ylabel('Count')
_ax1.set_title('Width')
_ax1.legend(fontsize=9)
sns.despine(ax=_ax1)

_ax2.hist(_patch_heights, bins=40, color='#9b59b6', edgecolor='white', linewidth=0.4)
_ax2.axvline(MIN_PATCH_PX, color='red', linestyle='--', linewidth=1.5, label=f'MIN={MIN_PATCH_PX}px')
_ax2.axvline(224, color='orange', linestyle='--', linewidth=1.5, label='Target 224px')
_ax2.set_xlabel('Patch Height (px)')
_ax2.set_title('Height')
_ax2.legend(fontsize=9)
sns.despine(ax=_ax2)

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'eda_patch_sizes.png', dpi=300)
plt.show()

_widths_arr  = np.array(_patch_widths)
_heights_arr = np.array(_patch_heights)
print(f'  Width  — median: {np.median(_widths_arr):.0f}px  mean: {_widths_arr.mean():.0f}px  min: {_widths_arr.min()}px  max: {_widths_arr.max()}px')
print(f'  Height — median: {np.median(_heights_arr):.0f}px  mean: {_heights_arr.mean():.0f}px  min: {_heights_arr.min()}px  max: {_heights_arr.max()}px')
print(f'  Upsampled (< 224px): {((_widths_arr < 224) & (_heights_arr < 224)).sum()} / {len(_widths_arr)} patches ({100*((_widths_arr<224)&(_heights_arr<224)).mean():.0f}%)')
print(f'  Figure saved → eda_patch_sizes.png')
print(f'\n[OK] Step 3.3 complete.')


### Step 3.4 — EDA Key Findings

**Colour separability**: The LAB a\*-vs-b\* scatter plot (Step 3.2) confirms that ripe berries cluster in the high-a\* region (shifted towards red) while unripe berries cluster towards low-a\*/high-b\* (green-yellow). This partial linear separability is the foundation for the HSV-histogram + LBP + HOG SVM baseline in Phase 5 — if colour alone provides ~70–80% accuracy, the SVM baseline is scientifically meaningful.

**LED domain shift mechanism (physical basis)**: When onboard white LEDs illuminate a strawberry, the L\* (luminance) channel increases substantially — berries are brighter under close-range artificial light. The a\* channel (red-green axis) may also shift if the LED spectrum has a different red-wavelength output than sunlight. Critically, the a\* shift means a ripe (high-a\*) berry under LED may appear with a different a\* value than a ripe berry under natural daylight — moving it towards the unripe cluster in colour space. This is the visual mechanism of domain shift.

**Patch size validation**: The `MIN_PATCH_PX=20` threshold excludes the extreme small-berry tail of the distribution. The median patch width is expected to be ~80–150px, well above the minimum. Most patches are upsampled to 224×224 (not downsampled), so Lanczos anti-aliasing is less critical than spatial context preservation — the padding choice of 10% is validated by the typical patch width.

**Class imbalance**: Any imbalance revealed in Step 2.2 is addressed by `CLASS_WEIGHTS` in loss functions (Phase 6) and `WeightedRandomSampler` in DataLoaders. Both macro-F1 (class-averaged) and AUC are used as primary metrics to ensure the minority class performance is not masked by accuracy.

---

## Phase 4 — LED Night Simulation Augmentation

Implements a physically-motivated LED illumination transform in LAB colour space. Onboard white LEDs (3000–4000K) increase luminance (L\*) and add a warm yellow cast (b\*). The transform is implemented as:
1. A **stochastic albumentations transform** for E5 training augmentation (random L\*/b\* shift sampled per image)
2. A **deterministic test-time transform** (fixed parameters) applied to `test_day` patches to generate the `test_night` evaluation set used in E3 and E5

### Step 4.1 — LED LAB Transform: Stochastic & Deterministic Variants

Defines `LEDNightSimulation` as an albumentations-compatible custom transform. Parameterised by L\* multiplier range and b\* additive shift range — both sampled uniformly per image during training (stochastic) or fixed for test-time simulation (deterministic).

In [ ]:
# ============================================================
# STEP 4.1 — LED LAB TRANSFORM
# ============================================================
print_section('Step 4.1 — LED Night Simulation Transform')


class LEDNightSimulation(ImageOnlyTransform):
    """
    Physically-motivated LED illumination augmentation in LAB colour space.

    Models the two primary effects of onboard agricultural robot LED lighting:
      1. Luminance increase (L* channel): LEDs produce high-intensity local illumination
      2. Warm colour cast (b* channel): warm white LEDs over-represent yellow-orange wavelengths

    Operates in OpenCV LAB encoding (L: 0-255, a: 0-255, b: 0-255).
    Standard LAB ranges: L* 0-100, a* -128 to +127, b* -128 to +127
    OpenCV maps these linearly: L=L*×255/100, a=a*+128, b=b*+128

    Args:
        l_scale_range : tuple of (min, max) multiplicative scale for L channel.
                        (1.10, 1.40) models 10-40% LED brightness increase.
        b_shift_range : tuple of (min, max) additive shift to b channel (yellow cast).
                        In OpenCV units (0-255); default (10, 40) corresponds to roughly
                        +5 to +20 in canonical b* (signed) units. Models warm-white LED
                        chromatic shift along the yellow-blue axis.
        a_shift_range : tuple of (min, max) additive shift to a channel.
                        Default (-10, 10) -- minor stochastic spectral variation across
                        LED bin/model. Kept small because the green-red axis is precisely
                        the ripe/unripe discriminator, so we avoid leaking class signal.
        p             : probability of applying this transform (albumentations convention).
    """

    def __init__(
        self,
        l_scale_range=(1.10, 1.40),
        b_shift_range=(10, 40),
        a_shift_range=(-10, 10),
        p=1.0
    ):
        super().__init__(p=p)
        self.l_scale_range = l_scale_range
        self.b_shift_range = b_shift_range
        self.a_shift_range = a_shift_range

    def apply(self, img, l_scale=1.25, b_shift=20, a_shift=0, **params):
        # img: HxWx3 RGB uint8 numpy array
        _bgr  = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        _lab  = cv2.cvtColor(_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

        _lab[:, :, 0] = np.clip(_lab[:, :, 0] * l_scale, 0, 255)   # L*: scale
        _lab[:, :, 1] = np.clip(_lab[:, :, 1] + a_shift,  0, 255)   # a*: shift
        _lab[:, :, 2] = np.clip(_lab[:, :, 2] + b_shift,  0, 255)   # b*: shift

        _lab_u8  = _lab.astype(np.uint8)
        _bgr_out = cv2.cvtColor(_lab_u8, cv2.COLOR_LAB2BGR)
        return cv2.cvtColor(_bgr_out, cv2.COLOR_BGR2RGB)

    def get_params(self):
        # Called once per image — albumentations samples parameters here
        return {
            'l_scale': random.uniform(*self.l_scale_range),
            'b_shift': random.uniform(*self.b_shift_range),
            'a_shift': random.uniform(*self.a_shift_range),
        }

    def get_transform_init_args_names(self):
        return ('l_scale_range', 'b_shift_range', 'a_shift_range')


# ── Deterministic test-time transform (fixed parameters) ─────
# Used to generate test_night from test_day -- MUST be fixed for
# reproducible E3/E5 comparison.
#
# Parameter rationale (chosen *before* training, not tuned to results):
#   l_scale = 1.25  -- midpoint of (1.10, 1.40); +25% luminance gain matches
#                      the typical LED-vs-ambient-greenhouse-light brightness ratio.
#   b_shift = 20    -- conservative warm-yellow cast (range is (10, 40)).
#                      Picked below the (10, 40) midpoint of 25 to stay closer to
#                      diffuse LED reflection rather than peak direct-LED hotspots,
#                      so the test set characterises the *typical* night condition,
#                      not the worst case.
#   a_shift = 0     -- the a* axis carries the ripe/unripe colour signal, so we
#                      hold it neutral at test time to avoid confounding the metric.
LED_TEST_TRANSFORM = LEDNightSimulation(
    l_scale_range=(1.25, 1.25),   # fixed: 25% luminance increase
    b_shift_range=(20,   20),     # fixed: +20 b units warm yellow cast
    a_shift_range=(0,     0),     # fixed: no a* shift (preserves ripe/unripe colour signal)
    p=1.0
)

# ── Stochastic training transform ────────────────────────────
# Used in E5 DataLoader pipeline — random parameters per image
LED_TRAIN_TRANSFORM = LEDNightSimulation(
    l_scale_range=(1.10, 1.40),
    b_shift_range=(10,   40),
    a_shift_range=(-10,  10),
    p=1.0
)

print(f'  LEDNightSimulation transform defined.')
print(f'  LED_TEST_TRANSFORM  : L*x1.25 | b*+20 | a*+0  (fixed)')
print(f'  LED_TRAIN_TRANSFORM : L*x[1.10-1.40] | b*[+10-+40] | a*[-10-+10]  (stochastic)')
print(f'\n[OK] Step 4.1 complete.')


### Step 4.2 — Generate test_night Patch Set

Applies `LED_TEST_TRANSFORM` (fixed parameters) to every patch in `test_day` to generate the `test_night` evaluation set. The test_night set is used in E3 (domain shift evaluation) and E5 (adapted model evaluation). Generation is idempotent.

In [ ]:
# ============================================================
# STEP 4.2 — GENERATE test_night PATCHES
# ============================================================
print_section('Step 4.2 — Generate test_night Patch Set')


_night_ripe_dir   = DIRS['test_night_ripe']
_night_unripe_dir = DIRS['test_night_unripe']

# F4.2.b idempotency: skip only if BOTH classes are fully generated
# (count must match the corresponding test_day class). Catches the case
# where a prior run was interrupted with only one class complete.
_n_day_ripe   = len(list(DIRS['test_day_ripe'].glob('*.jpg')))
_n_day_unripe = len(list(DIRS['test_day_unripe'].glob('*.jpg')))
_n_night_ripe   = len(list(_night_ripe_dir.glob('*.jpg')))
_n_night_unripe = len(list(_night_unripe_dir.glob('*.jpg')))

_fully_generated = (
    _n_night_ripe   == _n_day_ripe   and _n_day_ripe   > 0 and
    _n_night_unripe == _n_day_unripe and _n_day_unripe > 0
)

if _fully_generated:
    print(f'  test_night already fully generated:')
    print(f'    ripe   : {_n_night_ripe:>4} / {_n_day_ripe} (day)')
    print(f'    unripe : {_n_night_unripe:>4} / {_n_day_unripe} (day)')
    print('  Skipping regeneration.')
else:
    if _n_night_ripe + _n_night_unripe > 0:
        print(f'  [WARN] partial test_night detected '
              f'({_n_night_ripe}/{_n_day_ripe} ripe, '
              f'{_n_night_unripe}/{_n_day_unripe} unripe) -- regenerating missing patches.')
    _day_src_dirs = [
        (DIRS['test_day_ripe'],   _night_ripe_dir,   0),
        (DIRS['test_day_unripe'], _night_unripe_dir, 1),
    ]

    _night_counter = Counter()
    for _src_dir, _dst_dir, _cls in _day_src_dirs:
        _day_patches = sorted(_src_dir.glob('*.jpg'))
        for _dp in tqdm(_day_patches, desc=f'  {CLASS_NAMES[_cls]} night', ncols=80):
            _img_rgb = np.array(Image.open(_dp).convert('RGB'))
            _dst_path = _dst_dir / _dp.name
            if _dst_path.exists():
                _night_counter[_cls] += 1
                continue   # F4.2.b: resume-safe, don't redo existing patches
            _night_rgb = LED_TEST_TRANSFORM.apply(
                _img_rgb,
                l_scale=1.25, b_shift=20, a_shift=0
            )
            Image.fromarray(_night_rgb).save(str(_dst_path), 'JPEG', quality=95)
            _night_counter[_cls] += 1

    print(f'\n  test_night patches generated:')
    for _cid, _cnt in sorted(_night_counter.items()):
        print(f'    {CLASS_NAMES[_cid]:<8} : {_cnt:,}')
    print(f'    Total    : {sum(_night_counter.values()):,}')

    METADATA['test_night'] = {
        'transform' : 'LEDNightSimulation fixed: L*x1.25, b*+20, a*+0',
        'source'    : 'test_day patches',
        'counts'    : {CLASS_NAMES[k]: v for k, v in _night_counter.items()},
    }
    with open(metadata_path, 'w') as f:
        json.dump(METADATA, f, indent=2)

print(f'  test_night_ripe    : {DIRS["test_night_ripe"]}')
print(f'  test_night_unripe  : {DIRS["test_night_unripe"]}')
print(f'\n[OK] Step 4.2 complete.')


### Step 4.3 — LED Augmentation Visualisation

Side-by-side visual comparison of day vs night patches for both classes, and a panel showing the effect of different LED parameter levels. Confirms the augmentation produces visually plausible illumination changes.

In [ ]:
# ============================================================
# STEP 4.3 — LED AUGMENTATION VISUALISATION
# ============================================================
print_section('Step 4.3 — LED Augmentation Visualisation')

set_seed(GLOBAL_SEED)
_N_VIS = 4  # patches per class to show

# ── Side-by-side: day vs night for ripe and unripe ───────────
_fig, _axes = plt.subplots(4, _N_VIS, figsize=(3 * _N_VIS, 13))
_fig.suptitle('LED Night Simulation: Day vs Night Patches', fontsize=14, fontweight='bold')

_row_labels = ['Ripe — Day', 'Ripe — Night', 'Unripe — Day', 'Unripe — Night']
for _row_i, _lbl in enumerate(_row_labels):
    _axes[_row_i, 0].set_ylabel(_lbl, fontsize=10, fontweight='bold', rotation=0,
                                 labelpad=80, va='center')

for _cls_i, _cls in enumerate([0, 1]):
    _day_dir = DIRS['test_day_ripe'] if _cls == 0 else DIRS['test_day_unripe']
    _day_files = random.sample(sorted(_day_dir.glob('*.jpg')), _N_VIS)

    for _col, _dp in enumerate(_day_files):
        _img_day = np.array(Image.open(_dp).convert('RGB'))
        _img_night = LED_TEST_TRANSFORM.apply(_img_day, l_scale=1.25, b_shift=20, a_shift=0)

        _row_day   = _cls_i * 2
        _row_night = _cls_i * 2 + 1

        _axes[_row_day, _col].imshow(_img_day)
        _axes[_row_day, _col].axis('off')
        _axes[_row_night, _col].imshow(_img_night)
        _axes[_row_night, _col].axis('off')

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'led_day_vs_night.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'  Figure saved → led_day_vs_night.png')

# ── Parameter sweep: L* at 1.0, 1.1, 1.2, 1.3, 1.4 ──────────
_l_scales = [1.0, 1.1, 1.2, 1.3, 1.4]
_sample_ripe = random.choice(sorted(DIRS['test_day_ripe'].glob('*.jpg')))
_img_base = np.array(Image.open(_sample_ripe).convert('RGB'))

_fig2, _axes2 = plt.subplots(1, len(_l_scales), figsize=(3 * len(_l_scales), 3.5))
_fig2.suptitle(f'L* Scale Sweep (b*+20 fixed) — Ripe Patch', fontsize=12, fontweight='bold')
for _ai, _ls in enumerate(_l_scales):
    _aug = LED_TEST_TRANSFORM.apply(_img_base, l_scale=_ls, b_shift=20, a_shift=0)
    _axes2[_ai].imshow(_aug)
    _axes2[_ai].set_title(f'L* x{_ls:.1f}', fontsize=11)
    _axes2[_ai].axis('off')
_fig2.tight_layout()
_fig2.savefig(DIRS['figures'] / 'led_l_scale_sweep.png', dpi=300)
plt.show()
print(f'  Figure saved → led_l_scale_sweep.png')
print(f'\n[OK] Step 4.3 complete.')


---

## Phase 5 — Experiment E2: Traditional ML Baseline (SVM)

Establishes the traditional ML performance ceiling using hand-crafted colour + texture features fed to an RBF-kernel SVM. Covers two evaluations:
- **E2-Day**: SVM trained on day features, evaluated on the held-out daytime test set
- **E2-Night**: Same frozen SVM evaluated on LED-simulated night patches — quantifies whether domain shift degrades traditional ML too

This is the primary **LO1 comparison point**: traditional ML vs deep learning, with rigorous leakage prevention and overfitting controls throughout.

### Step 5.1 — Hand-Crafted Feature Extraction Pipeline

Extracts a 259-dimensional feature vector from each 224×224 RGB patch:
- **HSV histogram** (96): 32 bins × 3 channels — primary colour distribution signal
- **LBP** (18): radius=2, n\_points=16, uniform — illumination-invariant micro-texture
- **HOG** (81): 3×3 compact cells, 9 orientation bins — gradient structure at colour boundaries
- **LAB a\*/b\* histogram** (64): 32 bins × 2 channels; L\* excluded (LED nuisance variable)

Feature extraction is **stateless per patch** — no global statistics, no leakage risk. The `StandardScaler` is deferred to the `Pipeline` in Step 5.2 where it is fitted exclusively on training data inside each GridSearchCV fold.

In [ ]:
# ============================================================
# STEP 5.1 — HAND-CRAFTED FEATURE EXTRACTION PIPELINE
# ============================================================
print_section('Step 5.1 — Feature Extraction Pipeline')


# ── Feature extractor ────────────────────────────────────────
def extract_features(img_rgb: np.ndarray) -> np.ndarray:
    """
    Extract a 259-dimensional hand-crafted feature vector from a single
    224x224 RGB patch.

    Feature composition:
      [0:96]   HSV histogram  — 32 bins per channel (H, S, V)
      [96:114] LBP histogram  — radius=2, n_points=16, uniform method (18 bins)
      [114:195] HOG descriptor — 3x3 cells, 9 orientation bins (81 values)
      [195:259] LAB a*/b* hist — 32 bins per channel, L* excluded

    Args:
        img_rgb: H x W x 3 uint8 numpy array in RGB format (output of PIL.Image)

    Returns:
        1D float32 numpy array of length 259

    Design notes:
      - L* excluded from LAB: luminance is the primary LED nuisance variable.
        Including it would train the SVM on the exact channel that domain shift
        disturbs, conflating class signal with domain shift.
      - V retained in HSV: removing one HSV channel breaks the standard
        decomposition; V carries some brightness information but is not the
        primary discriminative signal (H is).
      - All histograms are density-normalised (sum=1) for scale invariance.
      - HOG uses L2-Hys block normalisation (skimage default) for contrast
        normalisation — partially robust to illumination change.
    """
    assert img_rgb.ndim == 3 and img_rgb.shape[2] == 3, \
        f'Expected HxWx3 RGB, got shape {img_rgb.shape}'
    assert img_rgb.dtype == np.uint8, \
        f'Expected uint8, got {img_rgb.dtype}'

    features = []

    # ── 1. HSV histogram (96 features) ───────────────────────
    # cv2.cvtColor expects BGR input
    _bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    _hsv = cv2.cvtColor(_bgr, cv2.COLOR_BGR2HSV)
    # H: 0-179 in OpenCV (not 0-360); S,V: 0-255
    _hsv_ranges = [(0, 180), (0, 256), (0, 256)]
    for _ch in range(3):
        _hist = cv2.calcHist([_hsv], [_ch], None, [32], [_hsv_ranges[_ch][0], _hsv_ranges[_ch][1]])
        _hist = _hist.flatten()
        _hist_sum = _hist.sum()
        if _hist_sum > 0:
            _hist = _hist / _hist_sum   # density normalise
        features.append(_hist)

    # ── 2. LBP histogram (18 features) ───────────────────────
    # LBP operates on grayscale; uniform method gives rotation invariance
    _gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _lbp = local_binary_pattern(_gray, P=16, R=2, method='uniform')
    # uniform LBP with P=16 gives 16+2=18 possible pattern values
    _n_bins = 16 + 2   # P + 2 for uniform method
    _lbp_hist, _ = np.histogram(_lbp.ravel(), bins=_n_bins,
                                 range=(0, _n_bins), density=True)
    features.append(_lbp_hist.astype(np.float32))

    # ── 3. HOG descriptor (81 features) ──────────────────────
    # Compact configuration: 3x3 cells, 9 orientation bins
    # pixels_per_cell = (224//3, 224//3) ≈ (74, 74) — coarse grid
    # cells_per_block = (1, 1) — no block pooling (keeps it to 3x3x9=81)
    _ppc = (img_rgb.shape[0] // 3, img_rgb.shape[1] // 3)  # ~(74, 74)
    _hog_feat = skimage_hog(
        _gray,
        orientations=9,
        pixels_per_cell=_ppc,
        cells_per_block=(1, 1),
        block_norm='L2-Hys',
        feature_vector=True
    )  # shape: (3*3*1*1*9,) = (81,)
    features.append(_hog_feat.astype(np.float32))

    # ── 4. LAB a*/b* histogram (64 features) ─────────────────
    # L* deliberately excluded — it is the primary LED nuisance variable
    _lab = cv2.cvtColor(_bgr, cv2.COLOR_BGR2LAB)
    # OpenCV LAB: L 0-255, a 0-255 (centred at 128), b 0-255 (centred at 128)
    for _ch in [1, 2]:   # channels 1=a*, 2=b* only
        _hist = cv2.calcHist([_lab], [_ch], None, [32], [0, 256])
        _hist = _hist.flatten()
        _hist_sum = _hist.sum()
        if _hist_sum > 0:
            _hist = _hist / _hist_sum
        features.append(_hist)

    _vec = np.concatenate(features).astype(np.float32)
    return _vec


# ── Verify feature dimensions on a dummy patch ────────────────
_dummy = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
_dummy_feat = extract_features(_dummy)
FEATURE_DIM = len(_dummy_feat)
assert FEATURE_DIM == 259, f'Expected 259 features, got {FEATURE_DIM}'
print(f'  Feature vector dimension : {FEATURE_DIM}')
print(f'    HSV histogram          : 96  (indices 0:96)')
print(f'    LBP histogram          : 18  (indices 96:114)')
print(f'    HOG descriptor         : 81  (indices 114:195)')
print(f'    LAB a*/b* histogram    : 64  (indices 195:259)')
del _dummy, _dummy_feat


# ── Extract features for all splits ──────────────────────────
# Each split is processed independently — no global statistics computed here.
# StandardScaler fitting is deferred to the Pipeline in Step 5.2.

def _extract_split(split_name: str, dirs_ripe, dirs_unripe, desc: str):
    """
    Extract features + labels for all patches in a named split.
    Returns (X: float32 array [N, 259], y: int array [N], stems: list[str])
    """
    _X, _y, _stems = [], [], []
    for _cls_idx, _dir in enumerate([dirs_ripe, dirs_unripe]):
        _patches = sorted(_dir.glob('*.jpg'))
        for _p in tqdm(_patches, desc=f'  {desc} class={CLASS_NAMES[_cls_idx]}', ncols=80, leave=False):
            try:
                _img = np.array(Image.open(_p).convert('RGB'), dtype=np.uint8)
                _feat = extract_features(_img)
                _X.append(_feat)
                _y.append(_cls_idx)
                _stems.append(_p.stem)
            except Exception as _e:
                print(f'  [WARN] Skipped {_p.name}: {_e}')
    if not _X:
        raise RuntimeError(f'No features extracted for split={split_name}')
    return np.stack(_X, axis=0), np.array(_y, dtype=np.int32), _stems


# F5.1.b — Feature matrix disk cache.
# Serialises X/y arrays with joblib after first extraction.
# On re-run, cache is loaded instantly (~0.2 s) instead of
# re-running cv2/HOG over all patches (~2 min).
# Delete the cache directory to force a rebuild.
_feat_cache_dir = DIRS['splits'] / 'feature_cache'
_feat_cache_dir.mkdir(exist_ok=True)
_feat_cache_file = _feat_cache_dir / f'features_dim{FEATURE_DIM}.pkl'

if _feat_cache_file.exists():
    print(f'  [CACHE] Loading feature matrices from {_feat_cache_file} ...')
    _cached = joblib.load(_feat_cache_file)
    X_train,      y_train,      _train_stems      = _cached['X_train'],      _cached['y_train'],      _cached['train_stems']
    X_val,        y_val,        _val_stems        = _cached['X_val'],        _cached['y_val'],        _cached['val_stems']
    X_test_day,   y_test_day,   _test_day_stems   = _cached['X_test_day'],   _cached['y_test_day'],   _cached['test_day_stems']
    X_test_night, y_test_night, _test_night_stems = _cached['X_test_night'], _cached['y_test_night'], _cached['test_night_stems']
    _extract_time = 0.0
    print(f'  Feature cache loaded: {len(y_train):,} train | {len(y_val):,} val | '
          f'{len(y_test_day):,} test_day | {len(y_test_night):,} test_night')
else:
    _t0 = time.time()

    print(f'\n  Extracting training features...')
    X_train, y_train, _train_stems = _extract_split(
        'train', DIRS['train_ripe'], DIRS['train_unripe'], 'train')

    print(f'  Extracting validation features...')
    X_val, y_val, _val_stems = _extract_split(
        'val', DIRS['val_ripe'], DIRS['val_unripe'], 'val')

    print(f'  Extracting test_day features...')
    X_test_day, y_test_day, _test_day_stems = _extract_split(
        'test_day', DIRS['test_day_ripe'], DIRS['test_day_unripe'], 'test_day')

    print(f'  Extracting test_night features...')
    X_test_night, y_test_night, _test_night_stems = _extract_split(
        'test_night', DIRS['test_night_ripe'], DIRS['test_night_unripe'], 'test_night')

    _extract_time = time.time() - _t0

    joblib.dump({
        'X_train': X_train, 'y_train': y_train, 'train_stems': _train_stems,
        'X_val': X_val, 'y_val': y_val, 'val_stems': _val_stems,
        'X_test_day': X_test_day, 'y_test_day': y_test_day, 'test_day_stems': _test_day_stems,
        'X_test_night': X_test_night, 'y_test_night': y_test_night, 'test_night_stems': _test_night_stems,
    }, _feat_cache_file, compress=3)
    print(f'\n  Feature cache saved → {_feat_cache_file}')


# ── Leakage verification ─────────────────────────────────────
# Verify that no test patch stem appears in training stems.
# Patch stems encode the source image: '{source_stem}_{annot_idx:04d}'
# We compare source stems (prefix before last underscore+4digits)
_train_sources = {s.rsplit('_', 1)[0] for s in _train_stems}
_test_sources  = {s.rsplit('_', 1)[0] for s in _test_day_stems}
_leaked = _train_sources & _test_sources
assert len(_leaked) == 0, (
    f'DATA LEAKAGE DETECTED: {len(_leaked)} source images appear in both '
    f'train and test splits: {list(_leaked)[:5]}'
)
print(f'\n  Leakage verification     : PASSED (0 shared source images)')


# ── Integrity checks (all splits) ────────────────────────────
assert X_train.shape[1]      == FEATURE_DIM, 'Feature dim mismatch in train'
assert X_val.shape[1]        == FEATURE_DIM, 'Feature dim mismatch in val'
assert X_test_day.shape[1]   == FEATURE_DIM, 'Feature dim mismatch in test_day'
assert X_test_night.shape[1] == FEATURE_DIM, 'Feature dim mismatch in test_night'

# F5.1.a: check all splits, not just train — a corrupt test patch
# would otherwise propagate to evaluation and produce garbage metrics.
for _chk_name, _chk_X in [
    ('train',      X_train),
    ('val',        X_val),
    ('test_day',   X_test_day),
    ('test_night', X_test_night),
]:
    assert not np.any(np.isnan(_chk_X)),  f'NaN in {_chk_name} features'
    assert not np.any(np.isinf(_chk_X)),  f'Inf in {_chk_name} features'


# ── Label consistency: test_day and test_night must have same labels ─
# (same patches, different illumination — labels must be identical)
assert np.array_equal(y_test_day, y_test_night), \
    'test_day and test_night label mismatch — check patch generation order'


# ── Summary ──────────────────────────────────────────────────
print(f'\n  Feature extraction summary:')
print(f'    {"Split":<14}  {"Samples":>8}  {"Ripe":>6}  {"Unripe":>8}  {"Shape"}')
print(f'    {"-"*56}')
for _name, _X, _y in [
    ('train',      X_train,      y_train),
    ('val',        X_val,        y_val),
    ('test_day',   X_test_day,   y_test_day),
    ('test_night', X_test_night, y_test_night),
]:
    _n_ripe   = int((_y == 0).sum())
    _n_unripe = int((_y == 1).sum())
    print(f'    {_name:<14}  {len(_y):>8,}  {_n_ripe:>6,}  {_n_unripe:>8,}  {_X.shape}')

print(f'\n  Feature dimension        : {FEATURE_DIM}')
print(f'  Train sample:feature ratio: {len(y_train)}/{FEATURE_DIM} = {len(y_train)/FEATURE_DIM:.1f}:1')
print(f'  Extraction time          : {_extract_time:.1f} s')
print(f'  Leakage check            : PASSED')
print(f'  NaN/Inf check            : PASSED')
print(f'  Label consistency check  : PASSED')


# ── Update metadata ───────────────────────────────────────────
METADATA['svm_features'] = {
    'feature_dim'   : FEATURE_DIM,
    'components'    : {
        'hsv_histogram' : {'bins': 32, 'channels': 3, 'dim': 96},
        'lbp'           : {'radius': 2, 'n_points': 16, 'method': 'uniform', 'dim': 18},
        'hog'           : {'cells': '3x3', 'orientations': 9, 'pixels_per_cell': '~74x74', 'dim': 81},
        'lab_ab_hist'   : {'bins': 32, 'channels': 'a*+b* only (L* excluded)', 'dim': 64},
    },
    'leakage_verified' : True,
    'extraction_time_s': round(_extract_time, 1),
    'split_sizes'   : {
        'train'      : int(len(y_train)),
        'val'        : int(len(y_val)),
        'test_day'   : int(len(y_test_day)),
        'test_night' : int(len(y_test_night)),
    },
}
with open(metadata_path, 'w') as f:
    json.dump(METADATA, f, indent=2)

print(f'\n  Metadata updated : {metadata_path}')
print(f'\n[OK] Step 5.1 complete — feature matrices ready for SVM training.')


### Step 5.2 — SVM Training & Hyperparameter Tuning

Trains an RBF-kernel SVM via 5-fold stratified `GridSearchCV` on the training set only. The `StandardScaler` lives **inside** the `Pipeline` — it is re-fitted from scratch on each CV fold's training portion, preventing any leakage of test/val statistics into feature normalisation. Best hyperparameters are selected by macro-F1. The final model is re-fitted on the full training set and saved with `joblib`.

In [ ]:
# ============================================================
# STEP 5.2 — SVM TRAINING & HYPERPARAMETER TUNING
# ============================================================
print_section('Step 5.2 — SVM Training & Hyperparameter Tuning')

set_seed(GLOBAL_SEED)

# F5.2.e — Idempotency: reload fitted pipeline if already on disk.
# Re-running GridSearchCV (75 fits) takes ~5 min; skip if pkl exists.
_svm_pkl = DIRS['model_svm'] / 'svm_pipeline_E2.pkl'
if _svm_pkl.exists():
    print(f'  [CACHE] Loading fitted SVM pipeline from {_svm_pkl} ...')
    SVM_MODEL = joblib.load(_svm_pkl)
    _cv_results_path = DIRS['model_svm'] / 'svm_cv_results.json'
    if _cv_results_path.exists():
        with open(_cv_results_path) as _f:
            _cached_meta = json.load(_f)
        SVM_BEST_PARAMS = _cached_meta.get('best_params', {})
        SVM_BEST_CV_F1  = _cached_meta.get('best_cv_f1', float('nan'))
        print(f'  Best params   : {SVM_BEST_PARAMS}')
        print(f'  Best CV F1    : {SVM_BEST_CV_F1:.4f}')
    else:
        SVM_BEST_PARAMS, SVM_BEST_CV_F1 = {}, float('nan')
        print('  [WARN] cv_results.json not found — metadata unavailable.')
    print(f'  [OK] Step 5.2 skipped (cached). Delete {_svm_pkl} to retrain.')
else:

    # ── Pipeline architecture ─────────────────────────────────────
    # StandardScaler INSIDE the pipeline: during GridSearchCV, sklearn
    # calls scaler.fit_transform() only on each fold's training portion,
    # then scaler.transform() on the fold's validation portion.
    # This is the only correct way to prevent normalisation leakage in CV.
    _svm_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel='rbf',
            class_weight='balanced',   # corrects class imbalance
            probability=True,          # enables predict_proba() for AUC
            random_state=GLOBAL_SEED,
            cache_size=500,            # MB of kernel cache — speeds up RBF fitting
        ))
    ])

    # ── Hyperparameter grid ───────────────────────────────────────
    # C:     regularisation strength — wide range (4 orders of magnitude)
    #        Low C (0.01) → high regularisation, simple boundary
    #        High C (100) → low regularisation, fits training data tightly
    # gamma: RBF kernel bandwidth
    #        'scale' = 1/(n_features * X.var()) — data-driven, recommended
    #        Fixed values test whether 'scale' is already optimal
    _param_grid = {
        'svm__C'    : [0.01, 0.1, 1, 10, 100],
        'svm__gamma': ['scale', 0.01, 0.001],
    }
    _n_combos = len(_param_grid['svm__C']) * len(_param_grid['svm__gamma'])
    print(f'  Hyperparameter grid: {_n_combos} combinations')
    print(f'    C     : {_param_grid["svm__C"]}')
    print(f'    gamma : {_param_grid["svm__gamma"]}')

    # ── Cross-validation strategy ─────────────────────────────────
    # StratifiedKFold: each fold preserves ripe/unripe class ratio
    # shuffle=True: prevents ordering artefacts from patch extraction
    # Val set (X_val) is NOT passed to GridSearchCV — held back as
    # an independent overfitting diagnostic used after CV is complete.
    _cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)

    # ── Grid search ───────────────────────────────────────────────
    # n_jobs=-1: use all available cores (Colab 2-core or stronger instance);
    # avoids nested-parallelism warnings from Platt scaling (probability=True)
    # scoring='f1_macro': primary metric — consistent with all DL experiments
    _grid_search = GridSearchCV(
        estimator=_svm_pipeline,
        param_grid=_param_grid,
        cv=_cv_strategy,
        scoring='f1_macro',
        refit=True,          # re-fits best pipeline on full training set
        n_jobs=2,
        verbose=1,
        return_train_score=True,  # captures train F1 per fold for overfitting check
    )

    print(f'\n  Running GridSearchCV (5-fold StratifiedKFold, macro-F1)...')
    print(f'  Total fits: {_n_combos} combinations × 5 folds = {_n_combos * 5} SVM fits')
    print(f'  Estimated time: 3-10 minutes on Colab...\n')

    _t0 = time.time()
    _grid_search.fit(X_train, y_train)
    _cv_time = time.time() - _t0


    # ── Best parameters & CV score ───────────────────────────────
    SVM_BEST_PARAMS  = _grid_search.best_params_
    SVM_BEST_CV_F1   = _grid_search.best_score_
    SVM_MODEL        = _grid_search.best_estimator_   # Pipeline, re-fitted on full train set

    print(f'\n  GridSearchCV complete in {_cv_time:.1f} s')
    print(f'  Best params   : {SVM_BEST_PARAMS}')
    print(f'  Best CV F1    : {SVM_BEST_CV_F1:.4f}  (mean over 5 folds)')


    # ── CV results table (top 5 combinations) ────────────────────
    _cv_df = pd.DataFrame(_grid_search.cv_results_)
    _cv_top5 = (
        _cv_df[['param_svm__C', 'param_svm__gamma',
                 'mean_train_score', 'mean_test_score', 'std_test_score']]
        .sort_values('mean_test_score', ascending=False)
        .head(5)
        .reset_index(drop=True)
    )
    _cv_top5.columns = ['C', 'gamma', 'mean_train_F1', 'mean_val_F1', 'std_val_F1']
    print(f'\n  Top-5 hyperparameter combinations (by mean CV macro-F1):')
    print(_cv_top5.to_string(index=False, float_format='{:.4f}'.format))


    # ── Overfitting diagnostic: train vs val vs test ──────────────
    # Three-way check:
    #   train F1: computed on full X_train after refit (should be high)
    #   val F1:   X_val was never seen during GridSearchCV (clean diagnostic)
    #   If train F1 >> val F1 (gap > 0.15): overfitting flagged

    _y_train_pred = SVM_MODEL.predict(X_train)
    _y_val_pred   = SVM_MODEL.predict(X_val)

    _train_f1 = f1_score(y_train, _y_train_pred, average='macro')
    _val_f1   = f1_score(y_val,   _y_val_pred,   average='macro')
    _overfit_gap = _train_f1 - _val_f1

    print(f'\n  Overfitting diagnostic:')
    print(f'    Training set F1 (macro) : {_train_f1:.4f}')
    print(f'    CV score (5-fold mean)  : {SVM_BEST_CV_F1:.4f}')
    print(f'    Val set F1 (macro)      : {_val_f1:.4f}  [held-out, not used in CV]')
    print(f'    Overfitting gap         : {_overfit_gap:.4f}  (train - val)')

    if _overfit_gap > 0.15:
        print(f'  [WARNING] Overfitting gap > 0.15 — model is overfitting (under-regularised).')
        print(f'            Consider reducing C (stronger regularisation) or feature dimensionality.')
    else:
        print(f'  [OK] Overfitting gap within acceptable range (<= 0.15).')


    # ── Save fitted pipeline to disk ─────────────────────────────
    _svm_path = DIRS['model_svm'] / 'svm_pipeline_E2.pkl'
    joblib.dump(SVM_MODEL, _svm_path)
    print(f'\n  Model saved: {_svm_path}')


    # ── Update metadata ───────────────────────────────────────────
    METADATA['svm_training'] = {
        'kernel'          : 'rbf',
        'cv_strategy'     : '5-fold StratifiedKFold',
        'scoring'         : 'f1_macro',
        'param_grid'      : {k: [str(v) for v in vals] for k, vals in _param_grid.items()},
        'n_combinations'  : _n_combos,
        'best_params'     : {k: str(v) for k, v in SVM_BEST_PARAMS.items()},
        'best_cv_f1'      : round(float(SVM_BEST_CV_F1), 4),
        'train_f1'        : round(float(_train_f1), 4),
        'val_f1'          : round(float(_val_f1), 4),
        'overfit_gap'     : round(float(_overfit_gap), 4),
        'cv_time_s'       : round(_cv_time, 1),
        'model_path'      : str(_svm_path),
    }
    with open(metadata_path, 'w') as f:
        json.dump(METADATA, f, indent=2)

    # Persist CV results for cache reload
    with open(DIRS['model_svm'] / 'svm_cv_results.json', 'w') as _f:
        json.dump({'best_params': {k: str(v) for k, v in SVM_BEST_PARAMS.items()},
                   'best_cv_f1': round(float(SVM_BEST_CV_F1), 4)}, _f, indent=2)

    print(f'  Metadata updated : {metadata_path}')
    print(f'\n[OK] Step 5.2 complete — SVM trained, hyperparameters tuned, model saved.')


### Step 5.3 — SVM Evaluation on Daytime Test Set (E2-Day)

Single evaluation pass of the frozen SVM on the held-out daytime test set. Reports macro-F1 (primary, with 95% bootstrap CI), PR-AUC, ROC-AUC, MCC, accuracy, per-class P/R/F1, and confusion matrix. Permutation feature importance is computed on the **validation set** (not test) to avoid indirect test influence. All metrics are saved to `metrics/E2_day_metrics.json` in the standard schema used by all five experiments.

In [ ]:
# ============================================================
# STEP 5.3 — SVM EVALUATION ON DAYTIME TEST SET (E2-Day)
# ============================================================
from sklearn.metrics import (
    average_precision_score, roc_auc_score, matthews_corrcoef,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.inspection import permutation_importance

print_section('Step 5.3 — E2-Day: SVM Evaluation on Daytime Test Set')


# ── Single evaluation pass — test set used exactly once ──────
_y_pred_day  = SVM_MODEL.predict(X_test_day)
_y_prob_day  = SVM_MODEL.predict_proba(X_test_day)[:, 1]  # P(unripe) for ROC/PR


# ── Core metrics ─────────────────────────────────────────────
_macro_f1  = f1_score(y_test_day, _y_pred_day, average='macro')
_auc_roc   = roc_auc_score(y_test_day, _y_prob_day)
_auc_pr    = average_precision_score(y_test_day, _y_prob_day)
_mcc       = matthews_corrcoef(y_test_day, _y_pred_day)
_accuracy  = float((y_test_day == _y_pred_day).mean())
_cm        = confusion_matrix(y_test_day, _y_pred_day)
_report    = classification_report(
    y_test_day, _y_pred_day,
    target_names=CLASS_NAMES, output_dict=True
)


# ── Bootstrap 95% CI on macro-F1 (primary metric only) ───────
# Percentile bootstrap: resample test predictions N=1000 times,
# compute macro-F1 each time, take 2.5th and 97.5th percentiles.
# No distributional assumption — valid for bounded, skewed metrics.
def _bootstrap_macro_f1(y_true, y_pred, n_boot=1000, seed=GLOBAL_SEED):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    boot_scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_scores[i] = f1_score(y_true[idx], y_pred[idx],
                                   average='macro', zero_division=0)
    ci_low  = float(np.percentile(boot_scores, 2.5))
    ci_high = float(np.percentile(boot_scores, 97.5))
    return ci_low, ci_high

_ci_low, _ci_high = _bootstrap_macro_f1(y_test_day, _y_pred_day)


# ── Print metric summary ──────────────────────────────────────
print(f'\n  E2-Day — SVM Performance on Daytime Test Set')
print(f'  {"=" * 50}')
print(f'  Macro-F1           : {_macro_f1:.4f}  [95% CI: {_ci_low:.4f} – {_ci_high:.4f}]')
print(f'  PR-AUC             : {_auc_pr:.4f}')
print(f'  ROC-AUC            : {_auc_roc:.4f}')
print(f'  MCC                : {_mcc:.4f}')
print(f'  Accuracy           : {_accuracy:.4f}')
print(f'\n  Per-class breakdown:')
print(f'  {"Class":<10}  {"Precision":>10}  {"Recall":>8}  {"F1":>8}  {"Support":>8}')
print(f'  {"-" * 50}')
for _cls in CLASS_NAMES:
    _r = _report[_cls]
    print(f'  {_cls:<10}  {_r["precision"]:>10.4f}  {_r["recall"]:>8.4f}  {_r["f1-score"]:>8.4f}  {int(_r["support"]):>8}')


# ── Figure 1: Confusion matrices (raw + normalised) ───────────
_fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(10, 4))
_fig.suptitle('E2-Day: SVM Confusion Matrix — Daytime Test Set',
               fontsize=13, fontweight='bold')

# Raw counts
import seaborn as sns
sns.heatmap(_cm, annot=True, fmt='d', cmap='Blues', ax=_ax1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='grey')
_ax1.set_title('Raw Counts')
_ax1.set_ylabel('True Label')
_ax1.set_xlabel('Predicted Label')

# Row-normalised (recall per class)
_cm_norm = _cm.astype(float) / _cm.sum(axis=1, keepdims=True)
sns.heatmap(_cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=_ax2,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, linewidths=0.5, linecolor='grey')
_ax2.set_title('Row-Normalised (Recall per Class)')
_ax2.set_ylabel('True Label')
_ax2.set_xlabel('Predicted Label')

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'E2_day_confusion_matrix.png', dpi=300)
plt.show()
print(f'\n  Figure saved → E2_day_confusion_matrix.png')


# ── Figure 2: ROC + PR curves (side by side) ─────────────────
_fig2, (_ax3, _ax4) = plt.subplots(1, 2, figsize=(11, 4))
_fig2.suptitle('E2-Day: SVM — ROC & Precision-Recall Curves',
                fontsize=13, fontweight='bold')

RocCurveDisplay.from_predictions(
    y_test_day, _y_prob_day, ax=_ax3,
    name=f'SVM (AUC={_auc_roc:.3f})',
    color='#2980b9'
)
_ax3.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
_ax3.set_title('ROC Curve')
_ax3.legend(fontsize=9)
sns.despine(ax=_ax3)

PrecisionRecallDisplay.from_predictions(
    y_test_day, _y_prob_day, ax=_ax4,
    name=f'SVM (AP={_auc_pr:.3f})',
    color='#27ae60'
)
_ax4.set_title('Precision-Recall Curve')
_ax4.legend(fontsize=9)
sns.despine(ax=_ax4)

_fig2.tight_layout()
_fig2.savefig(DIRS['figures'] / 'E2_day_roc_pr_curves.png', dpi=300)
plt.show()
print(f'  Figure saved → E2_day_roc_pr_curves.png')


# ── Figure 3: Permutation feature importance (on VAL set) ─────
# Computed on X_val to avoid any indirect test influence.
# Groups features by component for interpretable importance.
_perm_result = permutation_importance(
    SVM_MODEL, X_val, y_val,
    n_repeats=20,
    random_state=GLOBAL_SEED,
    scoring='f1_macro',
    n_jobs=2
)

# Aggregate by feature group (mean importance per group)
_feat_groups = [
    ('HSV Histogram',    0,   96),
    ('LBP Texture',     96,  114),
    ('HOG Gradients',  114,  195),
    ('LAB a*/b*',      195,  259),
]
_group_importances = []
for _name, _start, _end in _feat_groups:
    _group_mean = _perm_result.importances_mean[_start:_end].mean()
    _group_std  = _perm_result.importances_std[_start:_end].mean()
    _group_importances.append((_name, _group_mean, _group_std))

_group_importances.sort(key=lambda x: x[1], reverse=True)
_names  = [g[0] for g in _group_importances]
_means  = [g[1] for g in _group_importances]
_stds   = [g[2] for g in _group_importances]

_fig3, _ax5 = plt.subplots(figsize=(7, 4))
_colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
_bars = _ax5.barh(_names, _means, xerr=_stds,
                   color=_colors[:len(_names)], edgecolor='black',
                   linewidth=0.6, capsize=4, height=0.5)
_ax5.set_xlabel('Mean Permutation Importance (macro-F1 drop)')
_ax5.set_title('Feature Group Importance\n(Permutation, N=20, val set)', fontsize=12)
_ax5.axvline(0, color='black', linewidth=0.8)
sns.despine(ax=_ax5)
_fig3.tight_layout()
_fig3.savefig(DIRS['figures'] / 'E2_day_feature_importance.png', dpi=300)
plt.show()

print(f'  Feature group importance (val set, macro-F1 drop):')
for _n, _m, _s in _group_importances:
    print(f'    {_n:<20} : {_m:+.4f} ± {_s:.4f}')
print(f'  Figure saved → E2_day_feature_importance.png')


# ── Save metrics in standard schema ──────────────────────────
E2_DAY_METRICS = {
    'experiment'    : 'E2-Day',
    'model'         : 'SVM (RBF)',
    'split'         : 'test_day',
    'n_samples'     : int(len(y_test_day)),
    'macro_f1'      : round(float(_macro_f1),  4),
    'macro_f1_ci_95': [round(_ci_low, 4), round(_ci_high, 4)],
    'auc_pr'        : round(float(_auc_pr),    4),
    'auc_roc'       : round(float(_auc_roc),   4),
    'mcc'           : round(float(_mcc),       4),
    'accuracy'      : round(float(_accuracy),  4),
    'per_class'     : {
        cls: {
            'precision': round(float(_report[cls]['precision']), 4),
            'recall'   : round(float(_report[cls]['recall']),    4),
            'f1'       : round(float(_report[cls]['f1-score']),  4),
            'support'  : int(_report[cls]['support']),
        }
        for cls in CLASS_NAMES
    },
    'confusion_matrix' : _cm.tolist(),
    'feature_importance': {
        n: round(float(m), 4) for n, m, _ in _group_importances
    },
}

_metrics_path = DIRS['metrics'] / 'E2_day_metrics.json'
with open(_metrics_path, 'w') as f:
    json.dump(E2_DAY_METRICS, f, indent=2)

print(f'\n  Metrics saved → {_metrics_path}')
print(f'\n[OK] Step 5.3 complete — E2-Day SVM evaluation done.')


### Step 5.4 — SVM Evaluation on Simulated Night Test Set (E2-Night)

Evaluates the **same trained `SVM_MODEL`** (fitted on daytime data in Step 5.2) on `X_test_night` — the LAB-shifted test patches generated in Step 4.2. No retraining; no scaler refitting. Any performance difference from E2-Day is attributable purely to the LED domain shift.

- **`StandardScaler` note**: fitted on `X_train` (daytime). Applied to night features, it is intentionally mis-calibrated — modelling real deployment where no night data was seen during training.
- **Figures**: confusion matrix (orange palette), ROC + PR curves, Day vs Night comparison bar chart with Δ annotations.
- **Output**: `E2_NIGHT_METRICS` global + `metrics/E2_night_metrics.json` (includes `day_night_delta` key for Phase 10 auto-assembly).

In [ ]:
# ============================================================
# STEP 5.4 — SVM EVALUATION ON SIMULATED NIGHT TEST SET (E2-Night)
# ============================================================
from sklearn.metrics import (
    average_precision_score, roc_auc_score, matthews_corrcoef,
    RocCurveDisplay, PrecisionRecallDisplay,
)

print_section('Step 5.4 — E2-Night: SVM Evaluation on Simulated Night Test Set')

# ── Prerequisite guards (F5.4.d: globals() is correct idiom in Jupyter) ──
assert 'SVM_MODEL'            in globals(), 'SVM_MODEL not found — run Step 5.2 first'
assert 'X_test_night'         in globals(), 'X_test_night not found — run Step 5.1 first'
assert 'E2_DAY_METRICS'       in globals(), 'E2_DAY_METRICS not found — run Step 5.3 first'
# F5.4.f: callable() is always True for functions; check existence via globals() instead
assert '_bootstrap_macro_f1'  in globals(), '_bootstrap_macro_f1 not found — run Step 5.3 first'

# ── Idempotency guard ─────────────────────────────────────────────────────
_night_metrics_path = DIRS['metrics'] / 'E2_night_metrics.json'
if _night_metrics_path.exists():
    print('  Step 5.4 already complete — loading cached E2_NIGHT_METRICS.')
    with open(_night_metrics_path) as _f:
        E2_NIGHT_METRICS = json.load(_f)
    print(f'  E2-Night macro-F1 : {E2_NIGHT_METRICS["macro_f1"]:.4f}')
else:
    # ── Single evaluation pass — test set used exactly once ──────────────
    # SVM_MODEL contains a StandardScaler fitted on X_train (daytime).
    # Applied to X_test_night, the scaler is intentionally mis-calibrated —
    # modelling real deployment where no night data was seen during fitting.
    # Ben-David et al. (2010): this covariate shift directly measures the
    # H-divergence between day and night feature distributions.
    _y_pred_night = SVM_MODEL.predict(X_test_night)
    _y_prob_night = SVM_MODEL.predict_proba(X_test_night)[:, 1]  # P(unripe)

    # ── Core metrics (identical pipeline to Step 5.3) ────────────────────
    _macro_f1_n = f1_score(y_test_night, _y_pred_night, average='macro')
    _auc_roc_n  = roc_auc_score(y_test_night, _y_prob_night)
    _auc_pr_n   = average_precision_score(y_test_night, _y_prob_night)
    _mcc_n      = matthews_corrcoef(y_test_night, _y_pred_night)
    _accuracy_n = float((y_test_night == _y_pred_night).mean())
    _cm_n       = confusion_matrix(y_test_night, _y_pred_night)
    _report_n   = classification_report(
        y_test_night, _y_pred_night,
        target_names=CLASS_NAMES, output_dict=True
    )

    # ── Bootstrap 95% CI on macro-F1 ─────────────────────────────────────
    _ci_low_n, _ci_high_n = _bootstrap_macro_f1(y_test_night, _y_pred_night)

    # ── Day → Night performance delta ─────────────────────────────────────
    _delta_f1  = round(_macro_f1_n  - E2_DAY_METRICS['macro_f1'],  4)
    _delta_pr  = round(_auc_pr_n    - E2_DAY_METRICS['auc_pr'],    4)
    _delta_roc = round(_auc_roc_n   - E2_DAY_METRICS['auc_roc'],   4)
    _delta_mcc = round(_mcc_n       - E2_DAY_METRICS['mcc'],       4)
    _delta_acc = round(_accuracy_n  - E2_DAY_METRICS['accuracy'],  4)

    # ── Print metric summary ──────────────────────────────────────────────
    print(f'\n  E2-Night — SVM Performance on Simulated Night Test Set')
    print(f'  {"=" * 58}')
    print(f'  Macro-F1  : {_macro_f1_n:.4f}  [95% CI: {_ci_low_n:.4f} \u2013 {_ci_high_n:.4f}]')
    print(f'  PR-AUC    : {_auc_pr_n:.4f}')
    print(f'  ROC-AUC   : {_auc_roc_n:.4f}')
    print(f'  MCC       : {_mcc_n:.4f}')
    print(f'  Accuracy  : {_accuracy_n:.4f}')

    print(f'\n  Day \u2192 Night Performance Delta (SVM Baseline)')
    print(f'  {"=" * 58}')
    print(f'  {"Metric":<14}  {"Day":>8}  {"Night":>8}  {"\u0394 (Night\u2212Day)":>14}')
    print(f'  {"-" * 50}')
    for _mn, _dv, _nv, _delta in [
        ('Macro-F1',  E2_DAY_METRICS['macro_f1'],  _macro_f1_n,  _delta_f1),
        ('PR-AUC',    E2_DAY_METRICS['auc_pr'],    _auc_pr_n,    _delta_pr),
        ('ROC-AUC',   E2_DAY_METRICS['auc_roc'],   _auc_roc_n,   _delta_roc),
        ('MCC',       E2_DAY_METRICS['mcc'],        _mcc_n,       _delta_mcc),
        ('Accuracy',  E2_DAY_METRICS['accuracy'],   _accuracy_n,  _delta_acc),
    ]:
        _sign = '+' if _delta >= 0 else ''
        print(f'  {_mn:<14}  {_dv:>8.4f}  {_nv:>8.4f}  {_sign}{_delta:>13.4f}')

    # ── F5.4.e — CI overlap significance test ───────────────────────────
    # The degradation is statistically significant (at 95% level) if the
    # day and night bootstrap CIs do NOT overlap. This is a conservative
    # but straightforward test appropriate for a graded submission.
    # Note: CIs are on the *same* test patches (paired), so non-overlap
    # is a strict criterion — overlapping CIs can still be significant
    # with a paired test, but non-overlap is unambiguous evidence.
    _day_ci_low, _day_ci_high = E2_DAY_METRICS['macro_f1_ci_95']
    _ci_overlap = not (_ci_high_n < _day_ci_low or _ci_low_n > _day_ci_high)
    print(f'\n  Statistical Significance of Domain-Shift Degradation (SVM)')
    print(f'  {"=" * 58}')
    print(f'  Day   F1 95% CI : [{_day_ci_low:.4f}, {_day_ci_high:.4f}]')
    print(f'  Night F1 95% CI : [{_ci_low_n:.4f}, {_ci_high_n:.4f}]')
    if not _ci_overlap:
        print(f'  CI overlap      : NO  → degradation is statistically significant (p<0.05 approx)')
    else:
        print(f'  CI overlap      : YES → degradation not statistically significant at 95% level')
        print(f'  [NOTE] CIs overlap but gap may still be meaningful — consider paired t-test.')

    print(f'\n  Per-class breakdown (night):')
    print(f'  {"Class":<10}  {"Precision":>10}  {"Recall":>8}  {"F1":>8}  {"Support":>8}')
    print(f'  {"-" * 50}')
    for _cls in CLASS_NAMES:
        _r = _report_n[_cls]
        print(f'  {_cls:<10}  {_r["precision"]:>10.4f}  {_r["recall"]:>8.4f}'
              f'  {_r["f1-score"]:>8.4f}  {int(_r["support"]):>8}')

    # ── Figure 1: Night confusion matrices (raw + normalised) ─────────────
    # Orange palette visually distinguishes night from E2-Day blue in report.
    _fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(10, 4))
    _fig.suptitle('E2-Night: SVM Confusion Matrix \u2014 Simulated Night Test Set',
                   fontsize=13, fontweight='bold')

    sns.heatmap(_cm_n, annot=True, fmt='d', cmap='Oranges', ax=_ax1,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, linecolor='grey')
    _ax1.set_title('Raw Counts')
    _ax1.set_ylabel('True Label')
    _ax1.set_xlabel('Predicted Label')

    _cm_n_norm = _cm_n.astype(float) / _cm_n.sum(axis=1, keepdims=True)
    sns.heatmap(_cm_n_norm, annot=True, fmt='.2f', cmap='Oranges', ax=_ax2,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                vmin=0, vmax=1, linewidths=0.5, linecolor='grey')
    _ax2.set_title('Row-Normalised (Recall per Class)')
    _ax2.set_ylabel('True Label')
    _ax2.set_xlabel('Predicted Label')

    _fig.tight_layout()
    _fig.savefig(DIRS['figures'] / 'E2_night_confusion_matrix.png', dpi=300)
    plt.show()
    print(f'\n  Figure saved \u2192 E2_night_confusion_matrix.png')

    # ── Figure 2: Night ROC + PR curves ──────────────────────────────────
    _fig2, (_ax3, _ax4) = plt.subplots(1, 2, figsize=(11, 4))
    _fig2.suptitle('E2-Night: SVM \u2014 ROC & Precision-Recall Curves',
                    fontsize=13, fontweight='bold')

    RocCurveDisplay.from_predictions(
        y_test_night, _y_prob_night, ax=_ax3,
        name=f'SVM Night (AUC={_auc_roc_n:.3f})',
        color='#e67e22'
    )
    _ax3.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    _ax3.set_title('ROC Curve')
    _ax3.legend(fontsize=9)
    sns.despine(ax=_ax3)

    PrecisionRecallDisplay.from_predictions(
        y_test_night, _y_prob_night, ax=_ax4,
        name=f'SVM Night (AP={_auc_pr_n:.3f})',
        color='#d35400'
    )
    _ax4.set_title('Precision-Recall Curve')
    _ax4.legend(fontsize=9)
    sns.despine(ax=_ax4)

    _fig2.tight_layout()
    _fig2.savefig(DIRS['figures'] / 'E2_night_roc_pr_curves.png', dpi=300)
    plt.show()
    print(f'  Figure saved \u2192 E2_night_roc_pr_curves.png')

    # ── Figure 3: Day vs Night side-by-side bar chart ─────────────────────
    # Key domain-shift diagnostic; delta annotated above each night bar.
    _fig3, _ax5 = plt.subplots(figsize=(9, 4))
    _metric_labels = ['Macro-F1', 'PR-AUC', 'ROC-AUC', 'MCC', 'Accuracy']
    _day_vals   = [
        E2_DAY_METRICS['macro_f1'], E2_DAY_METRICS['auc_pr'],
        E2_DAY_METRICS['auc_roc'],  E2_DAY_METRICS['mcc'],
        E2_DAY_METRICS['accuracy'],
    ]
    _night_vals = [_macro_f1_n, _auc_pr_n, _auc_roc_n, _mcc_n, _accuracy_n]
    _deltas     = [_delta_f1,  _delta_pr,  _delta_roc,  _delta_mcc, _delta_acc]

    _x     = np.arange(len(_metric_labels))
    _bar_w = 0.35
    _ax5.bar(_x - _bar_w / 2, _day_vals,   _bar_w,
             label='Day (E2-Day)',    color='#2980b9', alpha=0.85,
             edgecolor='black', linewidth=0.5)
    _ax5.bar(_x + _bar_w / 2, _night_vals, _bar_w,
             label='Night (E2-Night)', color='#e67e22', alpha=0.85,
             edgecolor='black', linewidth=0.5)

    # Annotate delta above each night bar
    for _xi, (_nv, _dlt) in enumerate(zip(_night_vals, _deltas)):
        _sign  = '+' if _dlt >= 0 else ''
        _color = '#27ae60' if _dlt >= 0 else '#c0392b'
        _ax5.text(
            _xi + _bar_w / 2, _nv + 0.012,
            f'{_sign}{_dlt:.3f}',
            ha='center', va='bottom', fontsize=8, color=_color, fontweight='bold'
        )

    _ax5.set_xticks(_x)
    _ax5.set_xticklabels(_metric_labels, fontsize=10)
    _ax5.set_ylabel('Score')
    _ax5.set_ylim(0, 1.18)
    _ax5.set_title(
        'SVM: Day vs Night Performance (E2 Baseline)\n\u0394 = Night \u2212 Day',
        fontsize=12, fontweight='bold'
    )
    _ax5.legend(fontsize=10)
    sns.despine(ax=_ax5)
    _fig3.tight_layout()
    _fig3.savefig(DIRS['figures'] / 'E2_day_vs_night_comparison.png', dpi=300)
    plt.show()
    print(f'  Figure saved \u2192 E2_day_vs_night_comparison.png')

    # ── Save metrics in standard schema ───────────────────────────────────
    E2_NIGHT_METRICS = {
        'experiment'     : 'E2-Night',
        'model'          : 'SVM (RBF)',
        'split'          : 'test_night',
        'n_samples'      : int(len(y_test_night)),
        'macro_f1'       : round(float(_macro_f1_n),  4),
        'macro_f1_ci_95' : [round(_ci_low_n, 4), round(_ci_high_n, 4)],
        'degradation_significant_95ci': not _ci_overlap,
        'auc_pr'         : round(float(_auc_pr_n),    4),
        'auc_roc'        : round(float(_auc_roc_n),   4),
        'mcc'            : round(float(_mcc_n),       4),
        'accuracy'       : round(float(_accuracy_n),  4),
        'per_class'      : {
            cls: {
                'precision': round(float(_report_n[cls]['precision']), 4),
                'recall'   : round(float(_report_n[cls]['recall']),    4),
                'f1'       : round(float(_report_n[cls]['f1-score']),  4),
                'support'  : int(_report_n[cls]['support']),
            }
            for cls in CLASS_NAMES
        },
        'confusion_matrix' : _cm_n.tolist(),
        'day_night_delta'  : {
            'macro_f1' : _delta_f1,
            'auc_pr'   : _delta_pr,
            'auc_roc'  : _delta_roc,
            'mcc'      : _delta_mcc,
            'accuracy' : _delta_acc,
        },
    }

    with open(_night_metrics_path, 'w') as _f:
        json.dump(E2_NIGHT_METRICS, _f, indent=2)

    print(f'\n  Metrics saved \u2192 {_night_metrics_path}')
    print(f'\n[OK] Step 5.4 complete \u2014 E2-Night SVM evaluation done.')


---

## Phase 6 — Experiment E1: Deep Learning Baseline (Day)

### Step 6.1 — DL Architecture Selection: ConvNeXt-Tiny

**Architecture decision**: **ConvNeXt-Tiny** (Liu et al., *A ConvNet for the 2020s*, CVPR 2022 — Meta AI Research)

#### Candidates evaluated
| Architecture | Params | ImageNet Top-1 | VRAM @bs=64 | Grad-CAM | LO3 |
|---|---|---|---|---|---|
| ResNet-18 (2015) | 11.7M | 69.8% | 1.2 GB | Excellent | low |
| EfficientNetV2-S (2021) | 21.5M | 82.7% | 2.3 GB | Good | medium |
| **ConvNeXt-Tiny (2022)** | **28.6M** | **82.1%** | **2.8 GB** | **Excellent** | **high** |
| Swin Transformer-Tiny (2021) | 27.6M | 81.2% | 3.5–4.0 GB | Good | highest |
| DeiT-Small (2021) | 22.1M | 79.9% | 3.8 GB | Moderate | high |

#### Why ConvNeXt-Tiny
1. **Enterprise-grade modernity**: CVPR 2022 state-of-the-art; titled *A ConvNet for the 2020s* — the canonical CNN–Transformer design convergence example, directly addressing LO3.
2. **Transformer design principles in a CNN**: depthwise 7x7 convolutions, LayerNorm, GELU, inverted bottleneck — ViT-competitive accuracy with CNN spatial locality and standard Grad-CAM compatibility.
3. **T4 safety**: 2.8 GB VRAM at bs=64 — all 3 seeds run without batch-size reduction or gradient checkpointing.
4. **Grad-CAM**: `model.features[-1]` outputs [B, 768, 7x7] spatial maps — zero token-reshaping complexity vs transformer variants.
5. **Small-dataset fine-tuning**: depthwise separable convolutions are more sample-efficient per parameter (Liu et al., 2022, Table 5); stages 2–4 train safely on ~4,000 patches with weight_decay=0.05 in Step 6.4.

#### Why not Swin-Tiny
Requires bs=32 on T4 (VRAM risk), Grad-CAM token reshaping from sequence to 2D (implementation fragility under 3-day deadline), and insufficient VRAM headroom for 3 seeds. Preferred on a dedicated persistent GPU server.

#### Freezing strategy
- **Frozen**: `features[0]` (stem, 56x56) + `features[1]` (Stage 1, 3 blocks, 96ch) — universal low-level features per Zeiler & Fergus (2014)
- **Trainable**: `features[2]–[7]` (Stages 2–4 + downsamples) + `classifier` — task-specific berry texture/colour representations

**Grad-CAM target**: `model.features[-1]` — Stage 4, 7x7 spatial resolution at 768 channels — highest-resolution activation map before global average pooling.


In [ ]:
# ============================================================
# STEP 6.1 -- DL ARCHITECTURE: ConvNeXt-Tiny
# ============================================================
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

print_section('Step 6.1 -- DL Architecture Selection: ConvNeXt-Tiny')

DL_MODEL_NAME  = 'convnext_tiny'
DL_NUM_CLASSES = 2   # binary: ripe (0) vs unripe (1)


def build_convnext_tiny(num_classes=2, pretrained=True):
    """
    Build ConvNeXt-Tiny with ImageNet pretrained weights.
    Replaces the 1000-class Linear head with a num_classes head.

    Stage layout (224x224 input):
      features[0] Stem          Conv2d(3,96,4x4,s=4)+LN  -> [B, 96,56,56]
      features[1] Stage 1       3 ConvNeXtBlocks  96ch   -> [B, 96,56,56]
      features[2] Downsample    LN+Conv2d(96->192,2x2)   -> [B,192,28,28]
      features[3] Stage 2       3 ConvNeXtBlocks 192ch   -> [B,192,28,28]
      features[4] Downsample    LN+Conv2d(192->384,2x2)  -> [B,384,14,14]
      features[5] Stage 3       9 ConvNeXtBlocks 384ch   -> [B,384,14,14]
      features[6] Downsample    LN+Conv2d(384->768,2x2)  -> [B,768, 7, 7]
      features[7] Stage 4       3 ConvNeXtBlocks 768ch   -> [B,768, 7, 7]
      avgpool                   AdaptiveAvgPool2d(1)     -> [B,768, 1, 1]
      classifier                LN2d+Flatten+Linear      -> [B,num_classes]

    Grad-CAM target: features[-1]  (7x7, 768ch)
    Re-used per seed in Step 6.4 -- always call fresh to avoid weight sharing.
    """
    _weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None
    _model   = convnext_tiny(weights=_weights)

    # Replace head robustly -- works for torchvision 0.13 and 0.14+
    _found = False
    for _i, _layer in enumerate(_model.classifier):
        if isinstance(_layer, torch.nn.Linear):
            _in_feat = _layer.in_features   # 768
            _model.classifier[_i] = torch.nn.Linear(_in_feat, num_classes)
            _found = True
            break
    if not _found:
        raise RuntimeError('Linear head not found in ConvNeXt classifier')

    return _model


def freeze_convnext_backbone(model):
    """
    Freeze features[0] (stem) and features[1] (Stage 1, 96ch, 56x56).
    Keep features[2]-[7], avgpool, classifier trainable.

    Rationale (Zeiler & Fergus, 2014; Liu et al., CVPR 2022 Table 5):
      Stem and Stage 1 at 56x56 detect universal low-level features
      (Gabor filters, colour blobs) applicable across visual domains.
      Freezing avoids overwriting them with only ~4K samples.
      Stages 2-4 encode task-specific berry texture/colour/shape
      representations that require fine-tuning for our domain.
      ConvNeXt depthwise convolutions are sample-efficient per param;
      training stages 2-4 with weight_decay=0.05 prevents overfitting.
    """
    model.features[0].requires_grad_(False)   # Stem
    model.features[1].requires_grad_(False)   # Stage 1
    return model


# -- Instantiate, freeze, send to device -----------------------------------
set_seed(GLOBAL_SEED)
DL_MODEL = build_convnext_tiny(num_classes=DL_NUM_CLASSES, pretrained=True)
DL_MODEL = freeze_convnext_backbone(DL_MODEL)
DL_MODEL = DL_MODEL.to(DEVICE)


# -- Parameter audit -------------------------------------------------------
_total_params     = sum(p.numel() for p in DL_MODEL.parameters())
_trainable_params = sum(p.numel() for p in DL_MODEL.parameters() if p.requires_grad)
_frozen_params    = _total_params - _trainable_params

print(f'\n  Architecture        : ConvNeXt-Tiny  (Liu et al., CVPR 2022 -- Meta AI)')
print(f'  Pretrained weights  : ImageNet-1K  (IMAGENET1K_V1)')
print(f'  Classification head : Linear(768, {DL_NUM_CLASSES})   [ripe=0, unripe=1]')
print(f'  Device              : {DEVICE}')
print(f'\n  Parameter summary:')
print(f'    Total       : {_total_params:>12,}')
print(f'    Trainable   : {_trainable_params:>12,}  ({100*_trainable_params/_total_params:.1f}%)')
print(f'    Frozen      : {_frozen_params:>12,}  ({100*_frozen_params/_total_params:.1f}%)')


# -- Layer-level freeze audit ----------------------------------------------
print(f'\n  Layer freeze audit:')
_stage_map = [
    ('features[0] Stem           96ch', DL_MODEL.features[0]),
    ('features[1] Stage1  3blks  96ch', DL_MODEL.features[1]),
    ('features[2] Downsamp      192ch', DL_MODEL.features[2]),
    ('features[3] Stage2  3blks 192ch', DL_MODEL.features[3]),
    ('features[4] Downsamp      384ch', DL_MODEL.features[4]),
    ('features[5] Stage3  9blks 384ch', DL_MODEL.features[5]),
    ('features[6] Downsamp      768ch', DL_MODEL.features[6]),
    ('features[7] Stage4  3blks 768ch', DL_MODEL.features[7]),
    ('classifier',                      DL_MODEL.classifier),
]
for _lname, _mod in _stage_map:
    _ps = list(_mod.parameters())
    if not _ps:
        continue
    _rg = [p.requires_grad for p in _ps]
    _status = 'TRAINABLE' if all(_rg) else ('FROZEN   ' if not any(_rg) else 'PARTIAL  ')
    _np = sum(p.numel() for p in _ps)
    print(f'    {_lname:<35}  {_status}  {_np:>10,} params')


# -- Grad-CAM target layer -------------------------------------------------
# features[-1] == features[7]: Stage 4, output [B, 768, 7, 7]
# Canonical ConvNeXt Grad-CAM target: highest spatial resolution before pooling.
# Ref: https://github.com/jacobgil/pytorch-grad-cam (ConvNeXt example)
GRADCAM_TARGET_LAYER = DL_MODEL.features[-1]
print(f'\n  Grad-CAM target     : DL_MODEL.features[-1]  (Stage 4, 7x7, 768ch)')
print(f'  Layer type          : {type(GRADCAM_TARGET_LAYER).__name__}')


# -- Forward pass sanity check ---------------------------------------------
DL_MODEL.eval()
with torch.no_grad():
    _x_dummy = torch.zeros(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
    _y_dummy = DL_MODEL(_x_dummy)
assert _y_dummy.shape == (2, DL_NUM_CLASSES), \
    f'Expected (2,{DL_NUM_CLASSES}), got {tuple(_y_dummy.shape)}'
print(f'\n  Forward pass check  : PASSED  {tuple(_x_dummy.shape)} -> {tuple(_y_dummy.shape)}')
del _x_dummy, _y_dummy
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()


# -- VRAM estimate ---------------------------------------------------------
_vram_gb = round(2.8 * (RECOMMENDED_BATCH_SIZE / 64), 2)
print(f'\n  VRAM estimate (bs={RECOMMENDED_BATCH_SIZE}): ~{_vram_gb} GB')
print(f'  RECOMMENDED_BATCH_SIZE : {RECOMMENDED_BATCH_SIZE}')


# -- Persist metadata ------------------------------------------------------
METADATA['dl_architecture'] = {
    'model'            : DL_MODEL_NAME,
    'paper'            : 'Liu et al., A ConvNet for the 2020s, CVPR 2022',
    'pretrained'       : 'ImageNet-1K (IMAGENET1K_V1)',
    'num_classes'      : DL_NUM_CLASSES,
    'class_map'        : {'0': 'ripe', '1': 'unripe'},
    'total_params'     : _total_params,
    'trainable_params' : _trainable_params,
    'frozen_params'    : _frozen_params,
    'frozen_layers'    : ['features[0] stem', 'features[1] stage1'],
    'trainable_layers' : ['features[2]-[7]', 'classifier'],
    'gradcam_target'   : 'features[-1] (Stage 4, 7x7, 768ch)',
    'classifier_head'  : f'Linear(768, {DL_NUM_CLASSES})',
    'vram_estimate_bs64': '~2.8 GB',
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)

print(f'\n  Metadata updated    : {metadata_path}')
print(f'\n[OK] Step 6.1 complete -- ConvNeXt-Tiny instantiated, frozen, verified.')


### Step 6.2 — Transfer Learning Configuration

Defines all training hyperparameter globals and the `create_optimizer_scheduler()` factory function. No training yet — Step 6.4 simply calls this factory once per seed.

**Key decisions vs. baseline fine-tuning recipes:**

| Hyperparameter | Value | Rationale |
|---|---|---|
| Optimizer | AdamW | Liu et al. CVPR 2022 canonical recipe; decoupled WD critical for LayerNorm |
| Base LR | 3e-4 | Geometric midpoint of documented fine-tuning window [1e-4, 5e-4] for ConvNeXt; conservative for 4K dataset |
| LLRD decay | 0.85 | Layer-wise LR decay per stage (BEiT, MAE); head trains at 3e-4, Stage 2 at ~1.8e-4 |
| Weight decay | 0.05 | ConvNeXt canonical; **excluded** from LayerNorm and bias params (Loshchilov & Hutter 2019) |
| Warmup | 3 epochs | ~5% of total steps — prevents large gradients from random head (Thrun 1995) |
| Scheduler | Cosine (37ep) | SquentialLR (LinearLR → CosineAnnealingLR); smooth decay, no performance cliffs |
| Label smoothing | 0.1 | Szegedy et al. 2016; prevents overconfidence on small dataset |
| Mixed precision | AMP | torch.cuda.amp; ~30–50% faster on T4 Tensor Cores, ~1.4 GB VRAM saved; deterministic on single-GPU |


In [ ]:
# ============================================================
# STEP 6.2 -- TRANSFER LEARNING CONFIGURATION
# ============================================================
import math
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

print_section('Step 6.2 -- Transfer Learning Configuration')


# ============================================================
# A. TRAINING HYPERPARAMETER GLOBALS
# ============================================================

# -- Optimizer -----------------------------------------------------------
DL_OPTIMIZER_NAME   = 'AdamW'
DL_LR               = 3e-4          # geometric midpoint of fine-tuning window [1e-4, 5e-4]
DL_WEIGHT_DECAY     = 0.05          # ConvNeXt canonical; applied selectively (see param groups)
DL_BETAS            = (0.9, 0.999)  # AdamW defaults matching Liu et al. (2022)
DL_EPS              = 1e-8
DL_LLRD_DECAY       = 0.85          # LR multiplier per stage deeper: head > Stage4 > Stage3 > Stage2

# -- Scheduler -----------------------------------------------------------
DL_SCHEDULER_NAME       = 'SequentialLR(LinearLR+CosineAnnealingLR)'
DL_WARMUP_EPOCHS        = 3                               # ~5% of total steps prevents head-gradient damage
DL_MAX_EPOCHS           = 40
DL_COSINE_EPOCHS        = DL_MAX_EPOCHS - DL_WARMUP_EPOCHS  # 37 epochs of cosine decay
DL_LR_WARMUP_INIT       = DL_LR * 0.01                   # 3e-6: warmup start
DL_LR_MIN               = DL_LR * 0.01                   # 3e-6: cosine floor, tied to base LR

# -- Early stopping ------------------------------------------------------
DL_EARLY_STOP_PATIENCE  = 10        # monitor val macro-F1

# -- Loss ----------------------------------------------------------------
DL_LABEL_SMOOTHING      = 0.1       # Szegedy et al. (2016); Liu et al. (2022) training recipe

# -- Batch & precision ---------------------------------------------------
DL_BATCH_SIZE           = RECOMMENDED_BATCH_SIZE  # 64 on T4
DL_USE_AMP              = (DEVICE.type == 'cuda')  # AMP only on GPU; no-op on CPU fallback


print('  Hyperparameter globals:')
print(f'    Optimizer            : {DL_OPTIMIZER_NAME}  betas={DL_BETAS}  eps={DL_EPS}')
print(f'    Base LR              : {DL_LR}')
print(f'    LLRD decay per stage : {DL_LLRD_DECAY}  (head=LR, Stage4=LR*{DL_LLRD_DECAY}, ...')
print(f'    Weight decay         : {DL_WEIGHT_DECAY}  (excl. LayerNorm + bias params)')
print(f'    Warmup               : {DL_WARMUP_EPOCHS} epochs  ({DL_LR_WARMUP_INIT:.1e} -> {DL_LR:.1e})')
print(f'    Cosine decay         : {DL_COSINE_EPOCHS} epochs  ({DL_LR:.1e} -> {DL_LR_MIN:.1e})')
print(f'    Max epochs           : {DL_MAX_EPOCHS}')
print(f'    Early stop patience  : {DL_EARLY_STOP_PATIENCE}  (val macro-F1)')
print(f'    Label smoothing      : {DL_LABEL_SMOOTHING}')
print(f'    Batch size           : {DL_BATCH_SIZE}')
print(f'    Mixed precision      : {DL_USE_AMP}  (torch.cuda.amp)')


# ============================================================
# B. CLASS WEIGHTS FROM y_train
# ============================================================
# Formula: w_c = N_total / (n_classes * N_c)  -- sklearn 'balanced' equivalent.
# Ensures equal loss contribution from ripe and unripe classes.
# Consistent with SVM class_weight='balanced' (Phase 5) for methodological parity (LO1).
# If imbalance ratio < 1.5:1 this is still harmless; we keep it for cross-experiment consistency.

_n_total       = len(y_train)
_n_classes     = DL_NUM_CLASSES
_class_counts  = [(y_train == c).sum() for c in range(_n_classes)]
_imbalance_ratio = max(_class_counts) / min(_class_counts)
_cw_list = [_n_total / (_n_classes * _class_counts[c]) for c in range(_n_classes)]
DL_CLASS_WEIGHTS = torch.tensor(_cw_list, dtype=torch.float32).to(DEVICE)

print('\n  Class imbalance analysis (y_train):')
for _c, (_cname, _cnt, _w) in enumerate(zip(CLASS_NAMES, _class_counts, _cw_list)):
    print(f'    class {_c} ({_cname:<8}): {_cnt:>5} samples  ->  weight = {_w:.4f}')
print(f'    Imbalance ratio      : {_imbalance_ratio:.2f}:1')
print(f'    Class weights tensor : {DL_CLASS_WEIGHTS.cpu().tolist()}')


# ============================================================
# C. LOSS FUNCTION
# ============================================================
DL_LOSS_FN = torch.nn.CrossEntropyLoss(
    weight          = DL_CLASS_WEIGHTS,
    label_smoothing = DL_LABEL_SMOOTHING,
)
print(f'\n  Loss function: CrossEntropyLoss')
print(f'    class_weights={[round(w,4) for w in _cw_list]}, label_smoothing={DL_LABEL_SMOOTHING}')


# ============================================================
# D. PARAM GROUP BUILDER (LLRD + SELECTIVE WEIGHT DECAY)
# ============================================================
def _build_param_groups(model, base_lr, weight_decay, llrd_decay):
    """
    Build AdamW parameter groups implementing:
    (1) Layer-wise LR Decay (LLRD): head trains at base_lr;
        each successive stage (4->3->2) decays by llrd_decay.
    (2) Selective weight decay: LayerNorm and all bias params
        are excluded from weight decay (Loshchilov & Hutter 2019).

    ConvNeXt-Tiny stage mapping:
       model.features[0-1]  : FROZEN (stem + Stage1)
       model.features[2-3]  : Stage 2  -> LR = base_lr * llrd_decay^2
       model.features[4-5]  : Stage 3  -> LR = base_lr * llrd_decay^1
       model.features[6-7]  : Stage 4  -> LR = base_lr * llrd_decay^0  (= base_lr)
       model.classifier     : head     -> LR = base_lr

    Returns list of param group dicts ready for AdamW().
    """
    _no_decay_keywords = ('norm', '.bias')

    def _is_no_decay(name):
        return any(kw in name for kw in _no_decay_keywords)

    # Stage boundaries in model.features (0-indexed)
    # features[0-1] frozen; features[2-3]=Stage2, [4-5]=Stage3, [6-7]=Stage4
    _stage_lr = {
        'stage2'     : base_lr * (llrd_decay ** 2),
        'stage3'     : base_lr * (llrd_decay ** 1),
        'stage4'     : base_lr * (llrd_decay ** 0),
        'classifier' : base_lr,
    }

    def _assign_stage(name):
        if name.startswith('classifier'):
            return 'classifier'
        if name.startswith('features.2') or name.startswith('features.3'):
            return 'stage2'
        if name.startswith('features.4') or name.startswith('features.5'):
            return 'stage3'
        if name.startswith('features.6') or name.startswith('features.7'):
            return 'stage4'
        return None  # frozen or unrecognised -> skip

    # Build {(stage, no_decay_flag): list_of_params} dict
    _groups = {}
    for _name, _param in model.named_parameters():
        if not _param.requires_grad:
            continue
        _stage = _assign_stage(_name)
        if _stage is None:
            continue
        _key = (_stage, _is_no_decay(_name))
        _groups.setdefault(_key, []).append(_param)

    _param_groups = []
    for (_stage, _no_decay), _params in _groups.items():
        _lr_val = _stage_lr[_stage]
        _wd_val = 0.0 if _no_decay else weight_decay
        _param_groups.append({
            'params'       : _params,
            'lr'           : _lr_val,
            'weight_decay' : _wd_val,
            'name'         : f'{_stage}|{"no_decay" if _no_decay else "decay"}',
        })

    return _param_groups


# ============================================================
# E. OPTIMIZER + SCHEDULER FACTORY
# ============================================================
def create_optimizer_scheduler(model):
    """
    Create fresh AdamW + SequentialLR + GradScaler for one training seed.

    Must be called ONCE PER SEED inside the training loop (Step 6.4 / Step 8.2)
    to guarantee independent optimizer + scheduler state per seed.

    Training schedule:
      Epochs 0..DL_WARMUP_EPOCHS-1  : LinearLR  (DL_LR_WARMUP_INIT -> DL_LR)
      Epochs DL_WARMUP_EPOCHS..end  : CosineAnnealingLR (DL_LR -> DL_LR_MIN)

    LLRD: head and Stage4 train at DL_LR; Stage3 at DL_LR*0.85;
          Stage2 at DL_LR*0.85^2. All per-group LRs scale together
          through warmup/cosine because schedulers multiply the INITIAL lr
          stored in each param group.

    Returns:
        optimizer  : AdamW with LLRD + selective WD param groups
        scheduler  : SequentialLR (LinearLR -> CosineAnnealingLR)
        scaler     : GradScaler (AMP); scale=1.0 fp32 no-op if DL_USE_AMP=False
    """
    _param_groups = _build_param_groups(
        model, DL_LR, DL_WEIGHT_DECAY, DL_LLRD_DECAY
    )

    _optimizer = AdamW(
        _param_groups,
        betas = DL_BETAS,
        eps   = DL_EPS,
    )

    # Warmup: each group's lr grows from lr*0.01 -> lr over DL_WARMUP_EPOCHS
    _start_factor = DL_LR_WARMUP_INIT / DL_LR  # ~0.01, applied proportionally
    _warmup_sched = LinearLR(
        _optimizer,
        start_factor = _start_factor,
        end_factor   = 1.0,
        total_iters  = DL_WARMUP_EPOCHS,
    )

    # Cosine: decays each group's lr to its eta_min proportionally
    _cosine_sched = CosineAnnealingLR(
        _optimizer,
        T_max   = DL_COSINE_EPOCHS,
        eta_min = DL_LR_MIN,
    )

    _scheduler = SequentialLR(
        _optimizer,
        schedulers = [_warmup_sched, _cosine_sched],
        milestones = [DL_WARMUP_EPOCHS],
    )

    # GradScaler: enabled on CUDA, disabled (scale=1.0 no-op) on CPU
    _scaler = torch.amp.GradScaler('cuda', enabled=DL_USE_AMP)

    return _optimizer, _scheduler, _scaler


# ============================================================
# F. FUNCTION VERIFICATION ON DL_MODEL (TEMPLATE RUN)
# ============================================================
_opt_v, _sch_v, _scaler_v = create_optimizer_scheduler(DL_MODEL)

print('\n  create_optimizer_scheduler() verification:')
print(f'    Param groups         : {len(_opt_v.param_groups)}')
print(f'    {"Group":<30} {"LR":>10} {"WD":>6} {"N params":>10}')
print(f'    {"-"*60}')
_total_opt_params = 0
for _g in _opt_v.param_groups:
    _n_p = sum(p.numel() for p in _g['params'])
    _total_opt_params += _n_p
    print(f'    {_g["name"]:<30} {_g["lr"]:>10.2e} {_g["weight_decay"]:>6.4f} {_n_p:>10,}')
print(f'    {"TOTAL":<30} {"": >10} {"": >6} {_total_opt_params:>10,}')

# LR schedule preview
print('\n  LR schedule preview (head group = base LR):')
print(f'    {"Epoch":<8} {"LR (head)":>14} {"Phase"}')
print(f'    {"-"*40}')
_preview_epochs = [0, 1, 2, 3, 10, 20, 30, 39]
for _ep in _preview_epochs:
    if _ep < DL_WARMUP_EPOCHS:
        _frac   = (_ep + 1) / DL_WARMUP_EPOCHS
        _lr_val = DL_LR_WARMUP_INIT + _frac * (DL_LR - DL_LR_WARMUP_INIT)
        _phase  = 'warmup'
    else:
        _t      = _ep - DL_WARMUP_EPOCHS
        _lr_val = DL_LR_MIN + 0.5*(DL_LR - DL_LR_MIN)*(1 + math.cos(math.pi*_t/DL_COSINE_EPOCHS))
        _phase  = 'cosine'
    print(f'    {_ep:<8} {_lr_val:>14.3e}  {_phase}')

print(f'\n  Mixed precision (AMP): enabled={_scaler_v.is_enabled()}')
print(f'  GradScaler initial scale: {_scaler_v.get_scale()}')

del _opt_v, _sch_v, _scaler_v


# ============================================================
# G. PERSIST TO METADATA
# ============================================================
METADATA['dl_training_config'] = {
    'optimizer'           : DL_OPTIMIZER_NAME,
    'lr'                  : DL_LR,
    'lr_min'              : DL_LR_MIN,
    'lr_warmup_init'      : DL_LR_WARMUP_INIT,
    'llrd_decay'          : DL_LLRD_DECAY,
    'weight_decay'        : DL_WEIGHT_DECAY,
    'selective_wd'        : True,
    'betas'               : list(DL_BETAS),
    'warmup_epochs'       : DL_WARMUP_EPOCHS,
    'max_epochs'          : DL_MAX_EPOCHS,
    'cosine_epochs'       : DL_COSINE_EPOCHS,
    'early_stop_patience' : DL_EARLY_STOP_PATIENCE,
    'early_stop_monitor'  : 'val_macro_f1',
    'label_smoothing'     : DL_LABEL_SMOOTHING,
    'batch_size'          : DL_BATCH_SIZE,
    'loss_fn'             : 'CrossEntropyLoss',
    'class_weights'       : {CLASS_NAMES[c]: round(_cw_list[c], 4) for c in range(_n_classes)},
    'scheduler'           : DL_SCHEDULER_NAME,
    'mixed_precision'     : DL_USE_AMP,
    'seeds'               : EXPERIMENT_SEEDS,
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)

print(f'\n  Metadata saved -> {metadata_path}')
print(f'\n[OK] Step 6.2 complete -- training configuration locked.')


### Step 6.3 — Training Augmentation Pipeline

Defines `DL_TRAIN_TRANSFORM` (standard generalisation augmentations) and `DL_VAL_TRANSFORM` (resize + normalise only), the `StrawberryPatchDataset` class, and DataLoader configuration constants. Visual verification with same-patch ×8 augmentation diversity grid + quantitative LED contamination checks.

> **Critical constraint**: These are generalisation augmentations only — completely separate from the LED simulation (Phase 4). Hue kept at 0.05 because hue is the primary ripe/unripe discriminator.

| Transform | Parameters | Rationale |
|---|---|---|
| RandomHorizontalFlip | p=0.5 | Berries have no lateral asymmetry |
| RandomVerticalFlip | p=0.5 | Robot camera orientation varies |
| RandomRotation | ±30°, fill=0 | Covers natural viewpoint angular variation |
| Pad + RandomCrop | pad=16 (reflect), crop=224 | ±16px translation; superior to RandomResizedCrop for small patches |
| ColorJitter | b=0.2, c=0.2, **s=0.15**, hue=0.05 | saturation=0.15 (not 0.1) captures overcast vs direct-sun variation; hue=0.05 protects class signal |
| Normalize | ImageNet mean/std | Required by pretrained ConvNeXt weights |


In [ ]:
# ============================================================
# STEP 6.3 -- TRAINING AUGMENTATION PIPELINE
# ============================================================
import torchvision.transforms as T

print_section('Step 6.3 -- Training Augmentation Pipeline')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# -- Training transform ---------------------------------------------------
# Standard generalisation augmentations ONLY.
# NO large hue shift / brightness drop mimicking LED -- that is Phase 4 / E5.
# Order: geometric (PIL) -> colour (PIL) -> ToTensor -> Normalize
DL_TRAIN_TRANSFORM = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(
        degrees=30,
        interpolation=T.InterpolationMode.BILINEAR,
        fill=0,          # black fill; 10% patch padding (Phase 2) keeps berry interior
    ),
    T.Pad(padding=16, padding_mode='reflect'),   # 224 -> 256, reflect avoids seam artifacts
    T.RandomCrop(PATCH_SIZE),                    # 256 -> 224; +-16px translation invariance
                                                 # Superior to RandomResizedCrop: no risk of
                                                 # cropping entire berry on small patches
    T.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.15,  # 0.15 (not 0.1): captures overcast vs direct-sun variation
                          # without touching hue (the ripe/unripe discriminator)
        hue=0.05,         # deliberately near-zero: hue IS the ripe/unripe signal
    ),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# -- Val / Test transform -------------------------------------------------
# No augmentation at inference -- pure evaluation of the learned representation.
DL_VAL_TRANSFORM = T.Compose([
    T.Resize((PATCH_SIZE, PATCH_SIZE)),   # ensure correct size
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('\n  DL_TRAIN_TRANSFORM stages:')
for _t in DL_TRAIN_TRANSFORM.transforms:
    print(f'    {type(_t).__name__}')
print('\n  DL_VAL_TRANSFORM stages:')
for _t in DL_VAL_TRANSFORM.transforms:
    print(f'    {type(_t).__name__}')


# -- DataLoader configuration constants -----------------------------------
# Used identically in Steps 6.4 (E1) and 8.2 (E5).
DL_NUM_WORKERS      = 2                          # matches Colab 2 vCPU
DL_PIN_MEMORY       = (DEVICE.type == 'cuda')    # async host->device transfer
DL_DROP_LAST        = True                       # avoids incomplete final batch gradient noise
DL_PERSISTENT_WORKERS = True                     # keeps workers alive across epochs (~60s saving)

print(f'\n  DataLoader constants:')
print(f'    num_workers          : {DL_NUM_WORKERS}')
print(f'    pin_memory           : {DL_PIN_MEMORY}')
print(f'    drop_last            : {DL_DROP_LAST}')
print(f'    persistent_workers   : {DL_PERSISTENT_WORKERS}')


# -- StrawberryPatchDataset -----------------------------------------------
class StrawberryPatchDataset(torch.utils.data.Dataset):
    """
    PyTorch Dataset for strawberry ripeness patch classification.

    Loads (PIL image, label) pairs from class-separated directories:
      ripe_dir   -> label 0 (ripe)
      unripe_dir -> label 1 (unripe)

    Lazy loading: images opened at __getitem__ time (not pre-loaded).
    Deterministic sample order via sorted() glob for reproducibility.

    led_transform: optional stochastic LED simulation callable.
      None              -> E1 / val / test datasets (no LED augmentation)
      LEDNightSimulation -> E5 domain-adapted training (Step 8.1)
      NOTE: call signature verified in Step 8.1 against Phase 4 implementation.
    """

    def __init__(self, ripe_dir, unripe_dir, transform=None, led_transform=None):
        self.transform     = transform
        self.led_transform = led_transform
        self.samples = []

        for _p in sorted(Path(ripe_dir).glob('*.jpg')):
            self.samples.append((str(_p), 0))   # 0 = ripe
        for _p in sorted(Path(unripe_dir).glob('*.jpg')):
            self.samples.append((str(_p), 1))   # 1 = unripe

        if not self.samples:
            raise RuntimeError(
                f'No .jpg patches found in {ripe_dir} or {unripe_dir}'
            )

    def __len__(self):
        return len(self.samples)

    def __repr__(self):
        _cc = self.class_counts()
        return (f'StrawberryPatchDataset(n={len(self)}, '
                f'ripe={_cc.get(0,0)}, unripe={_cc.get(1,0)}, '
                f'led={self.led_transform is not None})')

    def __getitem__(self, idx):
        _path, _label = self.samples[idx]
        _img = Image.open(_path).convert('RGB')

        # Optional stochastic LED simulation (E5 training only)
        # NOTE: call signature (.apply vs __call__) verified in Step 8.1
        if self.led_transform is not None:
            _img_np = np.array(_img, dtype=np.uint8)
            _img_np = self.led_transform.apply(_img_np)
            _img    = Image.fromarray(_img_np)

        if self.transform is not None:
            _img = self.transform(_img)

        return _img, _label

    def class_counts(self):
        from collections import Counter
        return dict(Counter(lbl for _, lbl in self.samples))


# -- Dataset instantiation and size verification --------------------------
_ds_train      = StrawberryPatchDataset(DIRS['train_ripe'],      DIRS['train_unripe'],      transform=DL_TRAIN_TRANSFORM)
_ds_val        = StrawberryPatchDataset(DIRS['val_ripe'],        DIRS['val_unripe'],        transform=DL_VAL_TRANSFORM)
_ds_test_day   = StrawberryPatchDataset(DIRS['test_day_ripe'],   DIRS['test_day_unripe'],   transform=DL_VAL_TRANSFORM)
_ds_test_night = StrawberryPatchDataset(DIRS['test_night_ripe'], DIRS['test_night_unripe'], transform=DL_VAL_TRANSFORM)

print('\n  Dataset sizes:')
for _dname, _ds in [('train', _ds_train), ('val', _ds_val),
                     ('test_day', _ds_test_day), ('test_night', _ds_test_night)]:
    _cc = _ds.class_counts()
    print(f'    {_dname:<14}: {len(_ds):>5} patches  '
          f'(ripe={_cc.get(0,0)}, unripe={_cc.get(1,0)})')

_steps_per_epoch = len(_ds_train) // DL_BATCH_SIZE
print(f'\n  Steps per epoch (bs={DL_BATCH_SIZE}): {_steps_per_epoch}')
print(f'  Estimated time / seed (T4, ~0.10s/step): '
      f'~{DL_MAX_EPOCHS * _steps_per_epoch * 0.10 / 60:.1f} min')


# -- Training set channel statistics (for report reference) ---------------
# Sample 200 patches to estimate RGB channel means. Used in report S11.2
# to note that strawberry imagery differs from ImageNet distribution.
_sample_paths = [p for p, _ in _ds_train.samples[:200]]
_ch_means = np.array([
    np.array(Image.open(_p).convert('RGB'), dtype=np.float32).mean(axis=(0, 1))
    for _p in _sample_paths
]).mean(axis=0) / 255.0
print(f'\n  Training set channel means (RGB, 0-1 scale, n=200 patches):')
print(f'    Measured  : R={_ch_means[0]:.3f}  G={_ch_means[1]:.3f}  B={_ch_means[2]:.3f}')
print(f'    ImageNet  : R=0.485  G=0.456  B=0.406')
print(f'    Note: higher R reflects dominant ripe-berry red content in training set.')


# -- Visual verification: same patch x8 augmentations (top row)
#                         vs same patch clean val transform (bottom row) -----
def _denorm(tensor):
    _m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    _s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * _s + _m).clamp(0, 1).permute(1, 2, 0).numpy()

set_seed(GLOBAL_SEED)
_fig, _axes = plt.subplots(4, 8, figsize=(18, 9))
_fig.suptitle(
    'Step 6.3 -- Augmentation Verification\n'
    'Rows 1-2: Same ripe/unripe patch augmented 8 times (shows diversity)\n'
    'Rows 3-4: Same patches through val transform (no augmentation)',
    fontsize=10, fontweight='bold'
)

# Find first ripe and first unripe patch
_ripe_idx   = next(i for i, (_, l) in enumerate(_ds_train.samples) if l == 0)
_unripe_idx = next(i for i, (_, l) in enumerate(_ds_train.samples) if l == 1)
_ref_paths  = [_ds_train.samples[_ripe_idx][0], _ds_train.samples[_unripe_idx][0]]
_row_labels = ['ripe (train)', 'unripe (train)', 'ripe (val)', 'unripe (val)']

for _row, (_path, _transform) in enumerate([
    (_ref_paths[0], DL_TRAIN_TRANSFORM),
    (_ref_paths[1], DL_TRAIN_TRANSFORM),
    (_ref_paths[0], DL_VAL_TRANSFORM),
    (_ref_paths[1], DL_VAL_TRANSFORM),
]):
    _pil = Image.open(_path).convert('RGB')
    for _col in range(8):
        _t = _transform(_pil)
        _axes[_row, _col].imshow(_denorm(_t))
        if _col == 0:
            _axes[_row, _col].set_ylabel(_row_labels[_row], fontsize=8)
        _axes[_row, _col].axis('off')

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'augmentation_verification.png', dpi=300)
plt.show()
print('\n  Figure saved -> augmentation_verification.png  (dpi=300)')


# -- Hue and LAB b* contamination check -----------------------------------
# Confirms colour jitter does NOT leak into LED territory:
#   - Hue shift << 15 units (LED produces ~30 units shift)
#   - LAB b* shift << 10 units (LED produces b*+20 shift fixed in Phase 4)
_orig_rgb   = np.array(Image.open(_ref_paths[0]).convert('RGB'), dtype=np.uint8)
_orig_lab   = cv2.cvtColor(cv2.cvtColor(_orig_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2LAB)
_orig_hue   = float(cv2.cvtColor(cv2.cvtColor(_orig_rgb, cv2.COLOR_RGB2BGR),
                                  cv2.COLOR_BGR2HSV)[:, :, 0].mean())
_orig_b_star = float(_orig_lab[:, :, 2].mean())   # OpenCV LAB b* channel

_pil_ref  = Image.open(_ref_paths[0]).convert('RGB')
_hue_deltas, _b_deltas = [], []
for _ in range(30):
    _aug_rgb = (_denorm(DL_TRAIN_TRANSFORM(_pil_ref)) * 255).clip(0, 255).astype(np.uint8)
    _aug_hsv = cv2.cvtColor(cv2.cvtColor(_aug_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    _aug_lab = cv2.cvtColor(cv2.cvtColor(_aug_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2LAB)
    _hue_deltas.append(abs(float(_aug_hsv[:, :, 0].mean()) - _orig_hue))
    _b_deltas.append(abs(float(_aug_lab[:, :, 2].mean()) - _orig_b_star))

print(f'\n  LED contamination check (30 augmented versions of one ripe patch):')
print(f'    Hue shift   -- mean={sum(_hue_deltas)/len(_hue_deltas):.2f}, '
      f'max={max(_hue_deltas):.2f}  [LED threshold: ~30 units]')
print(f'    LAB b* shift -- mean={sum(_b_deltas)/len(_b_deltas):.2f}, '
      f'max={max(_b_deltas):.2f}  [LED threshold: ~20 units]')
print(f'    [PASS if both max values << LED thresholds]')


# -- Persist to metadata --------------------------------------------------
METADATA['dl_augmentation'] = {
    'train_transforms': [
        'RandomHorizontalFlip(p=0.5)',
        'RandomVerticalFlip(p=0.5)',
        'RandomRotation(30, fill=0)',
        'Pad(16, reflect) + RandomCrop(224)',
        'ColorJitter(b=0.2, c=0.2, s=0.15, hue=0.05)',
        'ToTensor', 'Normalize(ImageNet)',
    ],
    'val_transforms'  : ['Resize(224)', 'ToTensor', 'Normalize(ImageNet)'],
    'imagenet_mean'   : IMAGENET_MEAN,
    'imagenet_std'    : IMAGENET_STD,
    'num_workers'     : DL_NUM_WORKERS,
    'pin_memory'      : DL_PIN_MEMORY,
    'drop_last'       : DL_DROP_LAST,
    'persistent_workers': DL_PERSISTENT_WORKERS,
    'train_ch_means_rgb': _ch_means.round(4).tolist(),
    'dataset_sizes'   : {
        'train'      : len(_ds_train),
        'val'        : len(_ds_val),
        'test_day'   : len(_ds_test_day),
        'test_night' : len(_ds_test_night),
    },
    'steps_per_epoch' : _steps_per_epoch,
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)

print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Step 6.3 complete -- transforms, Dataset, and DataLoader config ready.')


### Step 6.4 — Training Loop (E1: DL-Day Baseline)

Trains ConvNeXt-Tiny across 3 independent seeds on day patches.  
Deliverables: best checkpoint per seed, training-curve CSVs, `E1_TRAIN_RESULTS` dict.

**Key engineering decisions:**
- `scaler.unscale_(optimizer)` **before** `clip_grad_norm_()` — AMP correctness requirement
- `zero_grad(set_to_none=True)` — faster CUDA gradient reset
- Seeded `torch.Generator` in DataLoader — deterministic shuffle across re-runs
- `torch.no_grad()` (not `inference_mode`) — Grad-CAM library compatibility (Phase 9)
- Idempotency guard on `E1_train_results.json` — 'Run All' safe


In [ ]:
# ============================================================
# STEP 6.4 -- TRAINING LOOP (E1: DL-DAY BASELINE)
# ============================================================
import csv, time

print_section('Step 6.4 -- Training Loop (E1: DL-Day Baseline)')

# ── Idempotency guard ─────────────────────────────────────────────────────
# Gradient clip norm defined at module scope so it is always available
# (even when E1 idempotency branch is taken and the else block is skipped)
DL_GRAD_CLIP_NORM = 1.0   # max gradient norm (Liu et al. 2022; He et al. 2022)

_e1_results_path = DIRS['metrics'] / 'E1_train_results.json'
if _e1_results_path.exists():
    print('  Already complete -- loading cached E1 training results.')
    with open(_e1_results_path) as _f:
        E1_TRAIN_RESULTS = json.load(_f)
    E1_CHECKPOINTS = {
        s: DIRS['checkpoints'] / f'E1_seed{s}_best.pt'
        for s in EXPERIMENT_SEEDS
    }
    # Verify checkpoint files exist before proceeding to Step 6.5
    _missing = [str(p) for p in E1_CHECKPOINTS.values() if not p.exists()]
    if _missing:
        raise FileNotFoundError(
            f'E1 checkpoints missing (delete E1_train_results.json to re-train):\n'
            + '\n'.join(_missing)
        )
    print(f'  Mean val macro-F1 : {E1_TRAIN_RESULTS["mean_val_macro_f1"]:.4f}')
    print(f'  Std  val macro-F1 : {E1_TRAIN_RESULTS["std_val_macro_f1"]:.4f}')
else:

    # ── Training constants ─────────────────────────────────────────────────
    # ── Per-seed result containers ─────────────────────────────────────────
    _seed_logs        = {}   # {seed: list of per-epoch dicts}
    _seed_best_f1     = {}   # {seed: float}
    E1_CHECKPOINTS    = {}   # {seed: Path}
    _t_start_total    = time.time()

    for _seed in EXPERIMENT_SEEDS:
        print(f'\n  {"-"*52}')
        print(f'  Seed {_seed} ({EXPERIMENT_SEEDS.index(_seed)+1} / {len(EXPERIMENT_SEEDS)})')
        print(f'  {"-"*52}')

        # -- Fresh model, fresh optimizer/scheduler/scaler per seed
        set_seed(_seed)
        _model = build_convnext_tiny(DL_NUM_CLASSES, pretrained=True)
        _model = freeze_convnext_backbone(_model)
        _model = _model.to(DEVICE)

        _optimizer, _scheduler, _scaler = create_optimizer_scheduler(_model)

        # -- Seeded DataLoaders (deterministic shuffle across re-runs)
        _gen_train = torch.Generator()
        _gen_train.manual_seed(_seed)

        _loader_train = torch.utils.data.DataLoader(
            _ds_train,
            batch_size        = DL_BATCH_SIZE,
            shuffle           = True,
            num_workers       = DL_NUM_WORKERS,
            pin_memory        = DL_PIN_MEMORY,
            drop_last         = DL_DROP_LAST,
            persistent_workers= DL_PERSISTENT_WORKERS,
            generator         = _gen_train,
        )
        _loader_val = torch.utils.data.DataLoader(
            _ds_val,
            batch_size        = DL_BATCH_SIZE * 2,   # 2x bs safe at inference
            shuffle           = False,
            num_workers       = DL_NUM_WORKERS,
            pin_memory        = DL_PIN_MEMORY,
            persistent_workers= DL_PERSISTENT_WORKERS,
        )

        # -- CSV log
        _log_path = DIRS['logs'] / f'E1_seed{_seed}_training_log.csv'
        _ckpt_path = DIRS['checkpoints'] / f'E1_seed{_seed}_best.pt'

        _log_rows     = []
        _best_val_f1  = 0.0
        _patience_ctr = 0
        _t_seed_start = time.time()

        for _epoch in range(DL_MAX_EPOCHS):

            # ── TRAIN EPOCH ───────────────────────────────────────────────
            _model.train()
            _train_loss_sum = 0.0
            _n_train_steps  = 0

            for _imgs, _labels in _loader_train:
                _imgs   = _imgs.to(DEVICE, non_blocking=True)
                _labels = _labels.to(DEVICE, non_blocking=True)

                _optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()

                with torch.amp.autocast(device_type=DEVICE.type, enabled=DL_USE_AMP):
                    _logits = _model(_imgs)
                    _loss   = DL_LOSS_FN(_logits, _labels)

                _scaler.scale(_loss).backward()

                # CRITICAL: unscale before clipping (clips true gradients, not scaled)
                _scaler.unscale_(_optimizer)
                torch.nn.utils.clip_grad_norm_(
                    _model.parameters(), max_norm=DL_GRAD_CLIP_NORM
                )

                _scaler.step(_optimizer)
                _scaler.update()

                _train_loss_sum += _loss.item()
                _n_train_steps  += 1

            _train_loss_avg = _train_loss_sum / max(_n_train_steps, 1)

            # ── VALIDATION EPOCH ──────────────────────────────────────────
            _model.eval()
            _val_loss_sum = 0.0
            _val_all_preds, _val_all_labels = [], []

            with torch.no_grad():   # no_grad (not inference_mode) for Grad-CAM compat
                for _imgs, _labels in _loader_val:
                    _imgs   = _imgs.to(DEVICE, non_blocking=True)
                    _labels = _labels.to(DEVICE, non_blocking=True)

                    with torch.amp.autocast(device_type=DEVICE.type, enabled=DL_USE_AMP):
                        _logits = _model(_imgs)
                        _vloss  = DL_LOSS_FN(_logits, _labels)

                    _val_loss_sum += _vloss.item()
                    _val_all_preds.extend(_logits.argmax(dim=1).cpu().tolist())
                    _val_all_labels.extend(_labels.cpu().tolist())

            _val_loss_avg = _val_loss_sum / len(_loader_val)
            _val_f1       = f1_score(_val_all_labels, _val_all_preds,
                                     average='macro', zero_division=1)
            _val_acc      = sum(p == l for p, l in
                                zip(_val_all_preds, _val_all_labels)) / len(_val_all_labels)

            # ── SCHEDULER STEP (epoch-level, after val) ───────────────────
            _scheduler.step()
            # Find classifier group LR by name (robust to group insertion order)
            _clf_groups = [g for g in _optimizer.param_groups
                           if g.get('name', '').startswith('classifier')]
            _current_lr = _clf_groups[-1]['lr'] if _clf_groups else _optimizer.param_groups[-1]['lr']

            # ── LOG ROW ───────────────────────────────────────────────────
            _row = {
                'epoch'         : _epoch,
                'train_loss'    : round(_train_loss_avg, 5),
                'val_loss'      : round(_val_loss_avg, 5),
                'val_macro_f1'  : round(_val_f1, 5),
                'val_accuracy'  : round(_val_acc, 5),
                'lr'            : f'{_current_lr:.2e}',
            }
            _log_rows.append(_row)

            _ep_str = f'  Ep {_epoch:02d}/{DL_MAX_EPOCHS-1}'
            _f1_str = f'valF1={_val_f1:.4f}'
            _star   = ' *' if _val_f1 > _best_val_f1 else ''
            print(f'{_ep_str}  trLoss={_train_loss_avg:.4f}  '
                  f'vlLoss={_val_loss_avg:.4f}  {_f1_str}  '
                  f'acc={_val_acc:.3f}  lr={_current_lr:.1e}{_star}')

            # ── EARLY STOPPING + CHECKPOINT ───────────────────────────────
            if _val_f1 > _best_val_f1:
                _best_val_f1  = _val_f1
                _patience_ctr = 0
                torch.save({
                    'model_state'     : _model.state_dict(),
                    'optimizer_state' : _optimizer.state_dict(),
                    'epoch'           : _epoch,
                    'val_macro_f1'    : _val_f1,
                    'seed'            : _seed,
                    'architecture'    : DL_MODEL_NAME,
                }, _ckpt_path)
            else:
                _patience_ctr += 1
                if _patience_ctr >= DL_EARLY_STOP_PATIENCE:
                    print(f'  Early stop at epoch {_epoch} '
                          f'(patience={DL_EARLY_STOP_PATIENCE} exceeded)')
                    break

        # ── SAVE CSV LOG ──────────────────────────────────────────────────────
        with open(_log_path, 'w', newline='') as _csvf:
            _writer = csv.DictWriter(_csvf, fieldnames=_log_rows[0].keys())
            _writer.writeheader()
            _writer.writerows(_log_rows)

        _t_seed_elapsed = (time.time() - _t_seed_start) / 60
        print(f'\n  Seed {_seed} done in {_t_seed_elapsed:.1f} min  '
              f'| best val macro-F1 = {_best_val_f1:.4f}')
        print(f'  Checkpoint -> {_ckpt_path}')
        print(f'  CSV log    -> {_log_path}')

        _seed_logs[str(_seed)]    = _log_rows
        _seed_best_f1[str(_seed)] = _best_val_f1
        E1_CHECKPOINTS[_seed]     = _ckpt_path

        # Free GPU memory between seeds
        del _model, _optimizer, _scheduler, _scaler
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    # ── AGGREGATE STATS ───────────────────────────────────────────────────────
    _total_elapsed = (time.time() - _t_start_total) / 60
    _f1_vals = list(_seed_best_f1.values())
    _mean_f1 = float(np.mean(_f1_vals))
    _std_f1  = float(np.std(_f1_vals))

    print(f'\n  {"="*52}')
    print(f'  E1 Training complete  ({_total_elapsed:.1f} min total)')
    print(f'  Val macro-F1 per seed:')
    for _s, _f in _seed_best_f1.items():
        print(f'    seed {_s}: {_f:.4f}')
    print(f'  Mean ± std: {_mean_f1:.4f} ± {_std_f1:.4f}')
    print(f'  {"="*52}')

    # ── SAVE E1_TRAIN_RESULTS ─────────────────────────────────────────────────
    E1_TRAIN_RESULTS = {
        'experiment'         : 'E1-Day-Baseline',
        'model'              : DL_MODEL_NAME,
        'seeds'              : EXPERIMENT_SEEDS,
        'mean_val_macro_f1'  : _mean_f1,
        'std_val_macro_f1'   : _std_f1,
        'per_seed_best_val_f1': _seed_best_f1,
        'total_train_min'    : round(_total_elapsed, 2),
        'per_seed_logs'      : _seed_logs,
        'checkpoints'        : {str(s): str(p) for s, p in E1_CHECKPOINTS.items()},
    }
    with open(_e1_results_path, 'w') as _f:
        json.dump(E1_TRAIN_RESULTS, _f, indent=2)
    print(f'\n  E1 results saved -> {_e1_results_path}')

    # ── TRAINING CURVES PLOT ──────────────────────────────────────────────────
    _seed_colours = ['#2196F3', '#FF9800', '#4CAF50']   # blue, orange, green
    _fig, (_ax_loss, _ax_f1) = plt.subplots(1, 2, figsize=(14, 5))
    _fig.suptitle('Step 6.4 -- E1 Training Curves (3 seeds)', fontsize=13, fontweight='bold')

    for (_s, _logs), _col in zip(_seed_logs.items(), _seed_colours):
        _epochs     = [r['epoch']        for r in _logs]
        _tr_losses  = [r['train_loss']   for r in _logs]
        _vl_losses  = [r['val_loss']     for r in _logs]
        _vl_f1s     = [r['val_macro_f1'] for r in _logs]

        _ax_loss.plot(_epochs, _tr_losses, color=_col, linestyle='--',
                      alpha=0.7, label=f'train s{_s}')
        _ax_loss.plot(_epochs, _vl_losses, color=_col, linestyle='-',
                      alpha=0.9, label=f'val   s{_s}')
        _ax_f1.plot(_epochs, _vl_f1s, color=_col, linestyle='-',
                    label=f'seed {_s}')

    _ax_loss.set_xlabel('Epoch')
    _ax_loss.set_ylabel('CE Loss')
    _ax_loss.set_title('Train vs Val Loss')
    _ax_loss.legend(fontsize=8)
    _ax_loss.grid(alpha=0.3)

    _ax_f1.axhline(_mean_f1, color='black', linestyle=':', linewidth=1.5,
                   label=f'mean F1={_mean_f1:.3f}')
    _ax_f1.set_xlabel('Epoch')
    _ax_f1.set_ylabel('Val Macro-F1')
    _ax_f1.set_title('Val Macro-F1 (monitor for early stopping)')
    _ax_f1.legend(fontsize=8)
    _ax_f1.grid(alpha=0.3)

    _fig.tight_layout()
    _fig.savefig(DIRS['figures'] / 'E1_training_curves.png', dpi=300)
    plt.show()
    print('  Figure saved -> E1_training_curves.png')

# ── UPDATE METADATA ───────────────────────────────────────────────────────────
METADATA['E1_training'] = {
    'mean_val_macro_f1' : E1_TRAIN_RESULTS['mean_val_macro_f1'],
    'std_val_macro_f1'  : E1_TRAIN_RESULTS['std_val_macro_f1'],
    'seeds'             : EXPERIMENT_SEEDS,
    'checkpoints'       : {str(s): str(p) for s, p in E1_CHECKPOINTS.items()},
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)

print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Step 6.4 complete -- E1 training done, checkpoints saved.')


### Step 6.5 — E1 Evaluation on Day Test Set

Locks the **DL performance ceiling** by evaluating all 3 seed checkpoints on the held-out day test set.  
Test set is used **exactly once** here — never re-evaluated.

**Evaluation protocol:**
- All 3 seed checkpoints loaded and evaluated independently
- Predictions pooled → aggregate metrics + per-seed macro-F1 (mean ± std)
- `logits.float().softmax()` — FP32 cast before softmax (AMP correctness)
- Bootstrap CI: 1000 percentile samples on pooled macro-F1
- Metrics: macro-F1 + 95% CI, PR-AUC, ROC-AUC, MCC, accuracy, per-class breakdown


In [ ]:
# ============================================================
# STEP 6.5 -- E1 EVALUATION ON DAY TEST SET
# ============================================================
from sklearn.metrics import (
    average_precision_score, roc_auc_score, matthews_corrcoef,
    classification_report, roc_curve, auc as sklearn_auc
)

print_section('Step 6.5 -- E1 Evaluation on Day Test Set')

# ── Idempotency guard ─────────────────────────────────────────────────────
_e1_metrics_path = DIRS['metrics'] / 'E1_day_metrics.json'
if _e1_metrics_path.exists():
    print('  Already complete -- loading cached E1 day metrics.')
    with open(_e1_metrics_path) as _f:
        E1_DAY_METRICS = json.load(_f)
    print(f'  Macro-F1 : {E1_DAY_METRICS["macro_f1"]:.4f}  '
          f'[{E1_DAY_METRICS["macro_f1_ci_95"][0]:.4f}, '
          f'{E1_DAY_METRICS["macro_f1_ci_95"][1]:.4f}]')
    print(f'  ROC-AUC  : {E1_DAY_METRICS["auc_roc"]:.4f}')
    print(f'  MCC      : {E1_DAY_METRICS["mcc"]:.4f}')
else:

    # ── Test DataLoader (no shuffle, drop_last=False, 2x batch size) ──────
    _loader_test_day = torch.utils.data.DataLoader(
        _ds_test_day,
        batch_size        = DL_BATCH_SIZE * 2,
        shuffle           = False,
        num_workers       = DL_NUM_WORKERS,
        pin_memory        = DL_PIN_MEMORY,
        drop_last         = False,      # must evaluate ALL test samples
        persistent_workers= DL_PERSISTENT_WORKERS,
    )

    # ── 3-seed evaluation loop ────────────────────────────────────────────
    _all_preds  = []   # pooled across seeds
    _all_labels = []   # ground truth (same for each seed, appended 3x for pooling)
    _all_probas = []   # softmax proba, pooled
    _seed_f1s   = {}   # {seed: macro-F1} for mean/std

    for _seed in EXPERIMENT_SEEDS:
        _ckpt_path = DIRS['checkpoints'] / f'E1_seed{_seed}_best.pt'
        print(f'\n  Evaluating seed {_seed}: {_ckpt_path.name}')

        # Fresh model + load checkpoint
        set_seed(_seed)
        _model = build_convnext_tiny(DL_NUM_CLASSES, pretrained=False)
        _model = freeze_convnext_backbone(_model)
        _ckpt  = torch.load(_ckpt_path, map_location=DEVICE, weights_only=False)
        _model.load_state_dict(_ckpt['model_state'])
        _model = _model.to(DEVICE)
        _model.eval()

        print(f'    Loaded checkpoint: epoch={_ckpt["epoch"]}  '
              f'val_macro_f1={_ckpt["val_macro_f1"]:.4f}')

        _seed_preds, _seed_labels, _seed_probas = [], [], []

        with torch.no_grad():
            for _imgs, _labels in _loader_test_day:
                _imgs   = _imgs.to(DEVICE, non_blocking=True)
                _labels = _labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=DL_USE_AMP):
                    _logits = _model(_imgs)

                # FP32 cast before softmax -- AMP returns FP16, avoids precision loss
                _probas = _logits.float().softmax(dim=1)
                _preds  = _probas.argmax(dim=1)

                _seed_preds.extend(_preds.cpu().tolist())
                _seed_labels.extend(_labels.cpu().tolist())
                _seed_probas.extend(_probas.cpu().tolist())

        _seed_f1 = f1_score(_seed_labels, _seed_preds,
                            average='macro', zero_division=1)
        _seed_f1s[str(_seed)] = _seed_f1
        print(f'    Seed macro-F1 on test_day: {_seed_f1:.4f}')

        _all_preds.extend(_seed_preds)
        _all_labels.extend(_seed_labels)
        _all_probas.extend(_seed_probas)

        del _model
        torch.cuda.empty_cache()

    # ── Aggregate metrics on pooled predictions ───────────────────────────
    import numpy as np
    _all_labels_arr = np.array(_all_labels)
    _all_preds_arr  = np.array(_all_preds)
    _all_probas_arr = np.array(_all_probas)   # shape [N*3, 2]

    _macro_f1   = f1_score(_all_labels_arr, _all_preds_arr,
                           average='macro', zero_division=1)
    _f1_ci      = _bootstrap_macro_f1(_all_labels_arr, _all_preds_arr)
    # Pass full 2-D proba matrix for true binary macro-AP
    # (1-D y_score silently ignores average='macro' in sklearn)
    _auc_pr     = average_precision_score(
                    _all_labels_arr, _all_probas_arr, average='macro')
    _auc_roc    = roc_auc_score(
                    _all_labels_arr, _all_probas_arr[:, 1], average='macro')
    _mcc        = matthews_corrcoef(_all_labels_arr, _all_preds_arr)
    _accuracy   = float((_all_labels_arr == _all_preds_arr).mean())
    _cm         = confusion_matrix(_all_labels_arr, _all_preds_arr)
    _report     = classification_report(_all_labels_arr, _all_preds_arr,
                                        target_names=CLASS_NAMES,
                                        output_dict=True, zero_division=1)
    _mean_f1    = float(np.mean(list(_seed_f1s.values())))
    _std_f1     = float(np.std(list(_seed_f1s.values())))

    print(f'\n  {"-"*50}')
    print(f'  E1 Day Test Set Results (pooled 3-seed evaluation)')
    print(f'  {"-"*50}')
    print(f'  Macro-F1          : {_macro_f1:.4f}  '
          f'95% CI [{_f1_ci[0]:.4f}, {_f1_ci[1]:.4f}]')
    print(f'  Mean±Std (3 seeds) : {_mean_f1:.4f} ± {_std_f1:.4f}')
    print(f'  PR-AUC            : {_auc_pr:.4f}')
    print(f'  ROC-AUC           : {_auc_roc:.4f}')
    print(f'  MCC               : {_mcc:.4f}')
    print(f'  Accuracy          : {_accuracy:.4f}')
    print(f'\n  Per-class breakdown (pooled):')
    for _cls in CLASS_NAMES:
        _r = _report[_cls]
        print(f'    {_cls:<8}: P={_r["precision"]:.4f}  '
              f'R={_r["recall"]:.4f}  F1={_r["f1-score"]:.4f}  '
              f'N={int(_r["support"])}')
    print(f'\n  Confusion matrix (absolute counts, pooled):')
    print(f'    {_cm}')

    # ── Save E1_DAY_METRICS (standard schema) ─────────────────────────────
    E1_DAY_METRICS = {
        'experiment'        : 'E1-Day',
        'model'             : DL_MODEL_NAME,
        'split'             : 'test_day',
        'n_samples'         : len(_ds_test_day),
        'macro_f1'          : round(_macro_f1, 6),
        'macro_f1_ci_95'    : [round(_f1_ci[0], 6), round(_f1_ci[1], 6)],
        'auc_pr'            : round(_auc_pr, 6),
        'auc_roc'           : round(_auc_roc, 6),
        'mcc'               : round(_mcc, 6),
        'accuracy'          : round(_accuracy, 6),
        'mean_macro_f1'     : round(_mean_f1, 6),
        'std_macro_f1'      : round(_std_f1, 6),
        'training_seeds'    : EXPERIMENT_SEEDS,
        'per_seed_f1'       : {k: round(v, 6) for k, v in _seed_f1s.items()},
        'per_class'         : {
            _cls: {
                'precision' : round(_report[_cls]['precision'], 6),
                'recall'    : round(_report[_cls]['recall'], 6),
                'f1'        : round(_report[_cls]['f1-score'], 6),
                'support'   : int(_report[_cls]['support']),
            }
            for _cls in CLASS_NAMES
        },
        'confusion_matrix'  : _cm.tolist(),
    }
    with open(_e1_metrics_path, 'w') as _f:
        json.dump(E1_DAY_METRICS, _f, indent=2)
    print(f'\n  Metrics saved -> {_e1_metrics_path}')

    # ── Confusion matrix figure ───────────────────────────────────────────
    _cm_norm = _cm.astype('float') / _cm.sum(axis=1, keepdims=True)
    _fig_cm, _ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        _cm_norm, annot=False, fmt='.2f', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=_ax_cm, vmin=0, vmax=1,
    )
    # Annotate with both normalised and absolute counts
    for _i in range(len(CLASS_NAMES)):
        for _j in range(len(CLASS_NAMES)):
            _ax_cm.text(_j + 0.5, _i + 0.5,
                        f'{_cm_norm[_i,_j]:.2f}\n({_cm[_i,_j]})',
                        ha='center', va='center', fontsize=10,
                        color='white' if _cm_norm[_i,_j] > 0.6 else 'black')
    _ax_cm.set_xlabel('Predicted', fontsize=11)
    _ax_cm.set_ylabel('True', fontsize=11)
    _ax_cm.set_title(
        f'E1 (DL-Day) Confusion Matrix\n'
        f'Macro-F1={_macro_f1:.4f}  MCC={_mcc:.4f}  '
        f'(pooled 3 seeds, test_day)',
        fontsize=10
    )
    _fig_cm.tight_layout()
    _fig_cm.savefig(DIRS['figures'] / 'E1_confusion_matrix.png', dpi=300)
    plt.show()
    print('  Figure saved -> E1_confusion_matrix.png')

    # ── ROC curve figure ──────────────────────────────────────────────────
    _fpr, _tpr, _ = roc_curve(_all_labels_arr, _all_probas_arr[:, 1])
    _roc_auc_val  = sklearn_auc(_fpr, _tpr)

    _fig_roc, _ax_roc = plt.subplots(figsize=(6, 5))
    _ax_roc.plot(_fpr, _tpr, color='#2196F3', lw=2,
                 label=f'E1 ROC (AUC = {_roc_auc_val:.4f})')
    _ax_roc.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1)
    _ax_roc.set_xlabel('False Positive Rate', fontsize=11)
    _ax_roc.set_ylabel('True Positive Rate', fontsize=11)
    _ax_roc.set_title('E1 (DL-Day) ROC Curve  \u2014  test_day', fontsize=11)
    _ax_roc.legend(fontsize=10)
    _ax_roc.grid(alpha=0.3)
    _fig_roc.tight_layout()
    _fig_roc.savefig(DIRS['figures'] / 'E1_roc_curve.png', dpi=300)
    plt.show()
    print('  Figure saved -> E1_roc_curve.png')

# ── Update metadata ───────────────────────────────────────────────────────────
METADATA['E1_test_day'] = {
    'macro_f1'      : E1_DAY_METRICS['macro_f1'],
    'macro_f1_ci_95': E1_DAY_METRICS['macro_f1_ci_95'],
    'auc_roc'       : E1_DAY_METRICS['auc_roc'],
    'mcc'           : E1_DAY_METRICS['mcc'],
    'mean_macro_f1' : E1_DAY_METRICS['mean_macro_f1'],
    'std_macro_f1'  : E1_DAY_METRICS['std_macro_f1'],
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)

print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Step 6.5 complete -- E1_DAY_METRICS locked as DL performance ceiling.')


---
## Phase 7 — Experiments E3 & E4: Domain Shift Quantification

**Goal**: Demonstrate and quantify how performance degrades when the E1 model (trained on day) is evaluated on LED-simulated night patches. No retraining — E1 checkpoints are reused directly.

| Step | Action |
|---|---|
| 7.1 | Verify night test patch counts match day test exactly |
| 7.2 | E3: evaluate all 3 E1 seed checkpoints on test_night → `E3_NIGHT_METRICS` |
| 7.3 | E4: compute Δ metrics, per-class degradation, seed consistency table |


### Step 7.1 — Night Test Set Verification

Confirms that `_ds_test_night` has exactly the same patch count and class distribution as `_ds_test_day`.  
Visual side-by-side verification: same patches, day vs night appearance.


In [ ]:
# ============================================================
# STEP 7.1 -- NIGHT TEST SET VERIFICATION
# ============================================================
print_section('Step 7.1 -- Night Test Set Verification')

# ── Count check ───────────────────────────────────────────────────────────
_day_counts   = _ds_test_day.class_counts()
_night_counts = _ds_test_night.class_counts()

print('  Patch count comparison (test_day vs test_night):')
print(f'  {"Split":<14} {"Ripe":>8} {"Unripe":>8} {"Total":>8}')
print(f'  {"-"*42}')
print(f'  {"test_day":<14} {_day_counts.get(0,0):>8} {_day_counts.get(1,0):>8} {len(_ds_test_day):>8}')
print(f'  {"test_night":<14} {_night_counts.get(0,0):>8} {_night_counts.get(1,0):>8} {len(_ds_test_night):>8}')

assert len(_ds_test_day) == len(_ds_test_night), \
    f'MISMATCH: day={len(_ds_test_day)}, night={len(_ds_test_night)}'
assert _day_counts == _night_counts, \
    f'Class count mismatch: day={_day_counts}, night={_night_counts}'
print('\n  [PASS] Patch counts and class distributions match exactly.')

# ── Visual side-by-side grid: 4 ripe + 4 unripe (day | night) ───────────
def _denorm(tensor):
    _m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    _s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * _s + _m).clamp(0, 1).permute(1, 2, 0).numpy()

set_seed(GLOBAL_SEED)
# find 4 ripe indices and 4 unripe indices in test_day
_ripe_idxs   = [i for i, (_, l) in enumerate(_ds_test_day.samples) if l == 0][:4]
_unripe_idxs = [i for i, (_, l) in enumerate(_ds_test_day.samples) if l == 1][:4]
_show_idxs   = _ripe_idxs + _unripe_idxs  # 8 patches total
_show_labels = ['ripe']*4 + ['unripe']*4

_fig, _axes = plt.subplots(2, 8, figsize=(20, 5))
_fig.suptitle(
    'Step 7.1 -- Night Test Set Verification\n'
    'Top row: Day patches  |  Bottom row: Same patches after LED simulation',
    fontsize=10, fontweight='bold'
)

for _col, (_idx, _lbl) in enumerate(zip(_show_idxs, _show_labels)):
    _img_day, _  = _ds_test_day[_idx]
    _img_night, _ = _ds_test_night[_idx]

    _axes[0, _col].imshow(_denorm(_img_day))
    _axes[0, _col].set_title(_lbl, fontsize=8)
    _axes[0, _col].axis('off')

    _axes[1, _col].imshow(_denorm(_img_night))
    _axes[1, _col].set_title(f'{_lbl}\n(night)', fontsize=8)
    _axes[1, _col].axis('off')

_fig.tight_layout()
_fig.savefig(DIRS['figures'] / 'night_patch_verification.png', dpi=300)
plt.show()
print('  Figure saved -> night_patch_verification.png')
print('\n[OK] Step 7.1 complete -- night test set verified.')


### Step 7.2 — E3: E1 Model on Night Test Set

Evaluates the E1 model (trained on day only) on `test_night` — no retraining, no adaptation.  
Same 3-seed protocol as Step 6.5. Saves `E3_NIGHT_METRICS` with pre-computed `day_night_delta`.


In [ ]:
# ============================================================
# STEP 7.2 -- E3: E1 MODEL ON NIGHT TEST SET
# ============================================================
print_section('Step 7.2 -- E3: E1 Model on Night Test Set')

# ── Idempotency guard ──────────────────────────────────────────────────────
_e3_metrics_path = DIRS['metrics'] / 'E3_night_metrics.json'
if _e3_metrics_path.exists():
    print('  Already complete -- loading cached E3 night metrics.')
    with open(_e3_metrics_path) as _f:
        E3_NIGHT_METRICS = json.load(_f)
    print(f'  Macro-F1 : {E3_NIGHT_METRICS["macro_f1"]:.4f}  '
          f'[{E3_NIGHT_METRICS["macro_f1_ci_95"][0]:.4f}, '
          f'{E3_NIGHT_METRICS["macro_f1_ci_95"][1]:.4f}]')
    print(f'  ΔF1 vs E1 : {E3_NIGHT_METRICS["day_night_delta"]["delta_macro_f1"]:+.4f}')
else:

    # ── Night test DataLoader ──────────────────────────────────────────────
    _loader_test_night = torch.utils.data.DataLoader(
        _ds_test_night,
        batch_size         = DL_BATCH_SIZE * 2,
        shuffle            = False,
        num_workers        = DL_NUM_WORKERS,
        pin_memory         = DL_PIN_MEMORY,
        drop_last          = False,
        persistent_workers = DL_PERSISTENT_WORKERS,
    )

    # ── 3-seed evaluation on test_night ───────────────────────────────────
    _all_preds_n, _all_labels_n, _all_probas_n = [], [], []
    _seed_f1s_n = {}

    for _seed in EXPERIMENT_SEEDS:
        _ckpt_path = DIRS['checkpoints'] / f'E1_seed{_seed}_best.pt'
        print(f'\n  Evaluating seed {_seed} on test_night: {_ckpt_path.name}')

        set_seed(_seed)
        _model = build_convnext_tiny(DL_NUM_CLASSES, pretrained=False)
        _model = freeze_convnext_backbone(_model)
        _ckpt  = torch.load(_ckpt_path, map_location=DEVICE, weights_only=False)
        _model.load_state_dict(_ckpt['model_state'])
        _model = _model.to(DEVICE)
        _model.eval()

        _seed_preds_n, _seed_labels_n, _seed_probas_n = [], [], []

        with torch.no_grad():
            for _imgs, _labels in _loader_test_night:
                _imgs   = _imgs.to(DEVICE, non_blocking=True)
                _labels = _labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=DL_USE_AMP):
                    _logits = _model(_imgs)

                _probas = _logits.float().softmax(dim=1)   # FP32 cast
                _preds  = _probas.argmax(dim=1)

                _seed_preds_n.extend(_preds.cpu().tolist())
                _seed_labels_n.extend(_labels.cpu().tolist())
                _seed_probas_n.extend(_probas.cpu().tolist())

        _sf1 = f1_score(_seed_labels_n, _seed_preds_n, average='macro', zero_division=1)
        _seed_f1s_n[str(_seed)] = _sf1
        print(f'    Seed macro-F1 on test_night: {_sf1:.4f}')

        _all_preds_n.extend(_seed_preds_n)
        _all_labels_n.extend(_seed_labels_n)
        _all_probas_n.extend(_seed_probas_n)

        del _model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    # ── Aggregate metrics ──────────────────────────────────────────────────
    import numpy as np
    _an = np.array(_all_labels_n)
    _pn = np.array(_all_preds_n)
    _prn = np.array(_all_probas_n)

    _macro_f1_n  = f1_score(_an, _pn, average='macro', zero_division=1)
    _f1_ci_n     = _bootstrap_macro_f1(_an, _pn)
    # Pass full 2-D proba matrix for true binary macro-AP
    _auc_pr_n    = average_precision_score(_an, _prn, average='macro')
    _auc_roc_n   = roc_auc_score(_an, _prn[:, 1], average='macro')
    _mcc_n       = matthews_corrcoef(_an, _pn)
    _acc_n       = float((_an == _pn).mean())
    _cm_n        = confusion_matrix(_an, _pn)
    _report_n    = classification_report(_an, _pn, target_names=CLASS_NAMES,
                                          output_dict=True, zero_division=1)
    _mean_f1_n   = float(np.mean(list(_seed_f1s_n.values())))
    _std_f1_n    = float(np.std(list(_seed_f1s_n.values())))

    # ── Pre-compute day_night_delta (requires E1_DAY_METRICS) ─────────────
    _e1 = E1_DAY_METRICS
    _delta = {
        'delta_macro_f1'  : round(_e1['macro_f1']   - _macro_f1_n, 6),
        'delta_accuracy'  : round(_e1['accuracy']   - _acc_n, 6),
        'delta_auc_roc'   : round(_e1['auc_roc']    - _auc_roc_n, 6),
        'delta_mcc'       : round(_e1['mcc']        - _mcc_n, 6),
        'delta_ripe_f1'   : round(_e1['per_class']['ripe']['f1']   -
                                   _report_n['ripe']['f1-score'], 6),
        'delta_unripe_f1' : round(_e1['per_class']['unripe']['f1'] -
                                   _report_n['unripe']['f1-score'], 6),
        'pct_f1_degradation': round(
            (_e1['macro_f1'] - _macro_f1_n) / _e1['macro_f1'] * 100, 2),
    }

    print(f'\n  {"-"*50}')
    print(f'  E3 Night Test Set Results (pooled 3-seed)')
    print(f'  {"-"*50}')
    print(f'  Macro-F1    : {_macro_f1_n:.4f}  '
          f'[{_f1_ci_n[0]:.4f}, {_f1_ci_n[1]:.4f}]')
    print(f'  Mean±Std    : {_mean_f1_n:.4f} ± {_std_f1_n:.4f}')
    print(f'  ROC-AUC     : {_auc_roc_n:.4f}')
    print(f'  MCC         : {_mcc_n:.4f}')
    print(f'  Accuracy    : {_acc_n:.4f}')
    print(f'\n  Per-class breakdown:')
    for _cls in CLASS_NAMES:
        _r = _report_n[_cls]
        print(f'    {_cls:<8}: P={_r["precision"]:.4f}  '
              f'R={_r["recall"]:.4f}  F1={_r["f1-score"]:.4f}')
    print(f'\n  Domain shift delta vs E1:')
    print(f'    ΔMacro-F1  : {_delta["delta_macro_f1"]:+.4f}  '
          f'({_delta["pct_f1_degradation"]:+.1f}%)')
    print(f'    ΔRipe-F1   : {_delta["delta_ripe_f1"]:+.4f}')
    print(f'    ΔUnripe-F1 : {_delta["delta_unripe_f1"]:+.4f}')
    print(f'    ΔROC-AUC   : {_delta["delta_auc_roc"]:+.4f}')
    print(f'    ΔMCC       : {_delta["delta_mcc"]:+.4f}')

    # ── Save E3_NIGHT_METRICS ──────────────────────────────────────────────
    E3_NIGHT_METRICS = {
        'experiment'       : 'E3-DL-Day->Night',
        'model'            : DL_MODEL_NAME,
        'split'            : 'test_night',
        'n_samples'        : len(_ds_test_night),
        'macro_f1'         : round(_macro_f1_n, 6),
        'macro_f1_ci_95'   : [round(_f1_ci_n[0], 6), round(_f1_ci_n[1], 6)],
        'auc_pr'           : round(_auc_pr_n, 6),
        'auc_roc'          : round(_auc_roc_n, 6),
        'mcc'              : round(_mcc_n, 6),
        'accuracy'         : round(_acc_n, 6),
        'mean_macro_f1'    : round(_mean_f1_n, 6),
        'std_macro_f1'     : round(_std_f1_n, 6),
        'training_seeds'   : EXPERIMENT_SEEDS,
        'per_seed_f1'      : {k: round(v, 6) for k, v in _seed_f1s_n.items()},
        'per_class'        : {
            _cls: {
                'precision': round(_report_n[_cls]['precision'], 6),
                'recall'   : round(_report_n[_cls]['recall'], 6),
                'f1'       : round(_report_n[_cls]['f1-score'], 6),
                'support'  : int(_report_n[_cls]['support']),
            }
            for _cls in CLASS_NAMES
        },
        'confusion_matrix' : _cm_n.tolist(),
        'day_night_delta'  : _delta,
    }
    with open(_e3_metrics_path, 'w') as _f:
        json.dump(E3_NIGHT_METRICS, _f, indent=2)
    print(f'\n  Metrics saved -> {_e3_metrics_path}')

    # ── E3 Confusion matrix figure ─────────────────────────────────────────
    _cm_n_norm = _cm_n.astype('float') / _cm_n.sum(axis=1, keepdims=True)
    _fig_e3, _ax_e3 = plt.subplots(figsize=(6, 5))
    sns.heatmap(_cm_n_norm, annot=False, cmap='Reds',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=_ax_e3, vmin=0, vmax=1)
    for _i in range(len(CLASS_NAMES)):
        for _j in range(len(CLASS_NAMES)):
            _ax_e3.text(_j+0.5, _i+0.5,
                        f'{_cm_n_norm[_i,_j]:.2f}\n({_cm_n[_i,_j]})',
                        ha='center', va='center', fontsize=10,
                        color='white' if _cm_n_norm[_i,_j] > 0.6 else 'black')
    _ax_e3.set_xlabel('Predicted', fontsize=11)
    _ax_e3.set_ylabel('True', fontsize=11)
    _ax_e3.set_title(
        f'E3 (DL-Day→Night) Confusion Matrix\n'
        f'Macro-F1={_macro_f1_n:.4f}  MCC={_mcc_n:.4f}  (test_night)',
        fontsize=10)
    _fig_e3.tight_layout()
    _fig_e3.savefig(DIRS['figures'] / 'E3_confusion_matrix.png', dpi=300)
    plt.show()
    print('  Figure saved -> E3_confusion_matrix.png')

# ── Update metadata ───────────────────────────────────────────────────────
METADATA['E3_test_night'] = {
    'macro_f1'         : E3_NIGHT_METRICS['macro_f1'],
    'auc_roc'          : E3_NIGHT_METRICS['auc_roc'],
    'mcc'              : E3_NIGHT_METRICS['mcc'],
    'day_night_delta'  : E3_NIGHT_METRICS['day_night_delta'],
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)
print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Step 7.2 complete -- E3_NIGHT_METRICS locked.')


### Step 7.3 — E4: Degradation Quantification

Computes the full E1→E3 delta table and assesses seed-level consistency of the domain shift.  
No retraining — loads `E1_DAY_METRICS` and `E3_NIGHT_METRICS` from JSON.

> **Note on statistical significance:** With N=3 seeds, the minimum achievable p-value from a  
> paired permutation test is 1/2³ = 0.125. Formal p<0.05 is not achievable. We instead report  
> per-seed deltas and their directional consistency as the evidence of degradation.


In [ ]:
# ============================================================
# STEP 7.3 -- E4: DEGRADATION QUANTIFICATION
# ============================================================

print_section('Step 7.3 -- E4: Degradation Quantification')

# ── Load metrics (already computed -- fast) ────────────────────────────────
_e1 = E1_DAY_METRICS
_e3 = E3_NIGHT_METRICS
_delta = _e3['day_night_delta']   # pre-computed in Step 7.2

# ── E4 master delta table ─────────────────────────────────────────────────
E4_DEGRADATION = {
    'experiment_baseline' : 'E1-Day',
    'experiment_shifted'  : 'E3-DL-Day->Night',
    'delta_macro_f1'      : _delta['delta_macro_f1'],
    'delta_accuracy'      : _delta['delta_accuracy'],
    'delta_auc_roc'       : _delta['delta_auc_roc'],
    'delta_mcc'           : _delta['delta_mcc'],
    'delta_ripe_f1'       : _delta['delta_ripe_f1'],
    'delta_unripe_f1'     : _delta['delta_unripe_f1'],
    'pct_f1_degradation'  : _delta['pct_f1_degradation'],
    # Per-seed deltas for consistency check (N=3 precludes formal p<0.05)
    'per_seed_delta_f1'   : {
        str(_s): round(
            _e1['per_seed_f1'][str(_s)] - _e3['per_seed_f1'][str(_s)], 6
        )
        for _s in EXPERIMENT_SEEDS
    },
    'all_seeds_degrade'   : all(
        _e1['per_seed_f1'][str(_s)] > _e3['per_seed_f1'][str(_s)]
        for _s in EXPERIMENT_SEEDS
    ),
    'significance_note'   : (
        'N=3 seeds: min achievable p-value = 0.125 (paired permutation). '
        'Formal p<0.05 not achievable. Degradation reported as directionally '
        'consistent across all 3 seeds instead.'
    ),
}

# ── Print E4 table ────────────────────────────────────────────────────────
print('\n  E4 Degradation Table: E1 (Day) -> E3 (Night)')
print(f'  {"-"*55}')
_rows = [
    ('Macro-F1',    _e1['macro_f1'],  _e3['macro_f1'],  _delta['delta_macro_f1']),
    ('Accuracy',   _e1['accuracy'],  _e3['accuracy'],  _delta['delta_accuracy']),
    ('ROC-AUC',    _e1['auc_roc'],   _e3['auc_roc'],   _delta['delta_auc_roc']),
    ('MCC',        _e1['mcc'],       _e3['mcc'],       _delta['delta_mcc']),
    ('Ripe-F1',    _e1['per_class']['ripe']['f1'],
                   _e3['per_class']['ripe']['f1'],   _delta['delta_ripe_f1']),
    ('Unripe-F1',  _e1['per_class']['unripe']['f1'],
                   _e3['per_class']['unripe']['f1'], _delta['delta_unripe_f1']),
]
print(f'  {"Metric":<14} {"E1 (Day)":>10} {"E3 (Night)":>12} {"Delta":>10}')
print(f'  {"-"*50}')
for _name, _v1, _v3, _dv in _rows:
    print(f'  {_name:<14} {_v1:>10.4f} {_v3:>12.4f} {_dv:>+10.4f}')
print(f'\n  F1 degradation: {_delta["pct_f1_degradation"]:+.1f}%')

# ── Seed consistency table ────────────────────────────────────────────────
print(f'\n  Per-seed macro-F1 and delta (domain shift consistency):')
print(f'  {"Seed":<8} {"E1 F1":>10} {"E3 F1":>10} {"Delta":>10} {"Direction"}')
print(f'  {"-"*52}')
_n_degrade = 0
for _s in EXPERIMENT_SEEDS:
    _f1e1 = _e1['per_seed_f1'][str(_s)]
    _f1e3 = _e3['per_seed_f1'][str(_s)]
    _dv   = E4_DEGRADATION['per_seed_delta_f1'][str(_s)]
    _dir  = '↓ DEGRADES' if _dv > 0 else '↑ IMPROVES'
    if _dv > 0: _n_degrade += 1
    print(f'  {_s:<8} {_f1e1:>10.4f} {_f1e3:>10.4f} {_dv:>+10.4f}  {_dir}')
print(f'\n  {_n_degrade}/{len(EXPERIMENT_SEEDS)} seeds show degradation under domain shift.')
print(f'  Note: {E4_DEGRADATION["significance_note"]}')

# ── Save E4_DEGRADATION ───────────────────────────────────────────────────
_e4_path = DIRS['metrics'] / 'E4_degradation.json'
with open(_e4_path, 'w') as _f:
    json.dump(E4_DEGRADATION, _f, indent=2)
print(f'\n  E4 saved -> {_e4_path}')

# ── FIGURE 1: Side-by-side E1 vs E3 confusion matrices ───────────────────
_e1_cm     = np.array(_e1['confusion_matrix'])
_e3_cm     = np.array(_e3['confusion_matrix'])
_e1_cm_n   = _e1_cm.astype('float') / _e1_cm.sum(axis=1, keepdims=True)
_e3_cm_n   = _e3_cm.astype('float') / _e3_cm.sum(axis=1, keepdims=True)

_fig4, (_ax4a, _ax4b) = plt.subplots(1, 2, figsize=(12, 5))
_fig4.suptitle('E4: E1 (Day) vs E3 (Night) Confusion Matrices', fontsize=12, fontweight='bold')

for _ax4, _cm4, _cm4n, _title4, _cmap4 in [
    (_ax4a, _e1_cm, _e1_cm_n, f'E1 (Day)  Macro-F1={_e1["macro_f1"]:.4f}', 'Blues'),
    (_ax4b, _e3_cm, _e3_cm_n, f'E3 (Night) Macro-F1={_e3["macro_f1"]:.4f}', 'Reds'),
]:
    sns.heatmap(_cm4n, annot=False, cmap=_cmap4,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=_ax4, vmin=0, vmax=1)
    for _i in range(len(CLASS_NAMES)):
        for _j in range(len(CLASS_NAMES)):
            _ax4.text(_j+0.5, _i+0.5,
                      f'{_cm4n[_i,_j]:.2f}\n({_cm4[_i,_j]})',
                      ha='center', va='center', fontsize=10,
                      color='white' if _cm4n[_i,_j] > 0.6 else 'black')
    _ax4.set_xlabel('Predicted', fontsize=10)
    _ax4.set_ylabel('True', fontsize=10)
    _ax4.set_title(_title4, fontsize=10)

_fig4.tight_layout()
_fig4.savefig(DIRS['figures'] / 'E4_confusion_matrix_comparison.png', dpi=300)
plt.show()
print('  Figure saved -> E4_confusion_matrix_comparison.png')

# ── FIGURE 2: F1 comparison bar chart (E1, E2-Day, E2-Night, E3) ─────────
# Load E2 metrics (computed in Phase 5)
with open(DIRS['metrics'] / 'E2_day_metrics.json') as _f:
    _e2_day_metrics   = json.load(_f)
with open(DIRS['metrics'] / 'E2_night_metrics.json') as _f:
    _e2_night_metrics = json.load(_f)

_exp_labels = ['E1\nDL-Day', 'E2\nSVM-Day', 'E2-N\nSVM-Night', 'E3\nDL-Night']
_exp_f1s    = [
    _e1['macro_f1'],
    _e2_day_metrics['macro_f1'],
    _e2_night_metrics['macro_f1'],
    _e3['macro_f1'],
]
_exp_cols   = ['#2196F3', '#FF9800', '#FF5722', '#F44336']

_fig5, _ax5 = plt.subplots(figsize=(8, 5))
_bars = _ax5.bar(_exp_labels, _exp_f1s, color=_exp_cols, edgecolor='white',
                  linewidth=1.5, width=0.55)
_ax5.axhline(_e1['macro_f1'], color='#2196F3', linestyle='--',
              linewidth=1.2, label=f'E1 ceiling = {_e1["macro_f1"]:.4f}')
for _bar, _f1 in zip(_bars, _exp_f1s):
    _ax5.text(_bar.get_x() + _bar.get_width()/2,
               _bar.get_height() + 0.005,
               f'{_f1:.4f}', ha='center', va='bottom', fontsize=10)
_ax5.set_ylim(0, 1.05)
_ax5.set_ylabel('Macro-F1', fontsize=11)
_ax5.set_title(
    'E4: Macro-F1 Comparison (E1–E3)\n'
    f'ΔF1 (DL shift) = {_delta["delta_macro_f1"]:+.4f} '
    f'({_delta["pct_f1_degradation"]:+.1f}%)',
    fontsize=11, fontweight='bold')
_ax5.legend(fontsize=9)
_ax5.grid(axis='y', alpha=0.3)
_fig5.tight_layout()
_fig5.savefig(DIRS['figures'] / 'E4_f1_comparison_bar.png', dpi=300)
plt.show()
print('  Figure saved -> E4_f1_comparison_bar.png')

# ── Update metadata ───────────────────────────────────────────────────────
METADATA['E4_degradation'] = {
    'delta_macro_f1'      : E4_DEGRADATION['delta_macro_f1'],
    'pct_f1_degradation'  : E4_DEGRADATION['pct_f1_degradation'],
    'all_seeds_degrade'   : E4_DEGRADATION['all_seeds_degrade'],
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)
print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Step 7.3 / Phase 7 complete -- domain shift quantified.')


---
## Phase 8 — Experiment E5: Domain Adaptation (Mixed Day+Night)

**Goal**: Recover the performance loss shown in E3 by retraining on a stochastic mix of day  
and LED-simulated night patches (p_night=0.5). Hyperparameters identical to E1.  
Evaluates on **both** test_night (recovery check) and test_day (stability check).

| Step | Action |
|---|---|
| 8.1 | Verify LED transform call signature; instantiate adapted training dataset |
| 8.2 | E5 training (3 seeds) + dual evaluation on test_night and test_day |


### Step 8.1 — Adapted DataLoader (Stochastic LED Mixing)

Verifies the `LEDNightSimulation` call interface and defines `StrawberryPatchDatasetAdapted`  
with stochastic LED application at `__getitem__` time (p_night=0.5).


In [ ]:
# ============================================================
# STEP 8.1 -- ADAPTED DATALOADER (STOCHASTIC LED MIXING)
# ============================================================
print_section('Step 8.1 -- Adapted DataLoader (Stochastic LED Mixing)')

E5_P_NIGHT = 0.5   # probability each train patch gets LED simulation applied

# ── Verify LED transform call signature ──────────────────────────────────
# LED_TRAIN_TRANSFORM is defined in Phase 4 (Step 4.2).
# Two possible interfaces depending on implementation:
#   A) Albumentations-style: transform(image=arr)['image']
#   B) Custom .apply() method: transform.apply(arr)
# We test both and store the correct call pattern.

_dummy_img = np.zeros((224, 224, 3), dtype=np.uint8)
_led_call_mode = None

# Test A: Albumentations-style __call__
try:
    _result_a = LED_TRAIN_TRANSFORM(image=_dummy_img)['image']
    assert isinstance(_result_a, np.ndarray) and _result_a.shape == (224, 224, 3)
    _led_call_mode = 'albumentations'
    print('  LED transform interface: albumentations (image=arr)["image"]')
except Exception as _e_a:
    # Test B: custom .apply() method
    try:
        _result_b = LED_TRAIN_TRANSFORM.apply(_dummy_img)
        assert isinstance(_result_b, np.ndarray) and _result_b.shape == (224, 224, 3)
        _led_call_mode = 'apply'
        print('  LED transform interface: custom .apply(arr)')
    except Exception as _e_b:
        raise RuntimeError(
            f'LED transform call failed both interfaces.\n'
            f'  Albumentations error: {_e_a}\n'
            f'  Apply error: {_e_b}'
        )

print(f'  p_night = {E5_P_NIGHT}  (50% of each batch LED-simulated per Ben-David et al. 2010)')


# ── Adapted Dataset class ─────────────────────────────────────────────────
class StrawberryPatchDatasetAdapted(torch.utils.data.Dataset):
    """
    Extended StrawberryPatchDataset for E5 domain adaptation training.

    Applies LED simulation stochastically at __getitem__ time with
    probability p_night (default 0.5). Each epoch sees a different
    random day/night mix, providing additional regularisation.

    Call interface is resolved at class initialisation from _led_call_mode.
    """

    def __init__(self, ripe_dir, unripe_dir, transform=None,
                 led_transform=None, p_night=0.5, led_call_mode='albumentations'):
        super().__init__()
        self.transform      = transform
        self.led_transform  = led_transform
        self.p_night        = p_night
        self.led_call_mode  = led_call_mode
        self.samples = []

        for _p in sorted(Path(ripe_dir).glob('*.jpg')):
            self.samples.append((str(_p), 0))
        for _p in sorted(Path(unripe_dir).glob('*.jpg')):
            self.samples.append((str(_p), 1))

        if not self.samples:
            raise RuntimeError(f'No patches found in {ripe_dir} or {unripe_dir}')

    def __len__(self):
        return len(self.samples)

    def __repr__(self):
        return (f'StrawberryPatchDatasetAdapted(n={len(self)}, '
                f'p_night={self.p_night}, led={self.led_transform is not None})')

    def _apply_led(self, img_np):
        """Apply LED transform using verified call interface."""
        if self.led_call_mode == 'albumentations':
            return self.led_transform(image=img_np)['image']
        else:
            return self.led_transform.apply(img_np)

    def __getitem__(self, idx):
        _path, _label = self.samples[idx]
        _img = Image.open(_path).convert('RGB')

        # Stochastic LED augmentation (p_night probability per sample)
        if (self.led_transform is not None) and (torch.rand(1).item() < self.p_night):
            _img_np = np.array(_img, dtype=np.uint8)
            _img_np = self._apply_led(_img_np)
            _img    = Image.fromarray(_img_np)

        if self.transform is not None:
            _img = self.transform(_img)

        return _img, _label

    def class_counts(self):
        from collections import Counter
        return dict(Counter(lbl for _, lbl in self.samples))


# ── Instantiate adapted training dataset ──────────────────────────────────
_ds_train_adapted = StrawberryPatchDatasetAdapted(
    DIRS['train_ripe'],
    DIRS['train_unripe'],
    transform     = DL_TRAIN_TRANSFORM,
    led_transform = LED_TRAIN_TRANSFORM,
    p_night       = E5_P_NIGHT,
    led_call_mode = _led_call_mode,
)

print(f'\n  Adapted dataset: {_ds_train_adapted}')
_cc = _ds_train_adapted.class_counts()
print(f'  Class counts: ripe={_cc.get(0,0)}, unripe={_cc.get(1,0)}, total={len(_ds_train_adapted)}')
print(f'  Expected night patches per epoch: ~{int(len(_ds_train_adapted) * E5_P_NIGHT)}')

# ── Visual sanity check: same patch, day vs adapted (stochastic) ──────────
set_seed(GLOBAL_SEED)
_fig8, _axes8 = plt.subplots(2, 6, figsize=(15, 5))
_fig8.suptitle(
    'Step 8.1 -- Adapted Dataset Sanity Check\n'
    'Top: original StrawberryPatchDataset (day)  |  '
    'Bottom: StrawberryPatchDatasetAdapted (stochastic 50% LED)',
    fontsize=9, fontweight='bold')

def _denorm(tensor):
    _m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    _s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * _s + _m).clamp(0, 1).permute(1, 2, 0).numpy()

for _col in range(6):
    _day_t, _lbl = _ds_train[_col]
    _adp_t, _    = _ds_train_adapted[_col]
    _axes8[0, _col].imshow(_denorm(_day_t))
    _axes8[0, _col].set_title(CLASS_NAMES[_lbl], fontsize=8)
    _axes8[0, _col].axis('off')
    _axes8[1, _col].imshow(_denorm(_adp_t))
    _axes8[1, _col].set_title('adapted', fontsize=8)
    _axes8[1, _col].axis('off')

_fig8.tight_layout()
_fig8.savefig(DIRS['figures'] / 'E5_adapted_dataset_verification.png', dpi=300)
plt.show()
print('  Figure saved -> E5_adapted_dataset_verification.png')
print('\n[OK] Step 8.1 complete -- adapted dataset ready.')


### Step 8.2 — E5 Training (3 Seeds) + Dual Evaluation

Trains on the adapted dataset (50% LED night) with **identical hyperparameters to E1** (Step 6.4).  
After training: evaluates on both `test_night` (ΔE3→E5 recovery) and `test_day` (E1 stability check).


In [ ]:
# ============================================================
# STEP 8.2 -- E5 TRAINING LOOP (3 SEEDS, IDENTICAL TO E1)
# ============================================================
import csv, time

print_section('Step 8.2 -- E5 Training (3 Seeds, Adapted Dataset)')

# ── Idempotency guard ─────────────────────────────────────────────────────
_e5_results_path = DIRS['metrics'] / 'E5_train_results.json'
if _e5_results_path.exists():
    print('  Already complete -- loading cached E5 training results.')
    with open(_e5_results_path) as _f:
        E5_TRAIN_RESULTS = json.load(_f)
    E5_CHECKPOINTS = {
        s: DIRS['checkpoints'] / f'E5_seed{s}_best.pt'
        for s in EXPERIMENT_SEEDS
    }
    _missing_e5 = [str(p) for p in E5_CHECKPOINTS.values() if not p.exists()]
    if _missing_e5:
        raise FileNotFoundError(
            'E5 checkpoints missing (delete E5_train_results.json to re-train):\n'
            + '\n'.join(_missing_e5)
        )
    print(f'  Mean val macro-F1 : {E5_TRAIN_RESULTS["mean_val_macro_f1"]:.4f}')
else:

    _seed_logs_e5    = {}
    _seed_best_f1_e5 = {}
    E5_CHECKPOINTS   = {}
    _t_start_e5      = time.time()

    for _seed in EXPERIMENT_SEEDS:
        print(f'\n  {"-"*52}')
        print(f'  Seed {_seed} ({EXPERIMENT_SEEDS.index(_seed)+1}/{len(EXPERIMENT_SEEDS)})')
        print(f'  {"-"*52}')

        # Fresh model + optimizer (IDENTICAL hyperparams to E1)
        set_seed(_seed)
        _model = build_convnext_tiny(DL_NUM_CLASSES, pretrained=True)
        _model = freeze_convnext_backbone(_model)
        _model = _model.to(DEVICE)
        _optimizer, _scheduler, _scaler = create_optimizer_scheduler(_model)

        # Seeded DataLoader on adapted dataset
        _gen_e5 = torch.Generator()
        _gen_e5.manual_seed(_seed)

        _loader_train_e5 = torch.utils.data.DataLoader(
            _ds_train_adapted,
            batch_size         = DL_BATCH_SIZE,
            shuffle            = True,
            num_workers        = DL_NUM_WORKERS,
            pin_memory         = DL_PIN_MEMORY,
            drop_last          = DL_DROP_LAST,
            persistent_workers = DL_PERSISTENT_WORKERS,
            generator          = _gen_e5,
        )
        # Val DataLoader: day patches (same as E1 — monitor val F1 on clean domain)
        _loader_val_e5 = torch.utils.data.DataLoader(
            _ds_val,
            batch_size         = DL_BATCH_SIZE * 2,
            shuffle            = False,
            num_workers        = DL_NUM_WORKERS,
            pin_memory         = DL_PIN_MEMORY,
            persistent_workers = DL_PERSISTENT_WORKERS,
        )

        _log_path_e5  = DIRS['logs']        / f'E5_seed{_seed}_training_log.csv'
        _ckpt_path_e5 = DIRS['checkpoints'] / f'E5_seed{_seed}_best.pt'

        _log_rows_e5   = []
        _best_val_f1   = 0.0
        _patience_ctr  = 0
        _t_seed_start  = time.time()

        for _epoch in range(DL_MAX_EPOCHS):

            # ── TRAIN ─────────────────────────────────────────────────────
            _model.train()
            _tr_loss_sum, _n_steps = 0.0, 0

            for _imgs, _labels in _loader_train_e5:
                _imgs   = _imgs.to(DEVICE, non_blocking=True)
                _labels = _labels.to(DEVICE, non_blocking=True)
                _optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=DL_USE_AMP):
                    _logits = _model(_imgs)
                    _loss   = DL_LOSS_FN(_logits, _labels)

                _scaler.scale(_loss).backward()
                _scaler.unscale_(_optimizer)
                torch.nn.utils.clip_grad_norm_(_model.parameters(), DL_GRAD_CLIP_NORM)
                _scaler.step(_optimizer)
                _scaler.update()

                _tr_loss_sum += _loss.item()
                _n_steps     += 1

            _tr_loss_avg = _tr_loss_sum / max(_n_steps, 1)

            # ── VAL ───────────────────────────────────────────────────────
            _model.eval()
            _vl_loss_sum, _vl_preds, _vl_labels = 0.0, [], []

            with torch.no_grad():
                for _imgs, _labels in _loader_val_e5:
                    _imgs   = _imgs.to(DEVICE, non_blocking=True)
                    _labels = _labels.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast(device_type=DEVICE.type, enabled=DL_USE_AMP):
                        _logits = _model(_imgs)
                        _vloss  = DL_LOSS_FN(_logits, _labels)
                    _vl_loss_sum += _vloss.item()
                    _vl_preds.extend(_logits.argmax(dim=1).cpu().tolist())
                    _vl_labels.extend(_labels.cpu().tolist())

            _vl_loss_avg = _vl_loss_sum / len(_loader_val_e5)
            _vl_f1       = f1_score(_vl_labels, _vl_preds, average='macro', zero_division=1)
            _vl_acc      = sum(p==l for p, l in zip(_vl_preds, _vl_labels)) / len(_vl_labels)

            _scheduler.step()
            _cur_lr = _optimizer.param_groups[-1]['lr']

            _row = {
                'epoch'        : _epoch,
                'train_loss'   : round(_tr_loss_avg, 5),
                'val_loss'     : round(_vl_loss_avg, 5),
                'val_macro_f1' : round(_vl_f1, 5),
                'val_accuracy' : round(_vl_acc, 5),
                'lr'           : f'{_cur_lr:.2e}',
            }
            _log_rows_e5.append(_row)

            _star = ' *' if _vl_f1 > _best_val_f1 else ''
            print(f'  Ep {_epoch:02d}/{DL_MAX_EPOCHS-1}  '
                  f'trLoss={_tr_loss_avg:.4f}  vlLoss={_vl_loss_avg:.4f}  '
                  f'valF1={_vl_f1:.4f}  lr={_cur_lr:.1e}{_star}')

            if _vl_f1 > _best_val_f1:
                _best_val_f1  = _vl_f1
                _patience_ctr = 0
                torch.save({
                    'model_state'     : _model.state_dict(),
                    'optimizer_state' : _optimizer.state_dict(),
                    'epoch'           : _epoch,
                    'val_macro_f1'    : _vl_f1,
                    'seed'            : _seed,
                    'architecture'    : DL_MODEL_NAME,
                    'experiment'      : 'E5-Adapted',
                }, _ckpt_path_e5)
            else:
                _patience_ctr += 1
                if _patience_ctr >= DL_EARLY_STOP_PATIENCE:
                    print(f'  Early stop at epoch {_epoch}')
                    break

        with open(_log_path_e5, 'w', newline='') as _csvf:
            csv.DictWriter(_csvf, fieldnames=_log_rows_e5[0].keys()).writeheader()
            csv.DictWriter(_csvf, fieldnames=_log_rows_e5[0].keys()).writerows(_log_rows_e5)

        _elapsed = (time.time() - _t_seed_start) / 60
        print(f'\n  Seed {_seed}: {_elapsed:.1f} min | best val F1 = {_best_val_f1:.4f}')

        _seed_logs_e5[str(_seed)]    = _log_rows_e5
        _seed_best_f1_e5[str(_seed)] = _best_val_f1
        E5_CHECKPOINTS[_seed]        = _ckpt_path_e5

        del _model, _optimizer, _scheduler, _scaler
        torch.cuda.empty_cache()

    _total_e5 = (time.time() - _t_start_e5) / 60
    _f1_vals  = list(_seed_best_f1_e5.values())
    _mean_e5  = float(np.mean(_f1_vals))
    _std_e5   = float(np.std(_f1_vals))

    print(f'\n  E5 val mean±std: {_mean_e5:.4f} ± {_std_e5:.4f}  ({_total_e5:.1f} min total)')

    E5_TRAIN_RESULTS = {
        'experiment'         : 'E5-Adapted',
        'model'              : DL_MODEL_NAME,
        'seeds'              : EXPERIMENT_SEEDS,
        'p_night'            : E5_P_NIGHT,
        'mean_val_macro_f1'  : _mean_e5,
        'std_val_macro_f1'   : _std_e5,
        'per_seed_best_val_f1': _seed_best_f1_e5,
        'total_train_min'    : round(_total_e5, 2),
        'per_seed_logs'      : _seed_logs_e5,
        'checkpoints'        : {str(s): str(p) for s, p in E5_CHECKPOINTS.items()},
    }
    with open(_e5_results_path, 'w') as _f:
        json.dump(E5_TRAIN_RESULTS, _f, indent=2)
    print(f'  E5 training results saved -> {_e5_results_path}')

print(f'\n[OK] Step 8.2 (training) complete.')


In [ ]:
# ============================================================
# STEP 8.2 (CONT.) -- E5 DUAL EVALUATION + RECOVERY METRICS
# ============================================================
print_section('Step 8.2 (Eval) -- E5 Dual Evaluation: Night + Day')

_e5_night_path = DIRS['metrics'] / 'E5_night_metrics.json'
_e5_day_path   = DIRS['metrics'] / 'E5_day_metrics.json'

if _e5_night_path.exists() and _e5_day_path.exists():
    print('  Already complete -- loading cached E5 evaluation metrics.')
    with open(_e5_night_path) as _f: E5_NIGHT_METRICS = json.load(_f)
    with open(_e5_day_path)   as _f: E5_DAY_METRICS   = json.load(_f)
else:

    def _evaluate_split(loader, seeds, ckpt_prefix, device, use_amp):
        """Evaluate all seed checkpoints on a DataLoader, return pooled results."""
        _all_preds, _all_labels, _all_probas, _seed_f1s = [], [], [], {}
        for _seed in seeds:
            _ckpt  = torch.load(ckpt_prefix.format(_seed), map_location=device, weights_only=False)
            _model = build_convnext_tiny(DL_NUM_CLASSES, pretrained=False)
            _model = freeze_convnext_backbone(_model)
            _model.load_state_dict(_ckpt['model_state'])
            _model = _model.to(device)
            _model.eval()
            _sp, _sl, _sr = [], [], []
            with torch.no_grad():
                for _imgs, _labels in loader:
                    _imgs   = _imgs.to(device, non_blocking=True)
                    _labels = _labels.to(device, non_blocking=True)
                    with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                        _logits = _model(_imgs)
                    _pr = _logits.float().softmax(dim=1)
                    _sp.extend(_pr.argmax(dim=1).cpu().tolist())
                    _sl.extend(_labels.cpu().tolist())
                    _sr.extend(_pr.cpu().tolist())
            _seed_f1s[str(_seed)] = f1_score(_sl, _sp, average='macro', zero_division=1)
            print(f'    seed {_seed} macro-F1: {_seed_f1s[str(_seed)]:.4f}')
            _all_preds.extend(_sp); _all_labels.extend(_sl); _all_probas.extend(_sr)
            del _model; torch.cuda.empty_cache()
        return np.array(_all_labels), np.array(_all_preds), np.array(_all_probas), _seed_f1s

    def _build_metrics(al, ap, apr, sf, n_samples, experiment, split):
        """Compute full METRICS_SCHEMA dict."""
        _report = classification_report(al, ap, target_names=CLASS_NAMES,
                                         output_dict=True, zero_division=1)
        return {
            'experiment'     : experiment,
            'model'          : DL_MODEL_NAME,
            'split'          : split,
            'n_samples'      : n_samples,
            'macro_f1'       : round(f1_score(al, ap, average='macro', zero_division=1), 6),
            'macro_f1_ci_95' : [round(v, 6) for v in _bootstrap_macro_f1(al, ap)],
            'auc_pr'         : round(average_precision_score(al, apr, average='macro'), 6),  # 2-D proba for true binary macro-AP
            'auc_roc'        : round(roc_auc_score(al, apr[:, 1], average='macro'), 6),
            'mcc'            : round(matthews_corrcoef(al, ap), 6),
            'accuracy'       : round(float((al == ap).mean()), 6),
            'mean_macro_f1'  : round(float(np.mean(list(sf.values()))), 6),
            'std_macro_f1'   : round(float(np.std(list(sf.values()))), 6),
            'training_seeds' : EXPERIMENT_SEEDS,
            'per_seed_f1'    : {k: round(v, 6) for k, v in sf.items()},
            'per_class'      : {
                _cls: {
                    'precision': round(_report[_cls]['precision'], 6),
                    'recall'   : round(_report[_cls]['recall'], 6),
                    'f1'       : round(_report[_cls]['f1-score'], 6),
                    'support'  : int(_report[_cls]['support']),
                } for _cls in CLASS_NAMES
            },
            'confusion_matrix': confusion_matrix(al, ap).tolist(),
        }

    _ckpt_tmpl   = str(DIRS['checkpoints'] / 'E5_seed{}_best.pt')
    _loader_day  = torch.utils.data.DataLoader(
        _ds_test_day,  batch_size=DL_BATCH_SIZE*2, shuffle=False,
        num_workers=DL_NUM_WORKERS, pin_memory=DL_PIN_MEMORY,
        drop_last=False, persistent_workers=DL_PERSISTENT_WORKERS)
    _loader_night = torch.utils.data.DataLoader(
        _ds_test_night, batch_size=DL_BATCH_SIZE*2, shuffle=False,
        num_workers=DL_NUM_WORKERS, pin_memory=DL_PIN_MEMORY,
        drop_last=False, persistent_workers=DL_PERSISTENT_WORKERS)

    print('\n  -- Evaluating on test_night --')
    _al_n, _ap_n, _apr_n, _sf_n = _evaluate_split(
        _loader_night, EXPERIMENT_SEEDS, _ckpt_tmpl, DEVICE, DL_USE_AMP)
    E5_NIGHT_METRICS = _build_metrics(
        _al_n, _ap_n, _apr_n, _sf_n, len(_ds_test_night), 'E5-Night', 'test_night')

    print('\n  -- Evaluating on test_day --')
    _al_d, _ap_d, _apr_d, _sf_d = _evaluate_split(
        _loader_day, EXPERIMENT_SEEDS, _ckpt_tmpl, DEVICE, DL_USE_AMP)
    E5_DAY_METRICS = _build_metrics(
        _al_d, _ap_d, _apr_d, _sf_d, len(_ds_test_day), 'E5-Day', 'test_day')

    with open(_e5_night_path, 'w') as _f: json.dump(E5_NIGHT_METRICS, _f, indent=2)
    with open(_e5_day_path,   'w') as _f: json.dump(E5_DAY_METRICS,   _f, indent=2)
    print(f'\n  E5 metrics saved.')

# ── Recovery and stability metrics ────────────────────────────────────────
_e1_f1  = E1_DAY_METRICS['macro_f1']
_e3_f1  = E3_NIGHT_METRICS['macro_f1']
_e5n_f1 = E5_NIGHT_METRICS['macro_f1']
_e5d_f1 = E5_DAY_METRICS['macro_f1']

_gap            = _e1_f1 - _e3_f1
_recovered      = _e5n_f1 - _e3_f1
_recovery_ratio = round(_recovered / _gap, 4) if _gap > 0 else float('nan')
_day_retention  = round(_e5d_f1 / _e1_f1, 4) if _e1_f1 > 0 else float('nan')

E5_RECOVERY = {
    'recovery_ratio'  : _recovery_ratio,
    'day_retention'   : _day_retention,
    'e1_day_f1'       : round(_e1_f1, 6),
    'e3_night_f1'     : round(_e3_f1, 6),
    'e5_night_f1'     : round(_e5n_f1, 6),
    'e5_day_f1'       : round(_e5d_f1, 6),
    'gap_e1_e3'       : round(_gap, 6),
    'recovered_f1'    : round(_recovered, 6),
}
with open(DIRS['metrics'] / 'E5_recovery.json', 'w') as _f:
    json.dump(E5_RECOVERY, _f, indent=2)

print(f'\n  {"-"*52}')
print(f'  E5 Recovery Summary')
print(f'  {"-"*52}')
print(f'  E1 Day F1    : {_e1_f1:.4f}  (performance ceiling)')
print(f'  E3 Night F1  : {_e3_f1:.4f}  (domain shift floor)')
print(f'  E5 Night F1  : {_e5n_f1:.4f}  (after adaptation)')
print(f'  E5 Day F1    : {_e5d_f1:.4f}  (day stability check)')
print(f'  Recovery ratio    : {_recovery_ratio:.4f}  '
      f'({_recovery_ratio*100:.1f}% of gap closed)')
print(f'  Day retention     : {_day_retention:.4f}  '
      f'({_day_retention*100:.1f}% of E1 day F1 maintained)')

# ── Final comparison figure: 6-bar F1 chart ───────────────────────────────
with open(DIRS['metrics'] / 'E2_day_metrics.json') as _f:
    _e2_day   = json.load(_f)
with open(DIRS['metrics'] / 'E2_night_metrics.json') as _f:
    _e2_night = json.load(_f)

_labels8 = ['E1\nDL-Day', 'E2\nSVM-Day', 'E2-N\nSVM-Night',
             'E3\nDL-Night', 'E5-N\nDL-Adapted\nNight', 'E5-D\nDL-Adapted\nDay']
_f1s8    = [_e1_f1, _e2_day['macro_f1'], _e2_night['macro_f1'],
            _e3_f1, _e5n_f1, _e5d_f1]
_cols8   = ['#2196F3','#FF9800','#FF5722','#F44336','#9C27B0','#673AB7']

_fig8b, _ax8b = plt.subplots(figsize=(11, 5))
_bars8 = _ax8b.bar(_labels8, _f1s8, color=_cols8, edgecolor='white', linewidth=1.5, width=0.6)
_ax8b.axhline(_e1_f1, color='#2196F3', linestyle='--', linewidth=1.2,
               label=f'E1 ceiling = {_e1_f1:.4f}')
for _bar, _f1 in zip(_bars8, _f1s8):
    _ax8b.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 0.005,
                f'{_f1:.4f}', ha='center', va='bottom', fontsize=9)
_ax8b.set_ylim(0, 1.05)
_ax8b.set_ylabel('Macro-F1', fontsize=11)
_ax8b.set_title(
    f'E1–E5 Macro-F1 Comparison\n'
    f'Recovery={_recovery_ratio*100:.1f}%  |  Day retention={_day_retention*100:.1f}%',
    fontsize=11, fontweight='bold')
_ax8b.legend(fontsize=9)
_ax8b.grid(axis='y', alpha=0.3)
_fig8b.tight_layout()
_fig8b.savefig(DIRS['figures'] / 'E5_full_comparison_bar.png', dpi=300)
plt.show()
print('  Figure saved -> E5_full_comparison_bar.png')

# ── Update metadata ───────────────────────────────────────────────────────
METADATA['E5_adaptation'] = {
    'e5_night_macro_f1' : E5_NIGHT_METRICS['macro_f1'],
    'e5_day_macro_f1'   : E5_DAY_METRICS['macro_f1'],
    'recovery_ratio'    : _recovery_ratio,
    'day_retention'     : _day_retention,
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)
print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Phase 8 complete -- E5 training and dual evaluation done.')


---
## Phase 9 — Grad-CAM XAI (Interpretability Audit)

Generates saliency maps for three model states to provide mechanistic evidence of domain shift and adaptation:
- **E1 on day patches** (correctly classified) — baseline attention
- **E1 on night patches** (misclassified) — domain shift disrupts attention
- **E5 on night patches** (adapted) — attention partially recovered

> **ConvNeXt note**: requires `reshape_transform` (NHWC→NCHW) and `torch.enable_grad()` context for Grad-CAM.


### Step 9.1 — Install & Verify `pytorch-grad-cam`


In [ ]:
# ============================================================
# STEP 9.1 -- INSTALL AND VERIFY pytorch-grad-cam
# ============================================================
print_section('Step 9.1 -- Install & Verify pytorch-grad-cam')

import subprocess, sys
_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', 'grad-cam'],
    capture_output=True, text=True
)
if _result.returncode != 0:
    print(f'  [WARN] pip install had non-zero exit. stderr:\n{_result.stderr[:400]}')

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    print('  [OK] pytorch-grad-cam imported successfully')
except ImportError as _e:
    raise ImportError(f'grad-cam install failed: {_e}')

# ── ConvNeXt reshape_transform (NHWC -> NCHW) ────────────────────────────
# ConvNeXt blocks output (B, H, W, C) not (B, C, H, W).
# Without this transform, GradCAM produces incorrect heatmaps.
def convnext_reshape_transform(tensor):
    """
    Permute ConvNeXt NHWC feature output to NCHW for Grad-CAM.
    Input:  (B, H, W, C)  -- ConvNeXt block output
    Output: (B, C, H, W)  -- standard NCHW conv feature format
    """
    return tensor.permute(0, 3, 1, 2)

# Quick self-test on random tensor
import torch
_t = torch.zeros(2, 7, 7, 768)   # (B, H, W, C) ConvNeXt stage-4 shape
_t_out = convnext_reshape_transform(_t)
assert _t_out.shape == (2, 768, 7, 7), f'reshape_transform output wrong: {_t_out.shape}'
print(f'  [OK] reshape_transform: (B,H,W,C) -> (B,C,H,W) verified: {tuple(_t.shape)} -> {tuple(_t_out.shape)}')
print('\n[OK] Step 9.1 complete -- pytorch-grad-cam ready.')


### Steps 9.2–9.5 — Grad-CAM Maps + 6×3 Comparison Grid

Selects 6 patches where E1 **correctly** classifies day but **misclassifies** night.  
Generates Grad-CAM overlays for all three model states.  
Produces publication-quality 6×3 grid with prediction + confidence annotations.


In [ ]:
# ============================================================
# STEPS 9.2-9.5 -- GRAD-CAM MAPS + 6x3 COMPARISON GRID
# ============================================================
print_section('Steps 9.2-9.5 -- Grad-CAM Maps + Comparison Grid')

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Idempotency guard ─────────────────────────────────────────────────────
_gradcam_fig_path = DIRS['figures'] / 'gradcam_comparison_grid.png'
if _gradcam_fig_path.exists():
    print('  Already complete -- reloading figure.')
    _fig9 = plt.figure(figsize=(14, 10))
    _ax9  = _fig9.add_subplot(111)
    _ax9.imshow(plt.imread(_gradcam_fig_path))
    _ax9.axis('off')
    plt.show()
else:

    # ── Helper: denorm tensor to float RGB [0,1] ──────────────────────────
    def _denorm_np(tensor):
        _m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
        _s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
        return (tensor * _s + _m).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

    # ── Helper: load E1 or E5 best checkpoint (by seed with best val F1) ──
    def _load_best_checkpoint(prefix, seeds):
        """Load checkpoint with highest val_macro_f1 across seeds.
        Stores the winning checkpoint dict during the scan so the best
        checkpoint is only read from disk once (not twice).
        """
        _best_f1, _best_ckpt, _best_ck_dict = -1, None, None
        for _s in seeds:
            _p  = DIRS['checkpoints'] / f'{prefix}_seed{_s}_best.pt'
            _ck = torch.load(_p, map_location=DEVICE, weights_only=False)
            if _ck['val_macro_f1'] > _best_f1:
                _best_f1, _best_ckpt, _best_ck_dict = _ck['val_macro_f1'], _p, _ck
        print(f'    Best {prefix} checkpoint: {_best_ckpt.name}  '
              f'(val F1={_best_f1:.4f})')
        _model = build_convnext_tiny(DL_NUM_CLASSES, pretrained=False)
        _model = freeze_convnext_backbone(_model)
        _model.load_state_dict(_best_ck_dict['model_state'])  # reuse stored dict
        _model = _model.to(DEVICE)
        _model.eval()
        return _model

    # ── Helper: get model prediction + confidence on a single image ────────
    def _predict(model, tensor):
        """Returns (pred_class, confidence) for a single (C,H,W) tensor."""
        with torch.no_grad():
            _logits = model(tensor.unsqueeze(0).to(DEVICE))
            _proba  = _logits.float().softmax(dim=1)[0]
        _pred = int(_proba.argmax().item())
        return _pred, float(_proba[_pred].item())

    # ── Helper: generate GradCAM overlay for a single image ───────────────
    def _gradcam_overlay(model, tensor, target_class, target_layer):
        """
        Returns (H, W, 3) uint8 overlay image.
        Uses torch.enable_grad() -- required for GradCAM backward pass.
        """
        _rgb_np = _denorm_np(tensor)   # float [0,1] for show_cam_on_image
        with GradCAM(
            model             = model,
            target_layers     = [target_layer],
            reshape_transform = convnext_reshape_transform,
        ) as _cam:
            with torch.enable_grad():   # GradCAM requires gradients
                _grayscale = _cam(
                    input_tensor = tensor.unsqueeze(0).to(DEVICE),
                    targets      = [ClassifierOutputTarget(target_class)],
                )[0]   # (H, W) float [0,1]
        return show_cam_on_image(_rgb_np.astype('float32'), _grayscale, use_rgb=True)

    # ── Load E1 and E5 best models ────────────────────────────────────────
    print('  Loading model checkpoints...')
    _model_e1 = _load_best_checkpoint('E1', EXPERIMENT_SEEDS)
    _model_e5 = _load_best_checkpoint('E5', EXPERIMENT_SEEDS)

    # Target layer: final block of Stage 4 (ConvNeXt highest semantic layer)
    _target_layer_e1 = _model_e1.features[7][-1]
    _target_layer_e5 = _model_e5.features[7][-1]

    # ── Strategic patch selection ─────────────────────────────────────────
    # Find patches where E1 CORRECTLY classifies day AND MISCLASSIFIES night.
    # We want 3 ripe + 3 unripe such patches.
    print('\n  Selecting strategically misclassified night patches...')

    _selected = {'ripe': [], 'unripe': []}   # list of (day_tensor, night_tensor, true_label)
    _N_WANT   = 3   # per class

    for _class_name, _class_id in [('ripe', 0), ('unripe', 1)]:
        _class_day_idxs = [
            i for i, (_, l) in enumerate(_ds_test_day.samples) if l == _class_id
        ]
        for _idx in _class_day_idxs:
            if len(_selected[_class_name]) >= _N_WANT:
                break
            _t_day,   _ = _ds_test_day[_idx]
            _t_night, _ = _ds_test_night[_idx]

            _pred_day,   _conf_day   = _predict(_model_e1, _t_day)
            _pred_night, _conf_night = _predict(_model_e1, _t_night)

            # E1 correct on day AND wrong on night (prefer high-confidence errors)
            if _pred_day == _class_id and _pred_night != _class_id:
                _selected[_class_name].append({
                    'day_t'      : _t_day,
                    'night_t'    : _t_night,
                    'true_label' : _class_id,
                    'conf_day'   : _conf_day,
                    'pred_night' : _pred_night,
                    'conf_night' : _conf_night,
                })

        # Fallback: if not enough misclassified night patches, use any correct-day patches
        if len(_selected[_class_name]) < _N_WANT:
            print(f'    [INFO] Only {len(_selected[_class_name])} misclassified night '
                  f'{_class_name} patches found, using any correct-day patches as fallback.')
            for _idx in _class_day_idxs:
                if len(_selected[_class_name]) >= _N_WANT:
                    break
                # avoid duplicates
                _existing_days = [s['day_t'] for s in _selected[_class_name]]
                _t_day, _ = _ds_test_day[_idx]
                if not any(torch.equal(_t_day, _ed) for _ed in _existing_days):
                    _t_night, _ = _ds_test_night[_idx]
                    _pred_day, _conf_day = _predict(_model_e1, _t_day)
                    _pred_night, _conf_night = _predict(_model_e1, _t_night)
                    if _pred_day == _class_id:
                        _selected[_class_name].append({
                            'day_t': _t_day, 'night_t': _t_night,
                            'true_label': _class_id,
                            'conf_day': _conf_day,
                            'pred_night': _pred_night,
                            'conf_night': _conf_night,
                        })

    print(f'  Selected: ripe={len(_selected["ripe"])}, unripe={len(_selected["unripe"])} patches')

    # Flatten to list of 6 patches: 3 ripe first, then 3 unripe
    _patches = _selected['ripe'][:3] + _selected['unripe'][:3]

    # ── Generate GradCAM overlays ─────────────────────────────────────────
    print('\n  Generating Grad-CAM overlays (6 patches x 3 model states)...')

    _overlays = []   # list of {col1, col2, col3, meta}
    for _i, _p in enumerate(_patches):
        _true_lbl = _p['true_label']

        # Col 1: E1 on day (target = true class)
        _ov1 = _gradcam_overlay(_model_e1, _p['day_t'],   _true_lbl, _target_layer_e1)
        # Col 2: E1 on night (target = true class — shows where model looks for correct class)
        _ov2 = _gradcam_overlay(_model_e1, _p['night_t'], _true_lbl, _target_layer_e1)
        # Col 3: E5 on night (target = true class)
        _pred_e5_night, _conf_e5_night = _predict(_model_e5, _p['night_t'])
        _ov3 = _gradcam_overlay(_model_e5, _p['night_t'], _true_lbl, _target_layer_e5)

        _overlays.append({
            'col1': _ov1, 'col2': _ov2, 'col3': _ov3,
            'true_label'     : CLASS_NAMES[_true_lbl],
            'conf_day_e1'    : _p['conf_day'],
            'pred_night_e1'  : CLASS_NAMES[_p['pred_night']],
            'conf_night_e1'  : _p['conf_night'],
            'pred_night_e5'  : CLASS_NAMES[_pred_e5_night],
            'conf_night_e5'  : _conf_e5_night,
        })
        print(f'    Patch {_i+1}/{len(_patches)}: true={CLASS_NAMES[_true_lbl]}  '
              f'E1-day={_p["conf_day"]:.2f}✓  '
              f'E1-night={_p["conf_night"]:.2f}✗  '
              f'E5-night={_conf_e5_night:.2f}'
              f'{"✓" if _pred_e5_night == CLASS_NAMES[_true_lbl] else "✗"}')

    # ── 6x3 comparison grid ───────────────────────────────────────────────
    _col_titles = [
        'E1 \u2014 Day Patch\n(correctly classified)',
        'E1 \u2014 Night Patch\n(domain shift)',
        'E5 \u2014 Night Patch\n(adapted)',
    ]

    _fig9, _axes9 = plt.subplots(6, 3, figsize=(12, 22))
    _fig9.suptitle(
        'Grad-CAM Saliency Maps: E1 vs E5 (ConvNeXt-Tiny, target=features[7][-1])\n'
        'Heatmap shows where the model attends when predicting the true class.',
        fontsize=12, fontweight='bold', y=1.01
    )

    # Column headers on row 0
    for _col, _ctitle in enumerate(_col_titles):
        _axes9[0, _col].set_title(_ctitle, fontsize=10, fontweight='bold', pad=6)

    for _row, _meta in enumerate(_overlays):
        _true_str = _meta['true_label']

        # Col 1: E1 day (correct)
        _v1 = _meta['col1']
        _axes9[_row, 0].imshow(_v1)
        _axes9[_row, 0].set_ylabel(_true_str.upper(), fontsize=9, fontweight='bold', rotation=90)
        _axes9[_row, 0].set_xlabel(
            f'Pred={_true_str} ({_meta["conf_day_e1"]:.2f}) ✓',
            fontsize=8, color='green')
        _axes9[_row, 0].set_xticks([]); _axes9[_row, 0].set_yticks([])

        # Col 2: E1 night (confused)
        _v2 = _meta['col2']
        _e1n_ok = _meta['pred_night_e1'] == _true_str
        _axes9[_row, 1].imshow(_v2)
        _axes9[_row, 1].set_xlabel(
            f'Pred={_meta["pred_night_e1"]} ({_meta["conf_night_e1"]:.2f}) '
            f'{"✓" if _e1n_ok else "✗"}',
            fontsize=8, color='green' if _e1n_ok else 'red')
        _axes9[_row, 1].set_xticks([]); _axes9[_row, 1].set_yticks([])

        # Col 3: E5 night (adapted)
        _v3 = _meta['col3']
        _e5n_ok = _meta['pred_night_e5'] == _true_str
        _axes9[_row, 2].imshow(_v3)
        _axes9[_row, 2].set_xlabel(
            f'Pred={_meta["pred_night_e5"]} ({_meta["conf_night_e5"]:.2f}) '
            f'{"✓" if _e5n_ok else "✗"}',
            fontsize=8, color='green' if _e5n_ok else 'red')
        _axes9[_row, 2].set_xticks([]); _axes9[_row, 2].set_yticks([])

    _fig9.tight_layout()
    _fig9.savefig(_gradcam_fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'\n  Figure saved -> {_gradcam_fig_path}')

    # ── Free GPU memory ───────────────────────────────────────────────────
    del _model_e1, _model_e5
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    # ── Update metadata ───────────────────────────────────────────────────
    METADATA['Grad_CAM'] = {
        'target_layer'    : 'features[7][-1]',
        'method'          : 'GradCAM',
        'reshape_transform': 'convnext_NHWC_to_NCHW',
        'n_patches'       : 6,
        'selection'       : 'E1-correct-day + E1-confused-night (strategic)',
        'figure'          : str(_gradcam_fig_path),
    }
    with open(metadata_path, 'w') as _f:
        json.dump(METADATA, _f, indent=2)
    print(f'  Metadata updated -> {metadata_path}')

print(f'\n[OK] Phase 9 complete -- Grad-CAM comparison grid generated.')


---
## Phase 10 — Results Consolidation

Loads all saved JSON metrics, compiles the master results table, generates the 4-panel summary figure,  
performs notebook cleanup, inserts the README cell, and runs the reproducibility audit.  
No training — estimated runtime: <2 min.


In [ ]:
# ============================================================
# STEPS 10.1 + 10.2 -- MASTER RESULTS TABLE + 4-PANEL FIGURE
# ============================================================
import pandas as pd

print_section('Steps 10.1 + 10.2 -- Master Results Table + Summary Figure')

# ── Load all metrics JSONs ────────────────────────────────────────────────
_metrics_files = {
    'E1-Day'     : 'E1_day_metrics.json',
    'E2-Day'     : 'E2_day_metrics.json',
    'E2-Night'   : 'E2_night_metrics.json',
    'E3-Night'   : 'E3_night_metrics.json',
    'E5-Night'   : 'E5_night_metrics.json',
    'E5-Day'     : 'E5_day_metrics.json',
}

_all_metrics = {}
for _exp, _fname in _metrics_files.items():
    _path = DIRS['metrics'] / _fname
    if not _path.exists():
        print(f'  [WARN] Missing: {_fname} -- E5 not yet trained? Using placeholder.')
        _all_metrics[_exp] = None
    else:
        with open(_path) as _f:
            _all_metrics[_exp] = json.load(_f)
        print(f'  [OK] Loaded: {_fname}')

# ── Build master DataFrame ────────────────────────────────────────────────
def _safe(m, key, default='N/A'):
    if m is None: return default
    v = m.get(key, default)
    return round(v, 4) if isinstance(v, float) else v

def _ci_str(m):
    if m is None: return 'N/A'
    ci = m.get('macro_f1_ci_95', [None, None])
    return f'[{ci[0]:.4f}, {ci[1]:.4f}]' if ci[0] is not None else 'N/A'

def _meanstd_str(m):
    if m is None: return 'N/A'
    mu = m.get('mean_macro_f1'); sd = m.get('std_macro_f1')
    return f'{mu:.4f} ± {sd:.4f}' if mu is not None else 'N/A'

_rows = []
for _exp, _m in _all_metrics.items():
    _rows.append({
        'Experiment'    : _exp,
        'Model'         : _safe(_m, 'model'),
        'Split'         : _safe(_m, 'split'),
        'Macro-F1'      : _safe(_m, 'macro_f1'),
        'F1 CI 95%'     : _ci_str(_m),
        'Mean±Std'      : _meanstd_str(_m),
        'PR-AUC'        : _safe(_m, 'auc_pr'),
        'ROC-AUC'       : _safe(_m, 'auc_roc'),
        'MCC'           : _safe(_m, 'mcc'),
        'Accuracy'      : _safe(_m, 'accuracy'),
        'Ripe-F1'       : round(_m['per_class']['ripe']['f1'], 4) if _m and 'per_class' in _m else 'N/A',
        'Unripe-F1'     : round(_m['per_class']['unripe']['f1'], 4) if _m and 'per_class' in _m else 'N/A',
    })

MASTER_RESULTS_DF = pd.DataFrame(_rows)
print(f'\n  Master Results Table ({len(MASTER_RESULTS_DF)} experiments):')
try:
    print(MASTER_RESULTS_DF.to_markdown(index=False))
except ImportError:
    print('  [INFO] tabulate not installed; falling back to to_string().')
    print(MASTER_RESULTS_DF.to_string(index=False))

_csv_path = DIRS['metrics'] / 'master_results_table.csv'
MASTER_RESULTS_DF.to_csv(_csv_path, index=False)
print(f'\n  Saved -> {_csv_path}')

# ── 4-Panel Summary Figure ────────────────────────────────────────────────
_exp_order  = list(_all_metrics.keys())
_f1s        = [_safe(_all_metrics[e], 'macro_f1', 0) for e in _exp_order]
_aucs       = [_safe(_all_metrics[e], 'auc_roc', 0) for e in _exp_order]
_exp_labels = ['E1\nDL-Day', 'E2\nSVM-Day', 'E2-N\nSVM-Night',
                'E3\nDL-Night', 'E5-N\nAdapted\nNight', 'E5-D\nAdapted\nDay']
_bar_cols    = ['#2196F3','#FF9800','#FF5722','#F44336','#9C27B0','#673AB7']

_fig10, _axes10 = plt.subplots(2, 2, figsize=(14, 10))
_fig10.suptitle('Phase 10 — Experiment Summary (E1–E5)', fontsize=14, fontweight='bold')

# Panel A: Macro-F1 bar chart
_ax_a = _axes10[0, 0]
_bars_a = _ax_a.bar(_exp_labels, _f1s, color=_bar_cols, edgecolor='white', width=0.6)
_e1_f1_ref = _safe(_all_metrics['E1-Day'], 'macro_f1', 0)
_ax_a.axhline(_e1_f1_ref, color='#2196F3', linestyle='--', lw=1.2, label=f'E1={_e1_f1_ref:.4f}')
for _b, _v in zip(_bars_a, _f1s):
    _ax_a.text(_b.get_x()+_b.get_width()/2, _b.get_height()+0.005,
                f'{_v:.4f}', ha='center', va='bottom', fontsize=8)
_ax_a.set_ylim(0, 1.08); _ax_a.set_ylabel('Macro-F1', fontsize=10)
_ax_a.set_title('(A) Macro-F1 by Experiment', fontsize=10, fontweight='bold')
_ax_a.legend(fontsize=8); _ax_a.grid(axis='y', alpha=0.3)

# Panel B: ROC-AUC bar chart
_ax_b = _axes10[0, 1]
_bars_b = _ax_b.bar(_exp_labels, _aucs, color=_bar_cols, edgecolor='white', width=0.6)
_e1_auc_ref = _safe(_all_metrics['E1-Day'], 'auc_roc', 0)
_ax_b.axhline(_e1_auc_ref, color='#2196F3', linestyle='--', lw=1.2, label=f'E1={_e1_auc_ref:.4f}')
for _b, _v in zip(_bars_b, _aucs):
    _ax_b.text(_b.get_x()+_b.get_width()/2, _b.get_height()+0.005,
                f'{_v:.4f}', ha='center', va='bottom', fontsize=8)
_ax_b.set_ylim(0, 1.08); _ax_b.set_ylabel('ROC-AUC', fontsize=10)
_ax_b.set_title('(B) ROC-AUC by Experiment', fontsize=10, fontweight='bold')
_ax_b.legend(fontsize=8); _ax_b.grid(axis='y', alpha=0.3)

# Panel C: Delta heatmap (E1->E3 and E1->E5-Night)
_ax_c = _axes10[1, 0]
_e1_m  = _all_metrics['E1-Day']
_e3_m  = _all_metrics['E3-Night']
_e5n_m = _all_metrics['E5-Night']
_delta_labels = ['ΔMacro-F1', 'ΔROC-AUC', 'ΔMCC', 'ΔAccuracy',
                  'ΔRipe-F1', 'ΔUnripe-F1']
_col_labels   = ['E1→E3 (Shift)', 'E1→E5-N (Adapted)']
def _delta(a, b, key):
    av = a.get(key, 0) if a else 0; bv = b.get(key, 0) if b else 0
    return round(av - bv, 4)
def _delta_cls(a, b, cls, metric):
    av = a['per_class'][cls][metric] if a and 'per_class' in a else 0
    bv = b['per_class'][cls][metric] if b and 'per_class' in b else 0
    return round(av - bv, 4)

_dm = np.array([
    [_delta(_e1_m, _e3_m,  'macro_f1'), _delta(_e1_m, _e5n_m, 'macro_f1')],
    [_delta(_e1_m, _e3_m,  'auc_roc'),  _delta(_e1_m, _e5n_m, 'auc_roc')],
    [_delta(_e1_m, _e3_m,  'mcc'),      _delta(_e1_m, _e5n_m, 'mcc')],
    [_delta(_e1_m, _e3_m,  'accuracy'), _delta(_e1_m, _e5n_m, 'accuracy')],
    [_delta_cls(_e1_m, _e3_m, 'ripe', 'f1'),   _delta_cls(_e1_m, _e5n_m, 'ripe', 'f1')],
    [_delta_cls(_e1_m, _e3_m, 'unripe', 'f1'), _delta_cls(_e1_m, _e5n_m, 'unripe', 'f1')],
])
sns.heatmap(_dm, annot=True, fmt='.4f', cmap='RdYlGn_r',
            center=0, vmin=-0.4, vmax=0.1,
            xticklabels=_col_labels, yticklabels=_delta_labels,
            ax=_ax_c, linewidths=0.5)
_ax_c.set_title('(C) Metric Degradation Heatmap\n(positive = E1 better than night)', fontsize=10, fontweight='bold')

# Panel D: Recovery summary text
_ax_d = _axes10[1, 1]
_ax_d.axis('off')
_e5_rec_path = DIRS['metrics'] / 'E5_recovery.json'
_e5_rec = {}
if _e5_rec_path.exists():
    with open(_e5_rec_path) as _f:
        _e5_rec = json.load(_f)
_summary_lines = [
    '(D) E5 Adaptation Summary',
    '',
    f'  E1 Day F1  (ceiling) : {_safe(_e1_m, "macro_f1", "N/A")}',
    f'  E3 Night F1 (floor)  : {_safe(_e3_m, "macro_f1", "N/A")}',
    f'  E5 Night F1 (adapted): {_safe(_e5n_m, "macro_f1", "N/A")}',
    f'  E5 Day F1  (stable)  : {_safe(_all_metrics["E5-Day"], "macro_f1", "N/A")}',
    '',
    f'  Recovery ratio : {_e5_rec.get("recovery_ratio", "N/A")} ({(_e5_rec.get("recovery_ratio", 0) or 0)*100:.1f}%)',
    f'  Day retention  : {_e5_rec.get("day_retention", "N/A")} ({(_e5_rec.get("day_retention", 0) or 0)*100:.1f}%)',
    '',
    f'  E4 \u0394F1 (Shift)   : {_delta(_e1_m, _e3_m, "macro_f1"):+.4f}',
    f'  E5 \u0394F1 recovered : {(_safe(_e5n_m, "macro_f1", 0) or 0) - (_safe(_e3_m, "macro_f1", 0) or 0):+.4f}',
]
_ax_d.text(0.05, 0.95, '\n'.join(_summary_lines),
            transform=_ax_d.transAxes, fontsize=10,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.8))
_ax_d.set_title('(D) Adaptation Summary', fontsize=10, fontweight='bold')

_fig10.tight_layout()
_fig10.savefig(DIRS['figures'] / 'summary_4panel.png', dpi=300)
plt.show()
print('\n  Figure saved -> summary_4panel.png')

# Update metadata
METADATA['Phase10'] = {'master_csv': str(_csv_path), 'summary_figure': 'summary_4panel.png'}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)
print(f'[OK] Steps 10.1 + 10.2 complete.')


In [ ]:
# ============================================================
# STEPS 10.3 + 10.4 -- NOTEBOOK CLEANUP + README CELL
# NOTE: This cell modifies the notebook FILE on disk.
#       Re-open in Jupyter after running to see the README.
# ============================================================
print_section('Steps 10.3 + 10.4 -- Notebook Cleanup + README')

import json as _json_mod, datetime as _dt
from pathlib import Path as _Path

_nb_path = _Path('/content/drive/MyDrive/strawberry_domain_shift.ipynb')
if not _nb_path.exists():
    # Fallback: try local path (for local development)
    _nb_path = _Path('strawberry_domain_shift.ipynb')
    if not _nb_path.exists():
        print('  [SKIP] Notebook file not found at expected path.')
        print('         Run this cell from Colab with Drive mounted.')
else:
    with open(_nb_path) as _nbf:
        _nb = _json_mod.load(_nbf)

    # ── Step 10.3: Strip heavy training outputs ────────────────────────────
    # Cells with long epoch logs: look for cells whose output > 2000 chars
    _stripped = 0
    # Reset all execution counts
    for _cell in _nb['cells']:
        if _cell['cell_type'] == 'code':
            _cell['execution_count'] = None
            # Strip outputs from training loop cells (heuristic: >50 output lines = training log)
            if _cell.get('outputs'):
                _total_txt = ''.join(
                    ''.join(o.get('text', [])) for o in _cell['outputs']
                )
                if _total_txt.count('Ep ') > 10:   # epoch log heuristic
                    _cell['outputs'] = []
                    _stripped += 1

    print(f'  Step 10.3: execution counts reset; {_stripped} training-log output(s) stripped.')

    # ── Step 10.4: README cell (inserted at position 0) ───────────────────
    _today = _dt.date.today().isoformat()   # actual run date
    if 'MASTER_RESULTS_DF' not in globals():
        raise RuntimeError(
            'MASTER_RESULTS_DF not found. Run Steps 10.1+10.2 (cell above) first.'
        )
    _e1_f1_val  = MASTER_RESULTS_DF.loc[MASTER_RESULTS_DF['Experiment']=='E1-Day',   'Macro-F1'].values[0]
    _e3_f1_val  = MASTER_RESULTS_DF.loc[MASTER_RESULTS_DF['Experiment']=='E3-Night', 'Macro-F1'].values[0]
    _e5n_f1_val = MASTER_RESULTS_DF.loc[MASTER_RESULTS_DF['Experiment']=='E5-Night', 'Macro-F1'].values[0]

    _readme_source = [
        '# Strawberry Ripeness Domain Shift Study\n',
        '**Module**: AIDL Individual Assessment  \n',
        '**Date**: ' + _today + '  \n',
        '\n',
        '---\n',
        '## Experiment Summary\n',
        '\n',
        '| Experiment | Model | Split | Macro-F1 |\n',
        '|---|---|---|---|\n',
        f'| E1 DL-Day       | ConvNeXt-Tiny | test_day   | {_e1_f1_val} |\n',
        f'| E3 DL-Day->Night | ConvNeXt-Tiny | test_night | {_e3_f1_val} |\n',
        f'| E5 Adapted-Night | ConvNeXt-Tiny | test_night | {_e5n_f1_val} |\n',
        '\n',
        '---\n',
        '## How to Run\n',
        '1. Mount Google Drive: `from google.colab import drive; drive.mount("/content/drive")`\n',
        '2. **Runtime → Run All** (estimated: ~20 min on T4)\n',
        '3. Results saved to `MyDrive/aidl_strawberry/`\n',
        '\n',
        '## Environment\n',
        '- Python 3.10, PyTorch 2.x, torchvision 0.15+\n',
        '- Google Colab NVIDIA T4 GPU\n',
        '- Seeds: 42, 123, 456 (all random operations seeded)\n',
    ]

    _readme_cell = {
        'cell_type': 'markdown',
        'metadata': {},
        'source': _readme_source
    }

    # Check if README already exists at index 0
    if _nb['cells'][0].get('source') and '# Strawberry Ripeness' in ''.join(_nb['cells'][0]['source']):
        _nb['cells'][0] = _readme_cell
        print('  Step 10.4: README cell updated at index 0.')
    else:
        _nb['cells'].insert(0, _readme_cell)
        print(f'  Step 10.4: README cell inserted at index 0. Total cells: {len(_nb["cells"])}')

    with open(_nb_path, 'w') as _nbf:
        _json_mod.dump(_nb, _nbf, indent=1)
    print(f'  Notebook saved -> {_nb_path}')

print(f'\n[OK] Steps 10.3 + 10.4 complete.')


In [ ]:
# ============================================================
# STEP 10.5 -- REPRODUCIBILITY AUDIT
# ============================================================
print_section('Step 10.5 -- Reproducibility Audit')

_audit_results = {}
_pass = True

# ── Audit 1: GLOBAL_SEED defined ─────────────────────────────────────────
try:
    assert isinstance(GLOBAL_SEED, int)
    _audit_results['GLOBAL_SEED defined (int)'] = f'PASS  ({GLOBAL_SEED})'
except Exception:
    _audit_results['GLOBAL_SEED defined (int)'] = 'FAIL  (not an int)'
    _pass = False

# ── Audit 2: EXPERIMENT_SEEDS ────────────────────────────────────────────
try:
    assert len(EXPERIMENT_SEEDS) == 3
    _audit_results['EXPERIMENT_SEEDS (3 values)'] = f'PASS  ({EXPERIMENT_SEEDS})'
except Exception:
    _audit_results['EXPERIMENT_SEEDS (3 values)'] = 'FAIL'
    _pass = False

# ── Audit 3: All 6 metrics JSON files exist ───────────────────────────────
_req_metrics = [
    'E1_day_metrics.json', 'E2_day_metrics.json', 'E2_night_metrics.json',
    'E3_night_metrics.json', 'E5_night_metrics.json', 'E5_day_metrics.json'
]
_missing_m = [f for f in _req_metrics if not (DIRS['metrics'] / f).exists()]
_audit_results['6 metrics JSONs exist'] = (
    f'PASS  (all found)' if not _missing_m else f'FAIL  missing: {_missing_m}'
)
if _missing_m: _pass = False

# ── Audit 4: 6 checkpoint files exist (3xE1 + 3xE5) ─────────────────────
_req_ckpts = [f'E1_seed{s}_best.pt' for s in EXPERIMENT_SEEDS] + \
             [f'E5_seed{s}_best.pt' for s in EXPERIMENT_SEEDS]
_missing_c = [f for f in _req_ckpts if not (DIRS['checkpoints'] / f).exists()]
_audit_results['6 checkpoints exist (3xE1, 3xE5)'] = (
    'PASS  (all found)' if not _missing_c else f'FAIL  missing: {_missing_c}'
)
if _missing_c: _pass = False

# ── Audit 5: Required figure files exist ─────────────────────────────────
_req_figs = [
    'augmentation_verification.png', 'E1_training_curves.png',
    'E1_confusion_matrix.png', 'E1_roc_curve.png',
    'night_patch_verification.png', 'E3_confusion_matrix.png',
    'E4_confusion_matrix_comparison.png', 'E4_f1_comparison_bar.png',
    'E5_adapted_dataset_verification.png', 'E5_full_comparison_bar.png',
    'gradcam_comparison_grid.png', 'summary_4panel.png',
]
_missing_f = [f for f in _req_figs if not (DIRS['figures'] / f).exists()]
_audit_results[f'{len(_req_figs)} figures exist'] = (
    'PASS  (all found)' if not _missing_f else f'FAIL  missing: {_missing_f}'
)
if _missing_f: _pass = False

# ── Audit 6: metadata.json has required keys ──────────────────────────────
_req_meta_keys = [
    'dl_training_config', 'dl_augmentation', 'E1_training',
    'E3_test_night', 'E4_degradation', 'E5_adaptation', 'Grad_CAM'
]
_missing_mk = [k for k in _req_meta_keys if k not in METADATA]
_audit_results['metadata.json has required keys'] = (
    'PASS  (all found)' if not _missing_mk else f'FAIL  missing: {_missing_mk}'
)
if _missing_mk: _pass = False

# ── Audit 7: master_results_table.csv ─────────────────────────────────────
_csv_exists = (DIRS['metrics'] / 'master_results_table.csv').exists()
_audit_results['master_results_table.csv exists'] = (
    'PASS' if _csv_exists else 'FAIL'
)
if not _csv_exists: _pass = False

# ── Audit 8: E5_recovery.json ─────────────────────────────────────────────
_rec_exists = (DIRS['metrics'] / 'E5_recovery.json').exists()
_audit_results['E5_recovery.json exists'] = (
    'PASS' if _rec_exists else 'FAIL'
)
if not _rec_exists: _pass = False

# ── Print audit report ────────────────────────────────────────────────────
print(f'\n  {"-"*60}')
print(f'  Reproducibility Audit Report')
print(f'  {"-"*60}')
for _item, _status in _audit_results.items():
    _icon = '[PASS]' if _status.startswith('PASS') else '[FAIL]'
    print(f'  {_icon}  {_item}')
    if not _status.startswith('PASS'):
        print(f'          {_status}')

print(f'\n  {"-"*60}')
print(f'  Overall: {"PASS" if _pass else "FAIL -- see items above"}')
print(f'  {"-"*60}')

METADATA['reproducibility_audit'] = {
    'all_pass'    : _pass,
    'audit_items' : _audit_results,
}
with open(metadata_path, 'w') as _f:
    json.dump(METADATA, _f, indent=2)

print(f'\n  Metadata updated -> {metadata_path}')
print(f'\n[OK] Step 10.5 / Phase 10 complete.')
print(f'\n{"="*60}')
print(f'ALL PIPELINE CELLS IMPLEMENTED (Phases 0-10).')
print(f'Proceed to Phase 11 (Critical Reflection Report) in a separate document.')
print(f'{"="*60}')
